In [4]:
import csv
import json
import random
from datetime import date, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260624
NUMBER_OF_CONTRACTS = 20_000

random.seed(RANDOM_SEED)


# ============================================================
# PROJECT PATHS
# ============================================================

# Jupyter is already running inside risk_scoring_project,
# so use its current folder directly.
PROJECT_ROOT = Path.cwd()

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "145", "168", "201", "268"
]

PRODUCTS = [
    {"product_code": "CZL3", "product_category": "CONSUMER_LOAN"},
    {"product_code": "AUTO", "product_category": "AUTO_LOAN"},
    {"product_code": "MORT", "product_category": "MORTGAGE"},
    {"product_code": "BIZL", "product_category": "SME_LOAN"},
    {"product_code": "PERS", "product_category": "PERSONAL_LOAN"},
    {"product_code": "CRED", "product_category": "CREDIT_LINE"},
]

CUSTOMER_IDS = [str(customer_id) for customer_id in range(4_000_001, 4_015_001)]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    """Returns one random date between two dates."""

    number_of_days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, number_of_days))


def generate_account_number(
    branch_code: str,
    product_code: str,
    book_date: date,
    sequence_number: int
) -> str:
    """
    Creates a 16-character account number.

    Example:
    268CZL3171380001
    """

    year_part = book_date.strftime("%y")
    sequence_part = f"{sequence_number:07d}"

    return f"{branch_code}{product_code}{year_part}{sequence_part}"


def generate_application_number(
    book_date: date,
    sequence_number: int
) -> str:
    """
    Generates a maximum 16-character application number.
    Example: APP2600000000001
    """

    return f"APP{book_date:%y}{sequence_number:011d}"


def generate_contract_row(sequence_number: int) -> dict:
    """
    Generates one clean contract-level record.

    These fields are the identity and relationship fields needed
    before we generate the remaining loan attributes.
    """

    branch_code = random.choice(BRANCHES)
    product = random.choice(PRODUCTS)

    book_date = random_date(
        start_date=date(2018, 1, 1),
        end_date=date(2026, 5, 31)
    )

    account_number = generate_account_number(
        branch_code=branch_code,
        product_code=product["product_code"],
        book_date=book_date,
        sequence_number=sequence_number
    )

    customer_id = random.choice(CUSTOMER_IDS)

    return {
        "ACCOUNT_NUMBER": account_number,
        "BRANCH_CODE": branch_code,
        "APPLICATION_NUM": generate_application_number(
            book_date=book_date,
            sequence_number=sequence_number
        ),
        "CUSTOMER_ID": customer_id,
        "PRODUCT_CODE": product["product_code"],
        "PRODUCT_CATEGORY": product["product_category"],
        "BOOK_DATE": book_date.isoformat(),
        "VALUE_DATE": book_date.isoformat(),
        "ORIGINAL_ST_DATE": book_date.isoformat(),
        "PRIMARY_APPLICANT_ID": customer_id,
        "ALT_ACC_NO": account_number
    }


def validate_contracts(contract_rows: list[dict]) -> None:
    """
    Stops the script if contract identifiers are invalid.
    """

    account_numbers = [
        row["ACCOUNT_NUMBER"]
        for row in contract_rows
    ]

    application_numbers = [
        row["APPLICATION_NUM"]
        for row in contract_rows
    ]

    if len(contract_rows) != NUMBER_OF_CONTRACTS:
        raise ValueError(
            f"Expected {NUMBER_OF_CONTRACTS:,} rows, "
            f"but generated {len(contract_rows):,}."
        )

    if len(account_numbers) != len(set(account_numbers)):
        raise ValueError("Duplicate ACCOUNT_NUMBER values were generated.")

    if len(application_numbers) != len(set(application_numbers)):
        raise ValueError("Duplicate APPLICATION_NUM values were generated.")

    if any(not account_number for account_number in account_numbers):
        raise ValueError("A blank ACCOUNT_NUMBER was generated.") 

    if any(len(account_number) > 35 for account_number in account_numbers):
        raise ValueError(
            "At least one ACCOUNT_NUMBER exceeds VARCHAR2(35)."
        )


def write_csv(file_path: Path, contract_rows: list[dict]) -> None:
    """Writes contract data to CSV."""

    column_names = list(contract_rows[0].keys())

    with open(file_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=column_names
        )

        writer.writeheader()
        writer.writerows(contract_rows)


# ============================================================
# MAIN SCRIPT
# ============================================================

def main() -> None:
    print(f"Generating {NUMBER_OF_CONTRACTS:,} unique contracts...")

    contract_rows = [
        generate_contract_row(sequence_number=sequence_number)
        for sequence_number in range(1, NUMBER_OF_CONTRACTS + 1)
    ]

    validate_contracts(contract_rows)

    full_output_path = RAW_DATA_FOLDER / "rs_contracts_base.csv"
    sample_output_path = SAMPLE_DATA_FOLDER / "rs_contracts_base_sample.csv"
    summary_output_path = RAW_DATA_FOLDER / "rs_contracts_generation_summary.json"

    write_csv(
        file_path=full_output_path,
        contract_rows=contract_rows
    )

    write_csv(
        file_path=sample_output_path,
        contract_rows=contract_rows[:100]
    )

    generation_summary = {
        "table_name": "RS_CONTRACTS",
        "generated_rows": len(contract_rows),
        "unique_account_numbers": len(
            {row["ACCOUNT_NUMBER"] for row in contract_rows}
        ),
        "unique_application_numbers": len(
            {row["APPLICATION_NUM"] for row in contract_rows}
        ),
        "customer_id_pool_size": len(CUSTOMER_IDS),
        "random_seed": RANDOM_SEED,
        "full_file": str(full_output_path),
        "sample_file": str(sample_output_path)
    }

    with open(summary_output_path, "w", encoding="utf-8") as file:
        json.dump(
            generation_summary,
            file,
            indent=4
        )

    print("Generation completed successfully.")
    print(f"Full dataset: {full_output_path}")
    print(f"Sample dataset: {sample_output_path}")
    print(f"Unique ACCOUNT_NUMBER count: {generation_summary['unique_account_numbers']:,}")


if __name__ == "__main__":
    main()

Generating 20,000 unique contracts...
Generation completed successfully.
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contracts_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_contracts_base_sample.csv
Unique ACCOUNT_NUMBER count: 20,000


In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\ASUS\Desktop\project\risk_scoring_project


In [5]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260625
NUMBER_OF_CUSTOMERS = 15_000

# This matches the customer ID pool used in generate_contracts.py.
CUSTOMER_ID_START = 4_000_001
CUSTOMER_ID_END = 4_015_000

random.seed(RANDOM_SEED)


# ============================================================
# NOTEBOOK-SAFE PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CONTRACT_FILE = PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# CUSTOMER TABLE COLUMNS
# ============================================================

CUSTOMER_COLUMNS = [
    "GEN_DIM_CUST_ID",
    "CUSTOMER_NO",
    "PIN",
    "DOCUMENT_NO",
    "TAX_ID",
    "CUSTOMER_TYPE",
    "CUSTOMER_SUB_TYPE",
    "FULL_NAME",
    "FIRST_NAME",
    "LAST_NAME",
    "MIDDLE_NAME",
    "GENDER",
    "BIRTH_DT",
    "PLACE_OF_BIRTH",
    "MARITAL_STATUS",
    "NATIONALITY",
    "OPEN_REASON",
    "CREATION_DT",
    "OPENING_BRN",
    "FLG_IS_COURT_ISSUE",
    "OCCUPATION",
    "WORKPLACE_CUSTOMER_NO",
    "WORKPLACE_NAME",
    "FLG_IS_BLACK_LIST",
    "FLG_IS_VIP",
    "FLG_IS_EMPLOYEER",
    "NUM_OF_EMPLOYEER",
    "RM_NAME",
    "CUSTOMER_GENERATION",
    "FLG_IS_AFFLUENT",
    "FLG_IS_SALARY",
    "FLG_IS_PENSION",
    "FLG_IS_EDV",
    "RISK_LEVEL",
    "RESIDENT_STATUS",
    "RM_ID_CORP",
    "GROUP_ID_CORP",
    "CHECKER_DT_STAMP",
    "MAKER_DT_STAMP",
    "EMPL_PROPERTY_TYPE",
    "SHORT_NAME",
    "COUNTRY",
    "DOCUMENT_NAME",
    "LEGAL_GUARDIAN",
    "PASSPORT_COUNTRY",
    "PASSPORT_NATIONAL_ID",
    "PASSPORT_ISS_DT",
    "PASSPORT_EXP_DT",
    "P_ADDRESS1",
    "P_ADDRESS2",
    "P_ADDRESS3",
    "ENGLISH_FULL_NAME",
    "FLG_IS_BENEFICIARY",
    "CUSTOMER_SINCE",
    "DIRECTOR_CUST_NO",
    "FLG_IS_GREEN_CARD",
    "US_BIRTH",
    "FLG_IS_US_CITIZEN",
    "US_PAYMENT",
    "FLG_IS_US_RESIDENT_POA",
    "FLG_IS_US_ADDRESS",
    "FLG_IS_US_LIVING",
    "US_PASSPORT",
    "FLG_IS_US_PHONE",
    "FLG_IS_US_POA",
    "FLG_IS_US_POST_BOX",
    "US_RELATED_PERSON",
    "FLG_IS_US_TAX_RESIDENT",
    "WORKPLACE_ADDRESS",
    "WORKPLACE",
    "CIF_RECORD_STATUS",
    "FLG_IS_FROZEN",
    "INS_DTTM",
    "TAX_TYPE",
    "FLG_IS_INGROUP",
    "CUST_MIS_1",
    "SHORT_NAME2",
    "SWIFT_CD",
    "CHARGE_GROUP",
    "DEFAULT_MEDIA",
    "CUSTOMER_ADDRESS",
    "CORP_REGISTER_ADDRESS",
    "CORP_REGISTER_COUNTRY",
    "CHECKER_DT",
    "FLG_IS_MILITARY_PASSPORT",
    "CUST_SUB_SEGMENT",
    "AGRAR_PKD",
    "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY",
    "FLG_IS_REMOTE_ACC_ALLOWED",
    "ONLY_LAST_AND_FIRST_NAME_ENG",
    "EMPLOYER_ID",
    "INCORP_DATE",
    "DT_QHT_GROUP",
    "CUST_CLASSIFICATION",
    "LOAN_RESP_ID",
    "ULTIMATE_BENEFICIAL_OWNER",
    "FLG_IS_RELATED_PERSON",
    "COUNTRY_DESC",
    "REGISTRATION_ADDRESS",
    "DOMICILE_ADDRESS",
]


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "120", "145", "168", "201", "268"
]

CITIES = [
    "BAKI", "SUMQAYIT", "GANCA", "MINGECEVIR", "SIRVAN",
    "LENKERAN", "QUBA", "SEKI", "MASALLI", "SAMAXI",
    "YEVLAX", "QABALA", "NAXCIVAN", "ZAGATALA"
]

STREETS = [
    "NIZAMI KUCHESI",
    "ATATURK PROSPEKTI",
    "HEYDER ELIYEV PROSPEKTI",
    "XETAI PROSPEKTI",
    "TBLISI PROSPEKTI",
    "BABEK PROSPEKTI",
    "SULEYMAN RUSTEM KUCHESI",
    "AZADLIQ PROSPEKTI"
]

MALE_FIRST_NAMES = [
    "Elvin", "Rashad", "Murad", "Tural", "Orkhan", "Kamran",
    "Nihad", "Samir", "Fuad", "Javid", "Anar", "Emil",
    "Rovshan", "Elnur", "Aydin", "Teymur", "Ramil", "Aqil"
]

FEMALE_FIRST_NAMES = [
    "Aysel", "Nigar", "Leyla", "Gunel", "Sevda", "Narmin",
    "Aynur", "Khadija", "Sabina", "Aytac", "Zahra", "Arzu",
    "Laman", "Ulker", "Fatima", "Rena", "Aysu", "Turan"
]

FATHER_NAMES = [
    "Rasim", "Vugar", "Camil", "Tahir", "Nazim", "Farhad",
    "Ilham", "Elman", "Adil", "Rauf", "Natiq", "Samad"
]

SURNAMES = [
    "Mammadov", "Aliyev", "Hasanov", "Huseynov", "Ibrahimov",
    "Rahimov", "Karimov", "Quliyev", "Babayeva", "Hajiyeva",
    "Asgarov", "Jafarov", "Suleymanov", "Nabiyev", "Ismayilov",
    "Abbasov", "Rzayev", "Safarov", "Tagiyev", "Aghayev"
]

OCCUPATIONS = [
    "ACCOUNTANT", "TEACHER", "ENGINEER", "DOCTOR",
    "SALES SPECIALIST", "IT SPECIALIST", "DRIVER",
    "MANAGER", "LAWYER", "ANALYST", "ENTREPRENEUR",
    "TECHNICIAN", "CIVIL SERVANT", "DESIGNER"
]

WORKPLACES = [
    "CASPIAN LOGISTICS MMC",
    "BAKU RETAIL MMC",
    "AZER TELECOM MMC",
    "NORTH CONSTRUCTION MMC",
    "CITY EDUCATION CENTER",
    "MEDICAL SERVICE MMC",
    "FINANCE CONSULTING MMC",
    "REGIONAL TRADE MMC",
    "TECH SOLUTIONS MMC"
]

COMPANY_PREFIXES = [
    "Caspian", "Araz", "Baku", "Shirvan", "Absheron",
    "Zirve", "Khazar", "North", "Silk Road", "Azer",
    "Green", "Atlas"
]

COMPANY_ACTIVITIES = [
    "Logistics", "Construction", "Trade", "Food",
    "Technology", "Textile", "Agriculture", "Pharmacy",
    "Tourism", "Energy", "Consulting", "Manufacturing",
    "Transport", "Services"
]

COMPANY_SUFFIXES = ["MMC", "ASC"]

RELATIONSHIP_MANAGERS = [
    "A. Mammadov",
    "L. Aliyeva",
    "R. Hasanov",
    "N. Ibrahimova",
    "K. Huseynov",
    "S. Rahimova"
]

SWIFT_CODES = [
    "AZEBAZ22",
    "BAKUAJ22",
    "CASPAZ22",
    "FINAAZ22"
]

PIN_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ0123456789"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    if end_date < start_date:
        return start_date

    days_between = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, days_between))


def format_date(value):
    return value.isoformat() if value else ""


def format_timestamp(value):
    return value.strftime("%Y-%m-%d %H:%M:%S") if value else ""


def years_before(reference_date: date, years: int) -> date:
    try:
        return reference_date.replace(year=reference_date.year - years)
    except ValueError:
        return reference_date.replace(
            year=reference_date.year - years,
            month=2,
            day=28
        )


def generation_label(birth_date: date) -> str:
    if birth_date.year <= 1964:
        return "BABY_BOOMERS"
    if birth_date.year <= 1980:
        return "GEN_X"
    if birth_date.year <= 1996:
        return "MILLENNIALS_GEN_Y"

    return "GEN_Z"


def parse_iso_date(value: str):
    if not value:
        return None

    try:
        return datetime.strptime(value.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


def generate_unique_pin(existing_pins: set) -> str:
    while True:
        pin = (
            str(random.randint(1, 9))
            + "".join(random.choices(PIN_ALPHABET, k=6))
        )

        if pin not in existing_pins:
            existing_pins.add(pin)
            return pin


def generate_unique_document_number(existing_documents: set) -> str:
    while True:
        document_number = f"AZE{random.randint(10_000_000, 99_999_999)}"

        if document_number not in existing_documents:
            existing_documents.add(document_number)
            return document_number


def generate_unique_tax_id(existing_tax_ids: set) -> str:
    while True:
        tax_id = str(random.randint(1_000_000_000, 9_999_999_999))

        if tax_id not in existing_tax_ids:
            existing_tax_ids.add(tax_id)
            return tax_id


def make_address(city: str) -> str:
    return (
        f"{city}, {random.choice(STREETS)}, "
        f"EV {random.randint(1, 250)}, MEN {random.randint(1, 150)}"
    )


# ============================================================
# CONTRACT READING AND KEY MATCHING
# ============================================================

def load_contract_customer_details():
    """
    Reads contracts and collects:
    - customer IDs
    - earliest contract date per customer
    - products per customer
    """

    if not CONTRACT_FILE.exists():
        raise FileNotFoundError(
            f"Contract file not found: {CONTRACT_FILE}\n"
            "First run the contract generator."
        )

    customer_ids = set()
    first_contract_date = {}
    product_codes = defaultdict(set)

    with open(
        CONTRACT_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_ID",
            "BOOK_DATE",
            "PRODUCT_CODE"
        }

        missing_columns = required_columns - set(reader.fieldnames or [])

        if missing_columns:
            raise ValueError(
                "Missing contract columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_id = (row.get("CUSTOMER_ID") or "").strip()

            if not customer_id:
                continue

            customer_ids.add(customer_id)

            book_date = parse_iso_date(
                row.get("BOOK_DATE") or ""
            )

            if book_date:
                current_minimum = first_contract_date.get(customer_id)

                if (
                    current_minimum is None
                    or book_date < current_minimum
                ):
                    first_contract_date[customer_id] = book_date

            product_code = (row.get("PRODUCT_CODE") or "").strip()

            if product_code:
                product_codes[customer_id].add(product_code)

    return customer_ids, first_contract_date, product_codes


def build_customer_numbers(contract_customer_ids: set) -> list:
    """
    Creates exactly 15,000 CIF values.

    Every CUSTOMER_ID in contracts is included here, guaranteeing:
    CUSTOMER_ID = CUSTOMER_NO.
    """

    customer_numbers = [
        str(customer_id)
        for customer_id in range(
            CUSTOMER_ID_START,
            CUSTOMER_ID_END + 1
        )
    ]

    if len(customer_numbers) != NUMBER_OF_CUSTOMERS:
        raise ValueError(
            "Customer ID range must create exactly "
            f"{NUMBER_OF_CUSTOMERS:,} records."
        )

    unmatched_contract_ids = (
        contract_customer_ids - set(customer_numbers)
    )

    if unmatched_contract_ids:
        examples = ", ".join(sorted(unmatched_contract_ids)[:5])

        raise ValueError(
            "Some contract CUSTOMER_ID values are outside the expected "
            f"customer-ID range. Examples: {examples}"
        )

    random.shuffle(customer_numbers)

    return customer_numbers


def choose_customer_type(product_codes: set) -> str:
    """
    Business loans produce more corporate customers.
    Consumer loans produce mostly individual customers.
    """

    if "BIZL" in product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[35, 63, 2],
            k=1
        )[0]

    if product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[88, 11, 1],
            k=1
        )[0]

    return random.choices(
        ["I", "C", "B"],
        weights=[80, 19, 1],
        k=1
    )[0]


# ============================================================
# ROW GENERATORS
# ============================================================

def make_base_row(customer_no: str, generated_id: int) -> dict:
    row = {
        column_name: ""
        for column_name in CUSTOMER_COLUMNS
    }

    row.update({
        "GEN_DIM_CUST_ID": str(generated_id),
        "CUSTOMER_NO": customer_no,
        "FLG_IS_COURT_ISSUE": "N",
        "FLG_IS_BLACK_LIST": "N",
        "FLG_IS_VIP": "N",
        "FLG_IS_EMPLOYEER": "N",
        "FLG_IS_EDV": "Y",
        "FLG_IS_BENEFICIARY": "N",
        "FLG_IS_GREEN_CARD": "N",
        "US_BIRTH": "N",
        "FLG_IS_US_CITIZEN": "N",
        "US_PAYMENT": "N",
        "FLG_IS_US_RESIDENT_POA": "N",
        "FLG_IS_US_ADDRESS": "N",
        "FLG_IS_US_LIVING": "N",
        "US_PASSPORT": "N",
        "FLG_IS_US_PHONE": "N",
        "FLG_IS_US_POA": "N",
        "FLG_IS_US_POST_BOX": "N",
        "US_RELATED_PERSON": "N",
        "FLG_IS_US_TAX_RESIDENT": "N",
        "CIF_RECORD_STATUS": "O",
        "FLG_IS_FROZEN": "N",
        "FLG_IS_INGROUP": "N",
        "FLG_IS_MILITARY_PASSPORT": "N",
        "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY": "Y",
        "FLG_IS_REMOTE_ACC_ALLOWED": "Y",
        "FLG_IS_RELATED_PERSON": "N",
        "COUNTRY": "AZ",
        "COUNTRY_DESC": "AZERBAIJAN",
        "PASSPORT_COUNTRY": "AZ",
        "RESIDENT_STATUS": "R",
        "RISK_LEVEL": str(
            random.choices(
                [1, 2, 3, 4, 5],
                weights=[10, 30, 35, 20, 5],
                k=1
            )[0]
        ),
        "DEFAULT_MEDIA": random.choice(
            ["SMS", "EMAIL", "MOBILE_APP"]
        ),
        "OPENING_BRN": random.choice(BRANCHES),
        "INS_DTTM": "2026-06-21",
        "CHECKER_DT": "2026-06-21",
    })

    return row


def generate_individual(
    row: dict,
    first_contract_date: date,
    existing_pins: set,
    existing_documents: set
) -> dict:
    gender = random.choice(["M", "F"])

    first_name = (
        random.choice(MALE_FIRST_NAMES)
        if gender == "M"
        else random.choice(FEMALE_FIRST_NAMES)
    )

    last_name = random.choice(SURNAMES)
    father_name = random.choice(FATHER_NAMES)

    middle_name = (
        f"{father_name} OGLU"
        if gender == "M"
        else f"{father_name} QIZI"
    )

    age_at_loan = random.randint(21, 72)

    birth_date = years_before(
        first_contract_date,
        age_at_loan
    )

    birth_date -= timedelta(days=random.randint(0, 364))

    customer_creation_start = max(
        date(2005, 1, 1),
        birth_date + timedelta(days=18 * 365)
    )

    creation_date = random_date(
        customer_creation_start,
        first_contract_date
    )

    passport_issue_date = random_date(
        max(
            birth_date + timedelta(days=16 * 365),
            date(2010, 1, 1)
        ),
        creation_date
    )

    passport_expiry_date = (
        passport_issue_date
        + timedelta(days=random.randint(7 * 365, 10 * 365))
    )

    city = random.choice(CITIES)
    occupation = random.choice(OCCUPATIONS)
    workplace = random.choice(WORKPLACES)

    is_salary_customer = random.choices(
        ["Y", "N"],
        weights=[58, 42],
        k=1
    )[0]

    is_pensioner = (
        "Y"
        if age_at_loan >= 63 and random.random() < 0.55
        else "N"
    )

    is_affluent = "Y" if random.random() < 0.08 else "N"

    is_vip = (
        "Y"
        if is_affluent == "Y" and random.random() < 0.12
        else "N"
    )

    is_employee = "Y" if random.random() < 0.02 else "N"

    full_name = f"{last_name} {first_name} {middle_name}"
    english_name = f"{last_name.upper()} {first_name.upper()}"

    row.update({
        "PIN": generate_unique_pin(existing_pins),
        "DOCUMENT_NO": generate_unique_document_number(
            existing_documents
        ),
        "CUSTOMER_TYPE": "I",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["INDIVIDUAL", "RETAIL", "INDIVIDUAL OWNER"],
            weights=[50, 42, 8],
            k=1
        )[0],
        "FULL_NAME": full_name,
        "FIRST_NAME": first_name,
        "LAST_NAME": last_name,
        "MIDDLE_NAME": middle_name,
        "GENDER": gender,
        "BIRTH_DT": format_date(birth_date),
        "PLACE_OF_BIRTH": city,
        "MARITAL_STATUS": random.choices(
            ["S", "M", "D", "W"],
            weights=[35, 54, 7, 4],
            k=1
        )[0],
        "NATIONALITY": "AZ",
        "OPEN_REASON": (
            "PENSION"
            if is_pensioner == "Y"
            else "SALARY"
            if is_salary_customer == "Y"
            else random.choice(["CARD", "ABBM", "LOAN"])
        ),
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": occupation,
        "WORKPLACE_CUSTOMER_NO": (
            str(random.randint(5_000_001, 5_050_000))
            if random.random() < 0.45
            else ""
        ),
        "WORKPLACE_NAME": workplace,
        "FLG_IS_VIP": is_vip,
        "FLG_IS_EMPLOYEER": is_employee,
        "RM_NAME": (
            random.choice(RELATIONSHIP_MANAGERS)
            if is_affluent == "Y"
            else ""
        ),
        "CUSTOMER_GENERATION": generation_label(birth_date),
        "FLG_IS_AFFLUENT": is_affluent,
        "FLG_IS_SALARY": is_salary_customer,
        "FLG_IS_PENSION": is_pensioner,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=12,
                minutes=random.randint(0, 59)
            )
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=11,
                minutes=random.randint(0, 59)
            )
        ),
        "EMPL_PROPERTY_TYPE": random.choice([
            "PRIVATE_SECTOR",
            "PUBLIC_SECTOR",
            "SELF_EMPLOYED"
        ]),
        "SHORT_NAME": (
            f"{first_name[:7].upper()}{row['PIN'][:7]}"
        )[:20],
        "DOCUMENT_NAME": "IDENTITY_CARD",
        "PASSPORT_NATIONAL_ID": row["PIN"],
        "PASSPORT_ISS_DT": format_date(passport_issue_date),
        "PASSPORT_EXP_DT": format_date(passport_expiry_date),
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": english_name,
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(
            random.choice(CITIES)
        ),
        "WORKPLACE": workplace,
        "CUST_MIS_1": "RETAIL",
        "SHORT_NAME2": (
            f"{last_name[:8].upper()}{first_name[:4].upper()}"
        )[:20],
        "CHARGE_GROUP": "FIZIKI",
        "CUSTOMER_ADDRESS": make_address(city),
        "CUST_SUB_SEGMENT": (
            "AFFLUENT"
            if is_affluent == "Y"
            else "MASS"
        ),
        "ONLY_LAST_AND_FIRST_NAME_ENG": english_name,
        "EMPLOYER_ID": (
            f"EMP{random.randint(1000, 9999)}"
            if is_employee == "Y"
            else ""
        ),
        "CUST_CLASSIFICATION": (
            "VIP"
            if is_vip == "Y"
            else "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
    })

    return row


def generate_company(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    company_sequence: int
) -> dict:
    company_name = (
        f"{random.choice(COMPANY_PREFIXES)} "
        f"{random.choice(COMPANY_ACTIVITIES)} "
        f"{random.choice(COMPANY_SUFFIXES)} "
        f"{company_sequence:04d}"
    )

    incorporation_date = random_date(
        date(1995, 1, 1),
        first_contract_date - timedelta(days=30)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = random.choice(CITIES)

    employee_count = random.choices(
        [
            random.randint(5, 49),
            random.randint(50, 249),
            random.randint(250, 3000)
        ],
        weights=[60, 32, 8],
        k=1
    )[0]

    is_affluent = (
        "Y"
        if employee_count >= 250 or random.random() < 0.18
        else "N"
    )

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "C",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["SME/C", "CORPORATE"],
            weights=[72, 28],
            k=1
        )[0],
        "FULL_NAME": company_name,
        "OPEN_REASON": "BUSINESS",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "LEGAL_ENTITY",
        "NUM_OF_EMPLOYEER": str(employee_count),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": is_affluent,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "LEGAL_ENTITY",
        "SHORT_NAME": company_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "TAX_CERTIFICATE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": company_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": company_name,
        "TAX_TYPE": "LEGAL_ENTITY",
        "CUST_MIS_1": "CORP",
        "SHORT_NAME2": company_name[:20].upper(),
        "CHARGE_GROUP": "KORP",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": (
            "LARGE_CORP"
            if employee_count >= 250
            else "SME"
        ),
        "INCORP_DATE": format_date(incorporation_date),
        "DT_QHT_GROUP": f"GRP_{random.randint(1, 300):04d}",
        "CUST_CLASSIFICATION": (
            "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "FLG_IS_RELATED_PERSON": (
            "Y"
            if random.random() < 0.03
            else "N"
        ),
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"GRP{random.randint(1, 300):04d}",
    })

    return row


def generate_bank(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    bank_sequence: int
) -> dict:
    bank_name = (
        f"{random.choice(['Caspian', 'Azer', 'Shirvan', 'Atlas'])} "
        f"Bank ASC {bank_sequence:03d}"
    )

    incorporation_date = random_date(
        date(1992, 1, 1),
        first_contract_date - timedelta(days=365)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = "BAKI"

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "B",
        "CUSTOMER_SUB_TYPE": "BANK",
        "FULL_NAME": bank_name,
        "OPEN_REASON": "CORRESPONDENT",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "BANK",
        "NUM_OF_EMPLOYEER": str(
            random.randint(200, 2500)
        ),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": "Y",
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "BANK",
        "SHORT_NAME": bank_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "BANK_LICENSE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": bank_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": bank_name,
        "TAX_TYPE": "FINANCIAL_INSTITUTION",
        "CUST_MIS_1": "BANK",
        "SHORT_NAME2": bank_name[:20].upper(),
        "SWIFT_CD": random.choice(SWIFT_CODES),
        "CHARGE_GROUP": "BANK",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": "BANK",
        "INCORP_DATE": format_date(incorporation_date),
        "CUST_CLASSIFICATION": "PREMIUM",
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"BNK{bank_sequence:03d}",
    })

    return row


# ============================================================
# VALIDATION
# ============================================================

def validate_customer_rows(
    customer_rows: list,
    contract_customer_ids: set
) -> None:
    customer_numbers = [
        row["CUSTOMER_NO"]
        for row in customer_rows
    ]

    individual_pins = [
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] == "I"
    ]

    if len(customer_rows) != NUMBER_OF_CUSTOMERS:
        raise ValueError("Customer row count is incorrect.")

    if len(customer_numbers) != len(set(customer_numbers)):
        raise ValueError("Duplicate CUSTOMER_NO / CIF values exist.")

    if any(not customer_no for customer_no in customer_numbers):
        raise ValueError("Blank CUSTOMER_NO / CIF values exist.")

    if len(individual_pins) != len(set(individual_pins)):
        raise ValueError("Duplicate individual PIN values exist.")

    if any(not pin for pin in individual_pins):
        raise ValueError("An individual customer has no PIN.")

    if any(
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] in {"C", "B"}
    ):
        raise ValueError(
            "A company or bank incorrectly has a PIN."
        )

    missing_customers = (
        contract_customer_ids - set(customer_numbers)
    )

    if missing_customers:
        raise ValueError(
            "Some contract CUSTOMER_ID values have no "
            "matching CUSTOMER_NO."
        )


def write_csv(file_path: Path, rows: list) -> None:
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=CUSTOMER_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main():
    (
        contract_customer_ids,
        first_contract_date_by_customer,
        product_codes_by_customer
    ) = load_contract_customer_details()

    customer_numbers = build_customer_numbers(
        contract_customer_ids
    )

    existing_pins = set()
    existing_documents = set()
    existing_tax_ids = set()

    customer_rows = []

    individual_count = 0
    corporate_count = 0
    bank_count = 0

    for row_number, customer_no in enumerate(
        customer_numbers,
        start=1
    ):
        first_contract_date = first_contract_date_by_customer.get(
            customer_no,
            date(2026, 5, 31)
        )

        customer_products = product_codes_by_customer.get(
            customer_no,
            set()
        )

        customer_type = choose_customer_type(customer_products)

        row = make_base_row(
            customer_no=customer_no,
            generated_id=100_000_000 + row_number
        )

        if customer_type == "I":
            individual_count += 1

            row = generate_individual(
                row=row,
                first_contract_date=first_contract_date,
                existing_pins=existing_pins,
                existing_documents=existing_documents
            )

        elif customer_type == "C":
            corporate_count += 1

            row = generate_company(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                company_sequence=corporate_count
            )

        else:
            bank_count += 1

            row = generate_bank(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                bank_sequence=bank_count
            )

        customer_rows.append(row)

    validate_customer_rows(
        customer_rows=customer_rows,
        contract_customer_ids=contract_customer_ids
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_customers_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER / "rs_customers_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER / "rs_customers_generation_summary.json"
    )

    write_csv(full_output_file, customer_rows)
    write_csv(sample_output_file, customer_rows[:100])

    summary = {
        "table_name": "RS_CUSTOMERS",
        "generated_rows": len(customer_rows),
        "contract_customer_ids_found": len(contract_customer_ids),
        "individual_customers": individual_count,
        "corporate_customers": corporate_count,
        "bank_customers": bank_count,
        "unique_customer_numbers": len(
            {row["CUSTOMER_NO"] for row in customer_rows}
        ),
        "unique_individual_pins": len(existing_pins),
        "contract_customer_ids_without_match": 0,
        "join_rule": (
            "RS_CONTRACTS.CUSTOMER_ID "
            "= RS_CUSTOMERS.CUSTOMER_NO"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Customer generation completed successfully.")
    print(f"Generated customers: {len(customer_rows):,}")
    print(f"Individuals: {individual_count:,}")
    print(f"Corporate customers: {corporate_count:,}")
    print(f"Banks: {bank_count:,}")
    print(
        "Contract-to-customer unmatched IDs: 0"
    )
    print(f"Full dataset: {full_output_file}")
    print(f"Sample dataset: {sample_output_file}")


if __name__ == "__main__":
    main()

Customer generation completed successfully.
Generated customers: 15,000
Individuals: 11,363
Corporate customers: 3,482
Banks: 155
Contract-to-customer unmatched IDs: 0
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_customers_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_customers_base_sample.csv


In [1]:
import csv
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260626
NUMBER_OF_MKR_RECORDS = 40_000

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

# This works inside your Jupyter notebook.
PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# MKR MASTER TABLE COLUMNS
# ============================================================

MKR_MASTER_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "BALANCE",
    "REPORTID",
    "REPORTINGDATE",
    "INSERT_DATE",
    "DOCUMENT_NO"
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(value.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between start_date and end_date."""

    if start_date >= end_date:
        return start_date

    day_difference = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, day_difference)
    )


def get_customer_type(customer_row):
    """
    I = individual
    C = company
    B = bank

    If CUSTOMER_TYPE is unexpectedly blank,
    PIN presence is used as a fallback rule.
    """

    customer_type = (
        customer_row.get("CUSTOMER_TYPE") or ""
    ).strip().upper()

    if customer_type in {"I", "C", "B"}:
        return customer_type

    if (customer_row.get("PIN") or "").strip():
        return "I"

    return "C"


def generate_balance(customer_type):
    """
    Creates a realistic total reported liability balance.

    Later, when we create the liabilities child table,
    liability balances will sum exactly to this value.
    """

    if customer_type == "I":
        # Some people may have no active liability on the inquiry date.
        if random.random() < 0.18:
            return 0.00

        return round(
            random.triangular(
                low=200,
                high=30_000,
                mode=4_000
            ),
            2
        )

    if customer_type == "C":
        # Companies normally carry higher balances.
        if random.random() < 0.08:
            return 0.00

        return round(
            random.triangular(
                low=5_000,
                high=2_000_000,
                mode=180_000
            ),
            2
        )

    # Banks / financial institutions.
    if random.random() < 0.05:
        return 0.00

    return round(
        random.triangular(
            low=500_000,
            high=8_000_000,
            mode=2_000_000
        ),
        2
    )


def load_customers():
    """
    Reads the customer master dataset and returns:
    - customers_by_cif: dictionary where CIF is the key
    - customer_rows: all customer records
    """

    if not CUSTOMER_FILE.exists():
        raise FileNotFoundError(
            f"Customer file was not found:\n{CUSTOMER_FILE}\n\n"
            "Run the customer-generation cell first."
        )

    customers_by_cif = {}
    customer_rows = []

    with open(
        CUSTOMER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_NO",
            "CUSTOMER_TYPE",
            "PIN",
            "CREATION_DT"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "Missing customer columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_no = (
                row.get("CUSTOMER_NO") or ""
            ).strip()

            if not customer_no:
                continue

            customers_by_cif[customer_no] = row
            customer_rows.append(row)

    return customers_by_cif, customer_rows


def create_inquiry_customer_list(customer_rows):
    """
    Creates exactly 40,000 inquiry records.

    Every one of the 15,000 customers appears at least once.
    The remaining 25,000 inquiries are distributed across customers,
    so some customers have multiple credit-history inquiries.
    """

    if len(customer_rows) > NUMBER_OF_MKR_RECORDS:
        raise ValueError(
            "There are more customers than the requested "
            "number of MKR records."
        )

    inquiry_customers = customer_rows.copy()

    additional_inquiry_count = (
        NUMBER_OF_MKR_RECORDS - len(customer_rows)
    )

    weights = []

    for customer_row in customer_rows:
        customer_type = get_customer_type(customer_row)

        # Individuals are more likely to make repeated applications.
        if customer_type == "I":
            weights.append(1.8)

        elif customer_type == "C":
            weights.append(1.3)

        else:
            weights.append(0.7)

    additional_customers = random.choices(
        population=customer_rows,
        weights=weights,
        k=additional_inquiry_count
    )

    inquiry_customers.extend(additional_customers)

    # Mix the records so repeated inquiries are not next to each other.
    random.shuffle(inquiry_customers)

    return inquiry_customers


def create_mkr_row(customer_row, sequence_number):
    """
    Creates one MKR inquiry row.

    Individual:
        PIN_CODE is filled
        DOCUMENT_NO is blank

    Company / Bank:
        PIN_CODE is blank
        DOCUMENT_NO contains CUSTOMER_NO / CIF
    """

    customer_type = get_customer_type(customer_row)

    customer_creation_date = parse_date(
        customer_row.get("CREATION_DT") or ""
    )

    reporting_start_date = max(
        customer_creation_date or date(2018, 1, 1),
        date(2018, 1, 1)
    )

    reporting_end_date = date(2026, 6, 20)

    reporting_date = random_date(
        start_date=reporting_start_date,
        end_date=reporting_end_date
    )

    insert_date = reporting_date + timedelta(
        days=random.randint(0, 5)
    )

    search_id = (
        f"MKRM{reporting_date:%y}{sequence_number:010d}"
    )

    report_id = (
        f"RPT{reporting_date:%y%m}{sequence_number:09d}"
    )

    if customer_type == "I":
        pin_code = (customer_row.get("PIN") or "").strip()
        document_no = ""

        if not pin_code:
            raise ValueError(
                "An individual customer has no PIN. "
                f"CIF: {customer_row.get('CUSTOMER_NO')}"
            )

    else:
        pin_code = ""
        document_no = (
            customer_row.get("CUSTOMER_NO") or ""
        ).strip()

        if not document_no:
            raise ValueError(
                "A company/bank has no CUSTOMER_NO / CIF."
            )

    balance = generate_balance(customer_type)

    return {
        "SEARCH_ID": search_id,
        "PIN_CODE": pin_code,
        "BALANCE": f"{balance:.2f}",
        "REPORTID": report_id,
        "REPORTINGDATE": reporting_date.isoformat(),
        "INSERT_DATE": insert_date.isoformat(),
        "DOCUMENT_NO": document_no
    }


def validate_mkr_rows(mkr_rows, customers_by_cif):
    """Checks the row count, keys, and customer-table join logic."""

    search_ids = [
        row["SEARCH_ID"]
        for row in mkr_rows
    ]

    report_ids = [
        row["REPORTID"]
        for row in mkr_rows
    ]

    customer_pins = {
        (customer.get("PIN") or "").strip()
        for customer in customers_by_cif.values()
        if get_customer_type(customer) == "I"
    }

    customer_cifs_for_companies = {
        (customer.get("CUSTOMER_NO") or "").strip()
        for customer in customers_by_cif.values()
        if get_customer_type(customer) in {"C", "B"}
    }

    if len(mkr_rows) != NUMBER_OF_MKR_RECORDS:
        raise ValueError(
            f"Expected {NUMBER_OF_MKR_RECORDS:,} records, "
            f"but generated {len(mkr_rows):,}."
        )

    if len(search_ids) != len(set(search_ids)):
        raise ValueError("Duplicate SEARCH_ID values were generated.")

    if len(report_ids) != len(set(report_ids)):
        raise ValueError("Duplicate REPORTID values were generated.")

    for row in mkr_rows:
        pin_code = row["PIN_CODE"]
        document_no = row["DOCUMENT_NO"]

        if pin_code and document_no:
            raise ValueError(
                "One MKR record incorrectly contains both "
                "PIN_CODE and DOCUMENT_NO."
            )

        if not pin_code and not document_no:
            raise ValueError(
                "One MKR record contains neither PIN_CODE "
                "nor DOCUMENT_NO."
            )

        if pin_code and pin_code not in customer_pins:
            raise ValueError(
                f"PIN_CODE {pin_code} does not match a customer."
            )

        if document_no and document_no not in customer_cifs_for_companies:
            raise ValueError(
                f"DOCUMENT_NO {document_no} does not match "
                "a company/bank CIF."
            )


def write_csv(file_path, rows):
    """Writes data to a CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_MASTER_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_master():
    customers_by_cif, customer_rows = load_customers()

    inquiry_customers = create_inquiry_customer_list(
        customer_rows
    )

    mkr_rows = []

    for sequence_number, customer_row in enumerate(
        inquiry_customers,
        start=1
    ):
        mkr_rows.append(
            create_mkr_row(
                customer_row=customer_row,
                sequence_number=sequence_number
            )
        )

    validate_mkr_rows(
        mkr_rows=mkr_rows,
        customers_by_cif=customers_by_cif
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_master_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER / "rs_mkr_master_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_master_generation_summary.json"
    )

    write_csv(full_output_file, mkr_rows)
    write_csv(sample_output_file, mkr_rows[:100])

    individual_inquiries = sum(
        1 for row in mkr_rows
        if row["PIN_CODE"]
    )

    corporate_or_bank_inquiries = sum(
        1 for row in mkr_rows
        if row["DOCUMENT_NO"]
    )

    summary = {
        "table_name": "RS_MKR_MASTER",
        "generated_rows": len(mkr_rows),
        "unique_search_ids": len(
            {row["SEARCH_ID"] for row in mkr_rows}
        ),
        "unique_report_ids": len(
            {row["REPORTID"] for row in mkr_rows}
        ),
        "individual_inquiries_joined_by_pin": individual_inquiries,
        "company_bank_inquiries_joined_by_cif": (
            corporate_or_bank_inquiries
        ),
        "join_rules": {
            "individual": (
                "RS_MKR_MASTER.PIN_CODE = RS_CUSTOMERS.PIN"
            ),
            "company_or_bank": (
                "RS_MKR_MASTER.DOCUMENT_NO "
                "= RS_CUSTOMERS.CUSTOMER_NO"
            )
        },
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR master generation completed successfully.")
    print(f"Generated inquiry rows: {len(mkr_rows):,}")
    print(f"Individual inquiries by PIN: {individual_inquiries:,}")
    print(
        "Company/bank inquiries by CIF: "
        f"{corporate_or_bank_inquiries:,}"
    )
    print("Unmatched customer keys: 0")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_master()

MKR master generation completed successfully.
Generated inquiry rows: 40,000
Individual inquiries by PIN: 31,616
Company/bank inquiries by CIF: 8,384
Unmatched customer keys: 0
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_master_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_master_base_sample.csv


In [2]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260626
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# LIABILITIES TABLE COLUMNS
# ============================================================

MKR_LIABILITY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "ACCOUNTNO",
    "BANKID",
    "BANKNAME",
    "COLLATERALANYINFO",
    "COLLATERALCODE",
    "COLLATERALMARKETVALUE",
    "COLLATERALREGISTRYAGENCY",
    "COLLATERALREGISTRYDATE",
    "COLLATERALREGISTRYNO",
    "COLLATERALTYPENAME",
    "CREDITPURPOSE",
    "CREDITPURPOSENAME",
    "CREDITSTATUS",
    "CREDITSTATUSCLOSEDATE",
    "CREDITSTATUSNAME",
    "CREDITTYPE",
    "CREDITTYPENAME",
    "CURRENCY",
    "CURRENCYNAME",
    "DAYSINTERESTOVERDUE",
    "DAYSMAINSUMOVERDUE",
    "GRANTEDON",
    "ID",
    "INITIALAMOUNT",
    "INTERESTRATE",
    "LASTPAYMENTDATE",
    "LASTUPDATEDDATE",
    "LINEAMOUNT",
    "MKRID",
    "MONTHLYPAYMENTAMOUNT",
    "ORGIDTYPE",
    "OUTSTANDINGDEBTINTEREST",
    "OUTSTANDINGDEBTMAIN",
    "PROLONGATIONS",
    "CONTRACTDUEON",
    "INS_DTTM"
]


# ============================================================
# REFERENCE DATA
# ============================================================

BANKS = [
    ("001", "Azer Finance Bank"),
    ("002", "Caspian Commercial Bank"),
    ("003", "Baku Credit Bank"),
    ("004", "Shirvan Bank"),
    ("005", "Khazar Finance"),
    ("006", "Atlas Bank"),
    ("007", "Regional Development Bank")
]

INDIVIDUAL_CREDIT_TYPES = [
    {
        "code": "001",
        "name": "Consumer Loan",
        "purpose_code": "001",
        "purpose_name": "Consumer Expenses",
        "term_min": 12,
        "term_max": 60,
        "secured": False,
        "line": False
    },
    {
        "code": "002",
        "name": "Mortgage Loan",
        "purpose_code": "002",
        "purpose_name": "Property Purchase",
        "term_min": 60,
        "term_max": 240,
        "secured": True,
        "collateral_type": "Residential Property",
        "collateral_code": "001",
        "line": False
    },
    {
        "code": "003",
        "name": "Auto Loan",
        "purpose_code": "003",
        "purpose_name": "Vehicle Purchase",
        "term_min": 24,
        "term_max": 84,
        "secured": True,
        "collateral_type": "Vehicle",
        "collateral_code": "002",
        "line": False
    },
    {
        "code": "004",
        "name": "Credit Card",
        "purpose_code": "004",
        "purpose_name": "Card Spending",
        "term_min": 12,
        "term_max": 48,
        "secured": False,
        "line": True
    }
]

CORPORATE_CREDIT_TYPES = [
    {
        "code": "005",
        "name": "SME Business Loan",
        "purpose_code": "005",
        "purpose_name": "Business Development",
        "term_min": 12,
        "term_max": 84,
        "secured": True,
        "collateral_type": "Commercial Property",
        "collateral_code": "003",
        "line": False
    },
    {
        "code": "006",
        "name": "Business Credit Line",
        "purpose_code": "006",
        "purpose_name": "Working Capital",
        "term_min": 12,
        "term_max": 60,
        "secured": False,
        "line": True
    },
    {
        "code": "007",
        "name": "Equipment Financing",
        "purpose_code": "007",
        "purpose_name": "Equipment Purchase",
        "term_min": 24,
        "term_max": 72,
        "secured": True,
        "collateral_type": "Equipment",
        "collateral_code": "007",
        "line": False
    },
    {
        "code": "008",
        "name": "Commercial Mortgage",
        "purpose_code": "008",
        "purpose_name": "Commercial Property Purchase",
        "term_min": 60,
        "term_max": 180,
        "secured": True,
        "collateral_type": "Commercial Property",
        "collateral_code": "003",
        "line": False
    }
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between two dates."""

    if start_date >= end_date:
        return start_date

    number_of_days = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, number_of_days)
    )


def to_decimal(value):
    """Converts a numeric text value into Decimal safely."""

    try:
        return Decimal(str(value)).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

    except Exception:
        return Decimal("0.00")


def format_decimal(value):
    """Formats Decimal as text with exactly two decimals."""

    return format(
        value.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        ),
        "f"
    )


def choose_liability_count(balance, is_individual):
    """
    Determines how many child liabilities one inquiry has.

    Small balances usually have one liability.
    Larger balances can have several liabilities.
    """

    if balance < Decimal("1000"):
        return 1

    if is_individual:
        if balance < Decimal("5000"):
            return random.choices(
                [1, 2],
                weights=[75, 25],
                k=1
            )[0]

        return random.choices(
            [1, 2, 3, 4],
            weights=[30, 40, 22, 8],
            k=1
        )[0]

    return random.choices(
        [1, 2, 3, 4, 5],
        weights=[15, 30, 30, 18, 7],
        k=1
    )[0]


def split_balance(total_balance, number_of_liabilities):
    """
    Splits one MKR master BALANCE into child liability balances.

    The final child balances always sum exactly to the parent BALANCE.
    """

    total_cents = int(total_balance * 100)

    if number_of_liabilities == 1:
        return [total_cents]

    minimum_cents_per_liability = 100
    minimum_required = (
        number_of_liabilities
        * minimum_cents_per_liability
    )

    if total_cents <= minimum_required:
        return [total_cents]

    remaining_cents = total_cents - minimum_required

    weights = [
        random.uniform(0.4, 1.6)
        for _ in range(number_of_liabilities)
    ]

    total_weight = sum(weights)

    allocations = [
        minimum_cents_per_liability
        + int(
            remaining_cents
            * weight
            / total_weight
        )
        for weight in weights
    ]

    difference = total_cents - sum(allocations)

    allocations[-1] += difference

    return allocations


def choose_credit_status():
    """
    Generates a realistic active-liability status.

    All generated child liabilities have an outstanding balance,
    therefore closed status is not used here.
    """

    return random.choices(
        [
            ("001", "Active", 0),
            ("005", "Past Due", random.randint(1, 180)),
            ("007", "Restructured", random.randint(0, 60))
        ],
        weights=[78, 16, 6],
        k=1
    )[0]


def choose_credit_type(is_individual):
    """Chooses an appropriate credit type."""

    if is_individual:
        return random.choices(
            INDIVIDUAL_CREDIT_TYPES,
            weights=[44, 12, 17, 27],
            k=1
        )[0]

    return random.choices(
        CORPORATE_CREDIT_TYPES,
        weights=[38, 28, 17, 17],
        k=1
    )[0]


def generate_credit_dates(
    reporting_date,
    status_code,
    credit_type
):
    """
    Creates dates that make business sense:
    - active credit normally ends after reporting date
    - past-due credit may be near/after maturity
    """

    term_months = random.randint(
        credit_type["term_min"],
        credit_type["term_max"]
    )

    term_days = term_months * 30

    if status_code == "001":
        maximum_age = max(30, term_days - 30)

        granted_on = reporting_date - timedelta(
            days=random.randint(30, maximum_age)
        )

    else:
        granted_on = reporting_date - timedelta(
            days=random.randint(
                term_days - 60,
                term_days + 730
            )
        )

    contract_due_on = granted_on + timedelta(
        days=term_days
    )

    last_payment_start = max(
        granted_on,
        reporting_date - timedelta(days=180)
    )

    last_payment_date = random_date(
        last_payment_start,
        reporting_date
    )

    return (
        granted_on,
        contract_due_on,
        last_payment_date
    )


def generate_account_number(owner_key, liability_slot):
    """
    Creates a repeated external-account identifier for a customer.

    The same customer can see the same liability account
    in multiple later MKR inquiries.
    """

    safe_owner_key = (
        owner_key.replace(" ", "")
        .replace("-", "")
        .upper()
    )

    return f"EXT{safe_owner_key[-12:]}{liability_slot:02d}"


# ============================================================
# INPUT DATA
# ============================================================

def load_mkr_master():
    """Loads the parent MKR master file."""

    if not MKR_MASTER_FILE.exists():
        raise FileNotFoundError(
            f"MKR master file was not found:\n{MKR_MASTER_FILE}\n\n"
            "Run the MKR master generation cell first."
        )

    master_rows = []

    with open(
        MKR_MASTER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "SEARCH_ID",
            "PIN_CODE",
            "BALANCE",
            "REPORTINGDATE",
            "INSERT_DATE",
            "DOCUMENT_NO"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "MKR master is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            master_rows.append(row)

    return master_rows


# ============================================================
# ROW GENERATION
# ============================================================

def create_liability_rows_for_master(
    master_row,
    starting_id
):
    """
    Creates zero or more liabilities for one MKR inquiry.

    Zero BALANCE means the customer had no reported active liabilities,
    so no child liability rows are created.
    """

    search_id = (
        master_row.get("SEARCH_ID") or ""
    ).strip()

    pin_code = (
        master_row.get("PIN_CODE") or ""
    ).strip()

    document_no = (
        master_row.get("DOCUMENT_NO") or ""
    ).strip()

    reporting_date = parse_date(
        master_row.get("REPORTINGDATE") or ""
    )

    insert_date = parse_date(
        master_row.get("INSERT_DATE") or ""
    )

    parent_balance = to_decimal(
        master_row.get("BALANCE") or "0"
    )

    if not search_id:
        raise ValueError("MKR master contains blank SEARCH_ID.")

    if not reporting_date:
        raise ValueError(
            f"Invalid REPORTINGDATE for SEARCH_ID: {search_id}"
        )

    if parent_balance <= Decimal("0.00"):
        return []

    is_individual = bool(pin_code)

    owner_key = pin_code if is_individual else document_no

    if not owner_key:
        raise ValueError(
            f"SEARCH_ID {search_id} has no PIN_CODE "
            "and no DOCUMENT_NO."
        )

    liability_count = choose_liability_count(
        balance=parent_balance,
        is_individual=is_individual
    )

    balance_allocations = split_balance(
        total_balance=parent_balance,
        number_of_liabilities=liability_count
    )

    liability_rows = []

    for liability_slot, total_debt_cents in enumerate(
        balance_allocations,
        start=1
    ):
        total_debt = (
            Decimal(total_debt_cents)
            / Decimal("100")
        )

        (
            status_code,
            status_name,
            overdue_days
        ) = choose_credit_status()

        credit_type = choose_credit_type(
            is_individual=is_individual
        )

        bank_id, bank_name = random.choice(BANKS)

        (
            granted_on,
            contract_due_on,
            last_payment_date
        ) = generate_credit_dates(
            reporting_date=reporting_date,
            status_code=status_code,
            credit_type=credit_type
        )

        if overdue_days == 0:
            interest_ratio = Decimal(
                str(random.uniform(0.01, 0.08))
            )
        else:
            interest_ratio = Decimal(
                str(random.uniform(0.05, 0.20))
            )

        interest_debt = (
            total_debt * interest_ratio
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        main_debt = total_debt - interest_debt

        initial_multiplier = Decimal(
            str(random.uniform(1.10, 3.20))
        )

        initial_amount = (
            total_debt * initial_multiplier
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        interest_rate = round(
            random.uniform(9.0, 28.0),
            2
        )

        if credit_type["line"]:
            line_amount = (
                initial_amount * Decimal(
                    str(random.uniform(1.05, 1.40))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            monthly_payment = (
                total_debt * Decimal(
                    str(random.uniform(0.05, 0.15))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

        else:
            line_amount = Decimal("0.00")

            monthly_payment = max(
                Decimal("20.00"),
                (
                    total_debt * Decimal(
                        str(random.uniform(0.03, 0.12))
                    )
                ).quantize(
                    Decimal("0.01"),
                    rounding=ROUND_HALF_UP
                )
            )

        is_secured = credit_type.get(
            "secured",
            False
        )

        if is_secured:
            collateral_value = (
                initial_amount * Decimal(
                    str(random.uniform(1.05, 1.50))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            collateral_code = credit_type.get(
                "collateral_code",
                "007"
            )

            collateral_type = credit_type.get(
                "collateral_type",
                "Other Collateral"
            )

            collateral_registry_date = (
                granted_on
                + timedelta(days=random.randint(0, 30))
            )

            collateral_registry_no = (
                f"COL{granted_on:%y}{starting_id + liability_slot:09d}"
            )

            collateral_registry_agency = (
                "STATE COLLATERAL REGISTRY"
            )

            collateral_any_info = (
                f"{collateral_type.upper()} SECURES CREDIT"
            )

        else:
            collateral_value = Decimal("0.00")
            collateral_code = "000"
            collateral_type = ""
            collateral_registry_date = ""
            collateral_registry_no = ""
            collateral_registry_agency = ""
            collateral_any_info = ""

        record_id = str(
            starting_id + liability_slot
        )

        updated_date = reporting_date

        created_timestamp_date = (
            insert_date
            if insert_date
            else reporting_date
        )

        created_timestamp = datetime.combine(
            created_timestamp_date,
            datetime.min.time()
        ) + timedelta(
            hours=random.randint(8, 19),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        liability_rows.append({
            "SEARCH_ID": search_id,
            "PIN_CODE": pin_code,
            "ACCOUNTNO": generate_account_number(
                owner_key=owner_key,
                liability_slot=liability_slot
            ),
            "BANKID": bank_id,
            "BANKNAME": bank_name,
            "COLLATERALANYINFO": collateral_any_info,
            "COLLATERALCODE": collateral_code,
            "COLLATERALMARKETVALUE": format_decimal(
                collateral_value
            ),
            "COLLATERALREGISTRYAGENCY": (
                collateral_registry_agency
            ),
            "COLLATERALREGISTRYDATE": (
                collateral_registry_date.isoformat()
                if collateral_registry_date
                else ""
            ),
            "COLLATERALREGISTRYNO": collateral_registry_no,
            "COLLATERALTYPENAME": collateral_type,
            "CREDITPURPOSE": credit_type["purpose_code"],
            "CREDITPURPOSENAME": credit_type["purpose_name"],
            "CREDITSTATUS": status_code,
            "CREDITSTATUSCLOSEDATE": "",
            "CREDITSTATUSNAME": status_name,
            "CREDITTYPE": credit_type["code"],
            "CREDITTYPENAME": credit_type["name"],
            "CURRENCY": "AZN",
            "CURRENCYNAME": "Azerbaijani Manat",
            "DAYSINTERESTOVERDUE": str(overdue_days),
            "DAYSMAINSUMOVERDUE": str(overdue_days),
            "GRANTEDON": granted_on.isoformat(),
            "ID": record_id,
            "INITIALAMOUNT": format_decimal(initial_amount),
            "INTERESTRATE": str(interest_rate),
            "LASTPAYMENTDATE": last_payment_date.isoformat(),
            "LASTUPDATEDDATE": updated_date.isoformat(),
            "LINEAMOUNT": format_decimal(line_amount),
            "MKRID": f"MKR-{record_id}",
            "MONTHLYPAYMENTAMOUNT": format_decimal(
                monthly_payment
            ),
            "ORGIDTYPE": "001",
            "OUTSTANDINGDEBTINTEREST": format_decimal(
                interest_debt
            ),
            "OUTSTANDINGDEBTMAIN": format_decimal(
                main_debt
            ),
            "PROLONGATIONS": str(
                random.randint(1, 3)
                if status_code == "007"
                else random.choices(
                    [0, 1],
                    weights=[92, 8],
                    k=1
                )[0]
            ),
            "CONTRACTDUEON": contract_due_on.isoformat(),
            "INS_DTTM": created_timestamp.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        })

    return liability_rows


# ============================================================
# VALIDATION
# ============================================================

def validate_liabilities(
    master_rows,
    liability_rows
):
    """
    Validates:
    1. Every child SEARCH_ID exists in MKR master.
    2. Liability PIN matches the parent inquiry PIN.
    3. Child principal + interest sums to parent BALANCE.
    4. Every positive-balance parent has at least one liability.
    """

    master_by_search_id = {
        row["SEARCH_ID"].strip(): row
        for row in master_rows
    }

    balances_by_search_id = defaultdict(
        lambda: Decimal("0.00")
    )

    liability_search_ids = set()
    liability_ids = set()

    for row in liability_rows:
        search_id = row["SEARCH_ID"].strip()

        if search_id not in master_by_search_id:
            raise ValueError(
                f"Liability SEARCH_ID does not exist in master: "
                f"{search_id}"
            )

        parent_pin = (
            master_by_search_id[search_id]
            .get("PIN_CODE", "")
            .strip()
        )

        if row["PIN_CODE"].strip() != parent_pin:
            raise ValueError(
                f"PIN_CODE mismatch for SEARCH_ID: {search_id}"
            )

        if row["ID"] in liability_ids:
            raise ValueError(
                f"Duplicate liability ID found: {row['ID']}"
            )

        liability_ids.add(row["ID"])
        liability_search_ids.add(search_id)

        row_total = (
            to_decimal(row["OUTSTANDINGDEBTMAIN"])
            + to_decimal(row["OUTSTANDINGDEBTINTEREST"])
        )

        balances_by_search_id[search_id] += row_total

    for search_id, master_row in master_by_search_id.items():
        parent_balance = to_decimal(
            master_row.get("BALANCE") or "0"
        )

        child_balance = balances_by_search_id[search_id]

        if parent_balance > Decimal("0.00"):
            if search_id not in liability_search_ids:
                raise ValueError(
                    "Positive-balance MKR master has no child "
                    f"liability: {search_id}"
                )

            if child_balance != parent_balance:
                raise ValueError(
                    f"Balance mismatch for SEARCH_ID {search_id}. "
                    f"Master: {parent_balance}; "
                    f"Liabilities: {child_balance}"
                )

        else:
            if child_balance != Decimal("0.00"):
                raise ValueError(
                    f"Zero-balance MKR master has liabilities: "
                    f"{search_id}"
                )


def write_csv(file_path, rows):
    """Writes liabilities to CSV."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_LIABILITY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_liabilities():
    master_rows = load_mkr_master()

    liability_rows = []
    next_id = 1

    for master_row in master_rows:
        new_rows = create_liability_rows_for_master(
            master_row=master_row,
            starting_id=next_id
        )

        liability_rows.extend(new_rows)
        next_id += len(new_rows)

    validate_liabilities(
        master_rows=master_rows,
        liability_rows=liability_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_liabilities_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_liabilities_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liabilities_generation_summary.json"
    )

    write_csv(full_output_file, liability_rows)
    write_csv(sample_output_file, liability_rows[:100])

    unique_parent_search_ids = len(
        {row["SEARCH_ID"] for row in liability_rows}
    )

    total_master_records = len(master_rows)

    summary = {
        "table_name": "RS_MKR_LIABILITIES",
        "generated_rows": len(liability_rows),
        "unique_liability_ids": len(
            {row["ID"] for row in liability_rows}
        ),
        "master_search_ids": total_master_records,
        "master_search_ids_with_liabilities": (
            unique_parent_search_ids
        ),
        "join_rule": (
            "RS_MKR_LIABILITIES.SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "balance_rule": (
            "For every positive-balance SEARCH_ID, "
            "sum(OUTSTANDINGDEBTMAIN + "
            "OUTSTANDINGDEBTINTEREST) = "
            "RS_MKR_MASTER.BALANCE"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR liabilities generation completed successfully.")
    print(f"Generated liability rows: {len(liability_rows):,}")
    print(
        "MKR master search IDs with liabilities: "
        f"{unique_parent_search_ids:,}"
    )
    print(
        "MKR master search IDs without liabilities "
        "(zero balance): "
        f"{total_master_records - unique_parent_search_ids:,}"
    )
    print("Unmatched SEARCH_ID values: 0")
    print("Parent-to-child balance validation: passed")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_liabilities()

MKR liabilities generation completed successfully.
Generated liability rows: 70,921
MKR master search IDs with liabilities: 33,666
MKR master search IDs without liabilities (zero balance): 6,334
Unmatched SEARCH_ID values: 0
Parent-to-child balance validation: passed
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_liabilities_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_liabilities_base_sample.csv


In [3]:
import csv
import json
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

MKR_LIABILITIES_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_liabilities_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# LIABILITY HISTORY TABLE COLUMNS
# ============================================================

MKR_LIABILITY_HISTORY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "LIABILITY_ID",
    "OVERDUEDAYS",
    "REPORTINGPERIOD",
    "CREDIT_STATUS",
    "INS_DTTM"
]


# ============================================================
# FILE READING
# ============================================================

def load_csv_rows(file_path, required_columns, table_name):
    """Reads a CSV file and checks required columns."""

    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        available_columns = set(reader.fieldnames or [])
        missing_columns = required_columns - available_columns

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return list(reader)


# ============================================================
# HISTORY ROW CREATION
# ============================================================

def create_history_row(master_row, liability_row):
    """
    Creates one liability-history snapshot.

    Values are copied from the matching master and liability records
    so all reporting dates, overdue days, and statuses stay consistent.
    """

    return {
        "SEARCH_ID": liability_row["SEARCH_ID"].strip(),
        "PIN_CODE": liability_row.get("PIN_CODE", "").strip(),
        "LIABILITY_ID": liability_row["ID"].strip(),
        "OVERDUEDAYS": liability_row[
            "DAYSMAINSUMOVERDUE"
        ].strip(),
        "REPORTINGPERIOD": master_row[
            "REPORTINGDATE"
        ].strip(),
        "CREDIT_STATUS": liability_row[
            "CREDITSTATUS"
        ].strip(),
        "INS_DTTM": liability_row["INS_DTTM"].strip()
    }


# ============================================================
# VALIDATION
# ============================================================

def validate_and_create_history(master_rows, liability_rows):
    """
    Validates parent-child relationships, then creates history rows.

    Rules:
    1. Every liability SEARCH_ID exists in MKR master.
    2. Liability PIN_CODE equals the parent master PIN_CODE.
    3. SEARCH_ID + LIABILITY_ID is unique in history.
    4. Every liability receives exactly one history row.
    """

    master_by_search_id = {}

    for master_row in master_rows:
        search_id = master_row["SEARCH_ID"].strip()

        if not search_id:
            raise ValueError(
                "MKR master contains a blank SEARCH_ID."
            )

        if search_id in master_by_search_id:
            raise ValueError(
                f"Duplicate SEARCH_ID in MKR master: {search_id}"
            )

        master_by_search_id[search_id] = master_row

    history_rows = []
    history_keys = set()

    for liability_row in liability_rows:
        search_id = liability_row["SEARCH_ID"].strip()
        liability_id = liability_row["ID"].strip()

        if search_id not in master_by_search_id:
            raise ValueError(
                "Liability has no matching parent MKR master record. "
                f"SEARCH_ID: {search_id}"
            )

        if not liability_id:
            raise ValueError(
                f"Blank liability ID for SEARCH_ID: {search_id}"
            )

        master_row = master_by_search_id[search_id]

        master_pin = master_row.get(
            "PIN_CODE",
            ""
        ).strip()

        liability_pin = liability_row.get(
            "PIN_CODE",
            ""
        ).strip()

        if master_pin != liability_pin:
            raise ValueError(
                "PIN mismatch between MKR master and liability. "
                f"SEARCH_ID: {search_id}"
            )

        history_key = (search_id, liability_id)

        if history_key in history_keys:
            raise ValueError(
                "Duplicate SEARCH_ID + LIABILITY_ID combination: "
                f"{search_id} / {liability_id}"
            )

        history_keys.add(history_key)

        history_rows.append(
            create_history_row(
                master_row=master_row,
                liability_row=liability_row
            )
        )

    if len(history_rows) != len(liability_rows):
        raise ValueError(
            "History row count does not match liability row count."
        )

    return history_rows


# ============================================================
# FILE WRITING
# ============================================================

def write_csv(file_path, rows):
    """Writes rows into a CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_LIABILITY_HISTORY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_liability_history():
    master_rows = load_csv_rows(
        file_path=MKR_MASTER_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "REPORTINGDATE"
        },
        table_name="MKR master"
    )

    liability_rows = load_csv_rows(
        file_path=MKR_LIABILITIES_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "ID",
            "DAYSMAINSUMOVERDUE",
            "CREDITSTATUS",
            "INS_DTTM"
        },
        table_name="MKR liabilities"
    )

    history_rows = validate_and_create_history(
        master_rows=master_rows,
        liability_rows=liability_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liability_history_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_liability_history_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liability_history_generation_summary.json"
    )

    write_csv(full_output_file, history_rows)
    write_csv(sample_output_file, history_rows[:100])

    summary = {
        "table_name": "RS_MKR_LIABILITY_HISTORY",
        "generated_rows": len(history_rows),
        "mkr_liability_rows": len(liability_rows),
        "row_count_matches_liabilities": (
            len(history_rows) == len(liability_rows)
        ),
        "join_rule": (
            "RS_MKR_LIABILITY_HISTORY.SEARCH_ID "
            "= RS_MKR_LIABILITIES.SEARCH_ID "
            "AND RS_MKR_LIABILITY_HISTORY.LIABILITY_ID "
            "= RS_MKR_LIABILITIES.ID"
        ),
        "reporting_period_rule": (
            "REPORTINGPERIOD = RS_MKR_MASTER.REPORTINGDATE"
        ),
        "overdue_days_rule": (
            "OVERDUEDAYS = RS_MKR_LIABILITIES.DAYSMAINSUMOVERDUE"
        ),
        "credit_status_rule": (
            "CREDIT_STATUS = RS_MKR_LIABILITIES.CREDITSTATUS"
        )
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR liability history generation completed successfully.")
    print(f"Generated history rows: {len(history_rows):,}")
    print(
        "Liability-to-history row count match: passed"
    )
    print("Unmatched SEARCH_ID values: 0")
    print(
        "SEARCH_ID + LIABILITY_ID uniqueness validation: passed"
    )
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_liability_history()

MKR liability history generation completed successfully.
Generated history rows: 70,921
Liability-to-history row count match: passed
Unmatched SEARCH_ID values: 0
SEARCH_ID + LIABILITY_ID uniqueness validation: passed
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_liability_history_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_liability_history_base_sample.csv


In [4]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260627
LOOKBACK_DAYS = 730

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
)

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# INQUIRY HISTORY TABLE COLUMNS
# ============================================================

MKR_INQUIRY_HISTORY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "INQBANKID",
    "INQBANKNAME",
    "INQDATE",
    "INQORGIDTYPE",
    "INQPURPOSEID",
    "INQPURPOSENAME",
    "INS_DTTM",
    "DOCUMENTNO",
    "REPORTINGDATE"
]


# ============================================================
# REFERENCE DATA
# ============================================================

INQUIRY_BANKS = [
    ("481", "Azərbaycan Beynəlxalq Bankı ASC", "001"),
    ("001", "XXXX", "001"),
    ("002", "XXXX", "001"),
    ("003", "XXXX", "001"),
    ("004", "XXXX", "001"),
    ("005", "", "002"),
    ("006", "", "002"),
    ("007", "", "003")
]

INQUIRY_PURPOSES_INDIVIDUAL = [
    ("001", "Consumer Loan"),
    ("002", "Mortgage Loan"),
    ("003", "Auto Loan"),
    ("004", "Credit Card"),
    ("006", "Credit Line"),
    ("009", "Other")
]

INQUIRY_PURPOSES_CORPORATE = [
    ("005", "Business Loan"),
    ("006", "Working Capital Credit Line"),
    ("007", "Leasing"),
    ("008", "Guarantee"),
    ("009", "Other")
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between two dates."""

    if start_date >= end_date:
        return start_date

    days_between = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, days_between)
    )


def get_owner_key(pin_code, document_no):
    """
    Creates one consistent customer key.

    Individual:
        P:<PIN>

    Company / bank:
        D:<CIF>
    """

    pin_code = (pin_code or "").strip()
    document_no = (document_no or "").strip()

    if pin_code:
        return f"P:{pin_code}"

    if document_no:
        return f"D:{document_no}"

    raise ValueError(
        "Neither PIN_CODE nor DOCUMENT_NO is available."
    )


def choose_inquiry_event(is_individual, inquiry_date, event_number):
    """
    Creates one actual historical inquiry event.

    This event can appear in the history of multiple later
    MKR master searches for the same customer.
    """

    bank_id, bank_name, org_type = random.choice(
        INQUIRY_BANKS
    )

    if is_individual:
        purpose_id, purpose_name = random.choices(
            INQUIRY_PURPOSES_INDIVIDUAL,
            weights=[36, 10, 14, 22, 12, 6],
            k=1
        )[0]

    else:
        purpose_id, purpose_name = random.choices(
            INQUIRY_PURPOSES_CORPORATE,
            weights=[42, 28, 12, 8, 10],
            k=1
        )[0]

    return {
        "event_id": f"INQ_EVENT_{event_number:09d}",
        "inquiry_date": inquiry_date,
        "bank_id": bank_id,
        "bank_name": bank_name,
        "org_type": org_type,
        "purpose_id": purpose_id,
        "purpose_name": purpose_name
    }


# ============================================================
# INPUT DATA
# ============================================================

def load_customers():
    """
    Reads customers and creates a dictionary:
    owner key -> customer creation date.
    """

    if not CUSTOMER_FILE.exists():
        raise FileNotFoundError(
            f"Customer file was not found:\n{CUSTOMER_FILE}"
        )

    customer_creation_dates = {}

    with open(
        CUSTOMER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_NO",
            "PIN",
            "CUSTOMER_TYPE",
            "CREATION_DT"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "Customer table is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_type = (
                row.get("CUSTOMER_TYPE") or ""
            ).strip().upper()

            pin_code = (row.get("PIN") or "").strip()
            customer_no = (
                row.get("CUSTOMER_NO") or ""
            ).strip()

            if customer_type == "I":
                owner_key = get_owner_key(
                    pin_code=pin_code,
                    document_no=""
                )
            else:
                owner_key = get_owner_key(
                    pin_code="",
                    document_no=customer_no
                )

            creation_date = parse_date(
                row.get("CREATION_DT") or ""
            )

            customer_creation_dates[owner_key] = (
                creation_date or date(2010, 1, 1)
            )

    return customer_creation_dates


def load_mkr_master():
    """
    Reads MKR master records.

    SEARCH_ID is unique here.
    It will repeat in the inquiry-history child table.
    """

    if not MKR_MASTER_FILE.exists():
        raise FileNotFoundError(
            f"MKR master file was not found:\n{MKR_MASTER_FILE}"
        )

    master_rows = []

    with open(
        MKR_MASTER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "SEARCH_ID",
            "PIN_CODE",
            "DOCUMENT_NO",
            "REPORTINGDATE",
            "INSERT_DATE"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "MKR master is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        seen_search_ids = set()

        for row in reader:
            search_id = (
                row.get("SEARCH_ID") or ""
            ).strip()

            if not search_id:
                raise ValueError(
                    "MKR master contains a blank SEARCH_ID."
                )

            if search_id in seen_search_ids:
                raise ValueError(
                    f"Duplicate SEARCH_ID in MKR master: {search_id}"
                )

            seen_search_ids.add(search_id)

            pin_code = (row.get("PIN_CODE") or "").strip()
            document_no = (
                row.get("DOCUMENT_NO") or ""
            ).strip()

            owner_key = get_owner_key(
                pin_code=pin_code,
                document_no=document_no
            )

            reporting_date = parse_date(
                row.get("REPORTINGDATE") or ""
            )

            insert_date = parse_date(
                row.get("INSERT_DATE") or ""
            )

            if not reporting_date:
                raise ValueError(
                    f"Invalid REPORTINGDATE for {search_id}"
                )

            master_rows.append({
                "SEARCH_ID": search_id,
                "PIN_CODE": pin_code,
                "DOCUMENT_NO": document_no,
                "OWNER_KEY": owner_key,
                "REPORTINGDATE": reporting_date,
                "INSERT_DATE": insert_date or reporting_date
            })

    return master_rows


# ============================================================
# EVENT TIMELINE CREATION
# ============================================================

def build_customer_inquiry_events(
    master_rows,
    customer_creation_dates
):
    """
    Creates one actual inquiry-event timeline per customer.

    The timeline includes:
    1. external historical inquiries
    2. earlier MKR master searches

    A later parent search sees the earlier events from that timeline.
    This prevents random, disconnected inquiry history.
    """

    master_rows_by_owner = defaultdict(list)

    for row in master_rows:
        master_rows_by_owner[row["OWNER_KEY"]].append(row)

    events_by_owner = defaultdict(list)
    event_counter = 1

    for owner_key, owner_master_rows in master_rows_by_owner.items():
        owner_master_rows.sort(
            key=lambda row: (
                row["REPORTINGDATE"],
                row["SEARCH_ID"]
            )
        )

        is_individual = owner_key.startswith("P:")

        first_report_date = owner_master_rows[0][
            "REPORTINGDATE"
        ]

        last_report_date = owner_master_rows[-1][
            "REPORTINGDATE"
        ]

        customer_creation_date = customer_creation_dates.get(
            owner_key,
            date(2010, 1, 1)
        )

        external_history_start = max(
            customer_creation_date,
            first_report_date - timedelta(days=LOOKBACK_DAYS)
        )

        external_history_end = last_report_date - timedelta(days=1)

        # Number of separate external inquiry events for this customer.
        external_event_count = random.choices(
            [0, 1, 2, 3, 4, 5, 6, 7],
            weights=[6, 12, 19, 23, 19, 12, 6, 3],
            k=1
        )[0]

        # Customers with more MKR searches are more likely
        # to have an extensive inquiry history.
        external_event_count += max(
            0,
            len(owner_master_rows) - 2
        )

        external_event_count = min(
            external_event_count,
            10
        )

        # Create external historical inquiry events.
        for _ in range(external_event_count):
            inquiry_date = random_date(
                start_date=external_history_start,
                end_date=external_history_end
            )

            event = choose_inquiry_event(
                is_individual=is_individual,
                inquiry_date=inquiry_date,
                event_number=event_counter
            )

            events_by_owner[owner_key].append(event)
            event_counter += 1

        # Each current master search becomes a possible
        # historical event for future searches by that customer.
        for master_row in owner_master_rows:
            event = choose_inquiry_event(
                is_individual=is_individual,
                inquiry_date=master_row["REPORTINGDATE"],
                event_number=event_counter
            )

            event["source_search_id"] = master_row["SEARCH_ID"]

            events_by_owner[owner_key].append(event)
            event_counter += 1

        events_by_owner[owner_key].sort(
            key=lambda event: (
                event["inquiry_date"],
                event["event_id"]
            )
        )

    return events_by_owner


# ============================================================
# CHILD ROW CREATION
# ============================================================

def create_inquiry_history_rows(
    master_rows,
    events_by_owner
):
    """
    For each current MKR master search:
    - find all past inquiry events
    - keep only the previous 730 days
    - create one child row for each event
    """

    history_rows = []

    for master_row in master_rows:
        search_id = master_row["SEARCH_ID"]
        pin_code = master_row["PIN_CODE"]
        document_no = master_row["DOCUMENT_NO"]
        owner_key = master_row["OWNER_KEY"]
        reporting_date = master_row["REPORTINGDATE"]
        insert_date = master_row["INSERT_DATE"]

        lookback_start = reporting_date - timedelta(
            days=LOOKBACK_DAYS
        )

        eligible_events = [
            event
            for event in events_by_owner[owner_key]
            if (
                lookback_start <= event["inquiry_date"]
                and event["inquiry_date"] < reporting_date
            )
        ]

        for event in eligible_events:
            inserted_at = datetime.combine(
                insert_date,
                datetime.min.time()
            ) + timedelta(
                hours=random.randint(7, 20),
                minutes=random.randint(0, 59),
                seconds=random.randint(0, 59)
            )

            history_rows.append({
                "SEARCH_ID": search_id,
                "PIN_CODE": pin_code,
                "INQBANKID": event["bank_id"],
                "INQBANKNAME": event["bank_name"],
                "INQDATE": event[
                    "inquiry_date"
                ].isoformat(),
                "INQORGIDTYPE": event["org_type"],
                "INQPURPOSEID": event["purpose_id"],
                "INQPURPOSENAME": event[
                    "purpose_name"
                ],
                "INS_DTTM": inserted_at.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
                "DOCUMENTNO": document_no,
                "REPORTINGDATE": reporting_date.isoformat()
            })

    return history_rows


# ============================================================
# VALIDATION
# ============================================================

def validate_inquiry_history(
    master_rows,
    history_rows
):
    """
    Validates:
    1. Every child SEARCH_ID exists in MKR master.
    2. PIN_CODE / DOCUMENTNO match the parent search.
    3. Historical inquiry date is before current report date.
    4. Historical inquiry date is within 730 days.
    """

    master_by_search_id = {
        row["SEARCH_ID"]: row
        for row in master_rows
    }

    rows_per_search = defaultdict(int)

    for history_row in history_rows:
        search_id = history_row["SEARCH_ID"]

        if search_id not in master_by_search_id:
            raise ValueError(
                "Inquiry history has an unmatched SEARCH_ID: "
                f"{search_id}"
            )

        parent_row = master_by_search_id[search_id]

        if history_row["PIN_CODE"] != parent_row["PIN_CODE"]:
            raise ValueError(
                f"PIN_CODE mismatch for SEARCH_ID: {search_id}"
            )

        if (
            history_row["DOCUMENTNO"]
            != parent_row["DOCUMENT_NO"]
        ):
            raise ValueError(
                f"DOCUMENTNO mismatch for SEARCH_ID: {search_id}"
            )

        inquiry_date = parse_date(
            history_row["INQDATE"]
        )

        reporting_date = parse_date(
            history_row["REPORTINGDATE"]
        )

        if inquiry_date >= reporting_date:
            raise ValueError(
                "History inquiry is not earlier than the "
                f"current search. SEARCH_ID: {search_id}"
            )

        if (
            reporting_date - inquiry_date
        ).days > LOOKBACK_DAYS:
            raise ValueError(
                "History inquiry is older than two years. "
                f"SEARCH_ID: {search_id}"
            )

        rows_per_search[search_id] += 1

    return rows_per_search


# ============================================================
# FILE WRITING
# ============================================================

def write_csv(file_path, rows):
    """Writes inquiry history rows to CSV."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_INQUIRY_HISTORY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_inquiry_history():
    customer_creation_dates = load_customers()
    master_rows = load_mkr_master()

    events_by_owner = build_customer_inquiry_events(
        master_rows=master_rows,
        customer_creation_dates=customer_creation_dates
    )

    history_rows = create_inquiry_history_rows(
        master_rows=master_rows,
        events_by_owner=events_by_owner
    )

    rows_per_search = validate_inquiry_history(
        master_rows=master_rows,
        history_rows=history_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_inquiry_history_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_inquiry_history_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_inquiry_history_generation_summary.json"
    )

    write_csv(full_output_file, history_rows)
    write_csv(sample_output_file, history_rows[:100])

    searches_with_history = len(rows_per_search)
    searches_without_history = (
        len(master_rows) - searches_with_history
    )

    max_history_rows = max(
        rows_per_search.values(),
        default=0
    )

    summary = {
        "table_name": "RS_MKR_INQUIRY_HISTORY",
        "generated_rows": len(history_rows),
        "unique_parent_search_ids": len(master_rows),
        "parent_search_ids_with_history": searches_with_history,
        "parent_search_ids_without_history": (
            searches_without_history
        ),
        "maximum_history_rows_for_one_search": (
            max_history_rows
        ),
        "join_rule": (
            "RS_MKR_INQUIRY_HISTORY.SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "lookback_rule": (
            "INQDATE is before REPORTINGDATE and no more "
            "than 730 days earlier"
        ),
        "individual_customer_rule": (
            "PIN_CODE matches the parent MKR master PIN_CODE"
        ),
        "company_customer_rule": (
            "DOCUMENTNO matches the parent MKR master DOCUMENT_NO"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR inquiry history generation completed successfully.")
    print(f"Generated inquiry-history rows: {len(history_rows):,}")
    print(
        "Master searches with prior inquiries: "
        f"{searches_with_history:,}"
    )
    print(
        "Master searches with no prior inquiries: "
        f"{searches_without_history:,}"
    )
    print(
        "Maximum inquiry-history rows for one search: "
        f"{max_history_rows:,}"
    )
    print("Unmatched SEARCH_ID values: 0")
    print("Two-year lookback validation: passed")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_inquiry_history()

MKR inquiry history generation completed successfully.
Generated inquiry-history rows: 98,219
Master searches with prior inquiries: 34,367
Master searches with no prior inquiries: 5,633
Maximum inquiry-history rows for one search: 20
Unmatched SEARCH_ID values: 0
Two-year lookback validation: passed
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_inquiry_history_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_inquiry_history_base_sample.csv


In [5]:
import csv
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260628
REPORTING_CUTOFF_DATE = date(2026, 6, 1)

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CONTRACT_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
)

CUSTOMER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

CONTRACT_PROFILE_COLUMNS = [
    "ACCOUNT_NUMBER",
    "BRANCH_CODE",
    "CUSTOMER_ID",
    "CUSTOMER_TYPE",
    "PRODUCT_CODE",
    "PRODUCT_CATEGORY",
    "VALUE_DATE",
    "MATURITY_DATE",
    "CLOSE_DATE",
    "STATUS_AT_CUTOFF",
    "AMOUNT_FINANCED",
    "TERM_MONTHS",
    "INTEREST_RATE",
    "EMI_AMOUNT",
    "FIRST_PAYMENT_DATE",
    "RISK_PATTERN",
    "EXPECTED_DAILY_ROWS"
]


# ============================================================
# PRODUCT RULES
# ============================================================

PRODUCT_RULES = {
    "CZL3": {
        "category": "CONSUMER_LOAN",
        "term_min": 12,
        "term_max": 48,
        "amount_min": 500,
        "amount_max": 15_000,
        "rate_min": 14,
        "rate_max": 28
    },
    "PERS": {
        "category": "PERSONAL_LOAN",
        "term_min": 12,
        "term_max": 60,
        "amount_min": 1_000,
        "amount_max": 30_000,
        "rate_min": 12,
        "rate_max": 26
    },
    "AUTO": {
        "category": "AUTO_LOAN",
        "term_min": 24,
        "term_max": 84,
        "amount_min": 5_000,
        "amount_max": 70_000,
        "rate_min": 8,
        "rate_max": 18
    },
    "MORT": {
        "category": "MORTGAGE",
        "term_min": 60,
        "term_max": 240,
        "amount_min": 30_000,
        "amount_max": 250_000,
        "rate_min": 6,
        "rate_max": 14
    },
    "BIZL": {
        "category": "SME_LOAN",
        "term_min": 12,
        "term_max": 84,
        "amount_min": 10_000,
        "amount_max": 500_000,
        "rate_min": 8,
        "rate_max": 20
    },
    "CRED": {
        "category": "CREDIT_LINE",
        "term_min": 12,
        "term_max": 48,
        "amount_min": 1_000,
        "amount_max": 50_000,
        "rate_min": 14,
        "rate_max": 30
    }
}


# ============================================================
# HELPERS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text to a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def add_months(start_date, months):
    """Adds months to a date without extra libraries."""

    new_month = start_date.month - 1 + months
    new_year = start_date.year + new_month // 12
    new_month = new_month % 12 + 1

    month_days = [
        31,
        29 if new_year % 4 == 0 and (
            new_year % 100 != 0 or new_year % 400 == 0
        ) else 28,
        31, 30, 31, 30,
        31, 31, 30, 31, 30, 31
    ]

    new_day = min(
        start_date.day,
        month_days[new_month - 1]
    )

    return date(new_year, new_month, new_day)


def calculate_annuity_payment(
    principal,
    annual_interest_rate,
    term_months
):
    """Calculates an approximate monthly annuity payment."""

    monthly_rate = annual_interest_rate / 100 / 12

    if monthly_rate == 0:
        return round(principal / term_months, 2)

    numerator = principal * monthly_rate
    denominator = 1 - (
        (1 + monthly_rate) ** (-term_months)
    )

    return round(numerator / denominator, 2)


def load_customer_types():
    """Creates CUSTOMER_NO/CIF -> CUSTOMER_TYPE mapping."""

    if not CUSTOMER_FILE.exists():
        raise FileNotFoundError(
            f"Customer file not found: {CUSTOMER_FILE}"
        )

    customer_types = {}

    with open(
        CUSTOMER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        for row in reader:
            customer_no = (
                row.get("CUSTOMER_NO") or ""
            ).strip()

            customer_type = (
                row.get("CUSTOMER_TYPE") or ""
            ).strip().upper()

            if customer_no:
                customer_types[customer_no] = (
                    customer_type or "I"
                )

    return customer_types


def load_contracts():
    """Reads the contracts source file."""

    if not CONTRACT_FILE.exists():
        raise FileNotFoundError(
            f"Contract file not found: {CONTRACT_FILE}"
        )

    contracts = []

    with open(
        CONTRACT_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "ACCOUNT_NUMBER",
            "BRANCH_CODE",
            "CUSTOMER_ID",
            "PRODUCT_CODE",
            "BOOK_DATE"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "Contracts file is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            contracts.append(row)

    return contracts


def choose_risk_pattern(product_code, customer_type):
    """
    This controls future payment behavior.

    BAD means the contract will eventually reach 90+ DPD.
    We will later derive the model target from daily FCT rows,
    not directly from this hidden generation field.
    """

    if product_code in {"CZL3", "PERS"}:
        weights = [70, 14, 7, 9]

    elif product_code == "CRED":
        weights = [62, 16, 9, 13]

    elif product_code == "AUTO":
        weights = [76, 12, 6, 6]

    elif product_code == "MORT":
        weights = [82, 9, 5, 4]

    elif product_code == "BIZL":
        weights = [72, 12, 7, 9]

    else:
        weights = [72, 13, 7, 8]

    if customer_type in {"C", "B"}:
        weights = [
            max(1, weights[0] - 3),
            weights[1],
            weights[2] + 1,
            weights[3] + 2
        ]

    return random.choices(
        [
            "CLEAN",
            "SHORT_LATE",
            "SERIOUS_LATE",
            "BAD"
        ],
        weights=weights,
        k=1
    )[0]


def create_profile(contract_row, customer_types):
    """Creates one deterministic contract profile."""

    account_number = (
        contract_row.get("ACCOUNT_NUMBER") or ""
    ).strip()

    customer_id = (
        contract_row.get("CUSTOMER_ID") or ""
    ).strip()

    product_code = (
        contract_row.get("PRODUCT_CODE") or ""
    ).strip().upper()

    value_date = parse_date(
        contract_row.get("VALUE_DATE")
        or contract_row.get("BOOK_DATE")
        or ""
    )

    if not value_date:
        raise ValueError(
            f"Invalid VALUE_DATE for {account_number}"
        )

    product_rule = PRODUCT_RULES.get(
        product_code,
        PRODUCT_RULES["CZL3"]
    )

    customer_type = customer_types.get(
        customer_id,
        "I"
    )

    amount_max = product_rule["amount_max"]

    if customer_type in {"C", "B"}:
        amount_max *= 3

    amount_financed = round(
        random.triangular(
            product_rule["amount_min"],
            amount_max,
            product_rule["amount_min"] * 1.5
        ),
        2
    )

    term_months = random.randint(
        product_rule["term_min"],
        product_rule["term_max"]
    )

    interest_rate = round(
        random.uniform(
            product_rule["rate_min"],
            product_rule["rate_max"]
        ),
        2
    )

    maturity_date = add_months(
        value_date,
        term_months
    )

    first_payment_date = add_months(
        value_date,
        1
    )

    emi_amount = calculate_annuity_payment(
        principal=amount_financed,
        annual_interest_rate=interest_rate,
        term_months=term_months
    )

    risk_pattern = choose_risk_pattern(
        product_code=product_code,
        customer_type=customer_type
    )

    # Normal loans close at maturity.
    # A small proportion close early.
    # Bad loans can remain active beyond scheduled maturity.
    close_date = None

    if risk_pattern != "BAD":
        close_date = maturity_date

        if random.random() < 0.07:
            earliest_early_close = add_months(
                value_date,
                max(6, term_months // 2)
            )

            if earliest_early_close < maturity_date:
                close_date = earliest_early_close + timedelta(
                    days=random.randint(
                        0,
                        max(
                            1,
                            (maturity_date - earliest_early_close).days
                        )
                    )
                )

    if close_date and close_date <= REPORTING_CUTOFF_DATE:
        final_snapshot_date = close_date
        status_at_cutoff = "L"

    else:
        final_snapshot_date = REPORTING_CUTOFF_DATE
        status_at_cutoff = "A"

    expected_daily_rows = (
        final_snapshot_date - value_date
    ).days + 1

    return {
        "ACCOUNT_NUMBER": account_number,
        "BRANCH_CODE": (
            contract_row.get("BRANCH_CODE") or ""
        ).strip(),
        "CUSTOMER_ID": customer_id,
        "CUSTOMER_TYPE": customer_type,
        "PRODUCT_CODE": product_code,
        "PRODUCT_CATEGORY": (
            contract_row.get("PRODUCT_CATEGORY")
            or product_rule["category"]
        ).strip(),
        "VALUE_DATE": value_date.isoformat(),
        "MATURITY_DATE": maturity_date.isoformat(),
        "CLOSE_DATE": (
            close_date.isoformat()
            if close_date
            and close_date <= REPORTING_CUTOFF_DATE
            else ""
        ),
        "STATUS_AT_CUTOFF": status_at_cutoff,
        "AMOUNT_FINANCED": f"{amount_financed:.2f}",
        "TERM_MONTHS": str(term_months),
        "INTEREST_RATE": f"{interest_rate:.2f}",
        "EMI_AMOUNT": f"{emi_amount:.2f}",
        "FIRST_PAYMENT_DATE": first_payment_date.isoformat(),
        "RISK_PATTERN": risk_pattern,
        "EXPECTED_DAILY_ROWS": str(expected_daily_rows)
    }


def validate_profiles(profiles):
    """Checks profile keys and date logic."""

    account_numbers = [
        row["ACCOUNT_NUMBER"]
        for row in profiles
    ]

    if len(account_numbers) != len(set(account_numbers)):
        raise ValueError(
            "Duplicate ACCOUNT_NUMBER values in profiles."
        )

    for row in profiles:
        value_date = parse_date(row["VALUE_DATE"])
        maturity_date = parse_date(row["MATURITY_DATE"])
        close_date = parse_date(row["CLOSE_DATE"])

        if maturity_date <= value_date:
            raise ValueError(
                f"Invalid maturity date: {row['ACCOUNT_NUMBER']}"
            )

        if close_date and close_date < value_date:
            raise ValueError(
                f"Invalid close date: {row['ACCOUNT_NUMBER']}"
            )


def write_csv(file_path, rows):
    """Writes profiles into a CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=CONTRACT_PROFILE_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_create_contract_profiles():
    customer_types = load_customer_types()
    contract_rows = load_contracts()

    profiles = [
        create_profile(
            contract_row=contract_row,
            customer_types=customer_types
        )
        for contract_row in contract_rows
    ]

    validate_profiles(profiles)

    full_output_file = (
        RAW_DATA_FOLDER / "rs_contract_profile_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_contract_profile_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_contract_profile_generation_summary.json"
    )

    write_csv(full_output_file, profiles)
    write_csv(sample_output_file, profiles[:100])

    total_daily_rows = sum(
        int(row["EXPECTED_DAILY_ROWS"])
        for row in profiles
    )

    closed_contracts = sum(
        1 for row in profiles
        if row["STATUS_AT_CUTOFF"] == "L"
    )

    active_contracts = len(profiles) - closed_contracts

    summary = {
        "table_name": "RS_CONTRACT_PROFILE",
        "generated_profiles": len(profiles),
        "reporting_cutoff_date": (
            REPORTING_CUTOFF_DATE.isoformat()
        ),
        "estimated_daily_contract_current_rows": (
            total_daily_rows
        ),
        "closed_contracts_before_or_at_cutoff": (
            closed_contracts
        ),
        "active_contracts_at_cutoff": active_contracts,
        "daily_fact_key": (
            "ACCOUNT_NUMBER + TRN_DT"
        ),
        "closure_rule": (
            "Rows stop at CLOSE_DATE for liquidated contracts; "
            "active contracts continue to 2026-06-01."
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Contract profile generation completed successfully.")
    print(f"Contract profiles: {len(profiles):,}")
    print(f"Closed contracts by cutoff: {closed_contracts:,}")
    print(f"Active contracts at cutoff: {active_contracts:,}")
    print(
        "Estimated RS_CONTRACT_CURRENT daily rows: "
        f"{total_daily_rows:,}"
    )
    print(
        "Important: do not open the final daily CSV in Excel. "
        "It will be too large."
    )
    print(f"Profile file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_create_contract_profiles()

Contract profile generation completed successfully.
Contract profiles: 20,000
Closed contracts by cutoff: 9,401
Active contracts at cutoff: 10,599
Estimated RS_CONTRACT_CURRENT daily rows: 21,248,107
Important: do not open the final daily CSV in Excel. It will be too large.
Profile file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contract_profile_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_contract_profile_base_sample.csv


In [6]:
import csv
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path


PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
CONTRACT_FILE = PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
MKR_MASTER_FILE = PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"


def parse_date(value):
    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


# ------------------------------------------------------------
# 1. Build PIN -> CIF mapping for individual customers
# ------------------------------------------------------------

pin_to_customer_no = {}

with open(
    CUSTOMER_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    reader = csv.DictReader(file)

    for row in reader:
        customer_type = (
            row.get("CUSTOMER_TYPE") or ""
        ).strip().upper()

        customer_no = (
            row.get("CUSTOMER_NO") or ""
        ).strip()

        pin_code = (
            row.get("PIN") or ""
        ).strip()

        if customer_type == "I" and pin_code:
            pin_to_customer_no[pin_code] = customer_no


# ------------------------------------------------------------
# 2. Convert MKR searches into CUSTOMER_NO / CIF-based records
# ------------------------------------------------------------

mkr_dates_by_customer = defaultdict(list)

with open(
    MKR_MASTER_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    reader = csv.DictReader(file)

    for row in reader:
        pin_code = (
            row.get("PIN_CODE") or ""
        ).strip()

        document_no = (
            row.get("DOCUMENT_NO") or ""
        ).strip()

        reporting_date = parse_date(
            row.get("REPORTINGDATE") or ""
        )

        if not reporting_date:
            continue

        if pin_code:
            customer_no = pin_to_customer_no.get(pin_code)

        else:
            customer_no = document_no

        if customer_no:
            mkr_dates_by_customer[customer_no].append(
                reporting_date
            )


# ------------------------------------------------------------
# 3. Check each contract for earlier same-customer MKR searches
# ------------------------------------------------------------

total_contracts = 0
contracts_with_any_prior_search = 0
contracts_with_search_last_30_days = 0
contracts_with_search_last_60_days = 0
contracts_with_search_last_90_days = 0

with open(
    CONTRACT_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_contracts += 1

        customer_id = (
            row.get("CUSTOMER_ID") or ""
        ).strip()

        value_date = parse_date(
            row.get("VALUE_DATE")
            or row.get("BOOK_DATE")
            or ""
        )

        if not customer_id or not value_date:
            continue

        customer_mkr_dates = mkr_dates_by_customer.get(
            customer_id,
            []
        )

        prior_dates = [
            search_date
            for search_date in customer_mkr_dates
            if search_date <= value_date
        ]

        if prior_dates:
            contracts_with_any_prior_search += 1

        if any(
            value_date - timedelta(days=30)
            <= search_date
            <= value_date
            for search_date in customer_mkr_dates
        ):
            contracts_with_search_last_30_days += 1

        if any(
            value_date - timedelta(days=60)
            <= search_date
            <= value_date
            for search_date in customer_mkr_dates
        ):
            contracts_with_search_last_60_days += 1

        if any(
            value_date - timedelta(days=90)
            <= search_date
            <= value_date
            for search_date in customer_mkr_dates
        ):
            contracts_with_search_last_90_days += 1


print("MKR-to-contract timing audit completed.")
print(f"Total contracts: {total_contracts:,}")
print(
    "Contracts with any earlier MKR search: "
    f"{contracts_with_any_prior_search:,} "
    f"({contracts_with_any_prior_search / total_contracts:.1%})"
)
print(
    "Contracts with MKR search in previous 30 days: "
    f"{contracts_with_search_last_30_days:,} "
    f"({contracts_with_search_last_30_days / total_contracts:.1%})"
)
print(
    "Contracts with MKR search in previous 60 days: "
    f"{contracts_with_search_last_60_days:,} "
    f"({contracts_with_search_last_60_days / total_contracts:.1%})"
)
print(
    "Contracts with MKR search in previous 90 days: "
    f"{contracts_with_search_last_90_days:,} "
    f"({contracts_with_search_last_90_days / total_contracts:.1%})"
)

MKR-to-contract timing audit completed.
Total contracts: 20,000
Contracts with any earlier MKR search: 12,935 (64.7%)
Contracts with MKR search in previous 30 days: 608 (3.0%)
Contracts with MKR search in previous 60 days: 1,168 (5.8%)
Contracts with MKR search in previous 90 days: 1,703 (8.5%)


In [7]:
import csv
import hashlib
import json
import random
from collections import defaultdict, Counter
from pathlib import Path


# ============================================================
# MASKING POLICY
# ============================================================

MASKING_SEED = "risk_scoring_project_bank_mask_v1"

INTERNAL_CODE = "1000"
INTERNAL_NAME = "INTERNAL"

EXTERNAL_NAME = "EXTERNAL"
EXTERNAL_CODES = [
    str(code)
    for code in range(1001, 1021)
]

# Probability that a search contains at least one INTERNAL source.
INTERNAL_SEARCH_PROBABILITY = 0.28


# ============================================================
# FILE PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

LIABILITIES_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_liabilities_base.csv"
)

INQUIRY_HISTORY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_inquiry_history_base.csv"
)

LIABILITIES_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_liabilities_base_sample.csv"
)

INQUIRY_HISTORY_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_inquiry_history_base_sample.csv"
)

SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_bank_masking_summary.json"
)


# ============================================================
# CSV FUNCTIONS
# ============================================================

def read_csv(file_path):
    """Reads a CSV and preserves its original column order."""

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames

    return fieldnames, rows


def write_csv(file_path, fieldnames, rows):
    """Writes a CSV with UTF-8 BOM for easier Excel viewing."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MASKING FUNCTIONS
# ============================================================

def stable_random(search_id):
    """
    Creates a repeatable random generator for one SEARCH_ID.

    The mapping is different per SEARCH_ID but remains stable
    if this exact correction code is rerun.
    """

    seed_text = f"{MASKING_SEED}|{search_id}"

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).hexdigest()

    numeric_seed = int(digest[:16], 16)

    return random.Random(numeric_seed)


def is_already_masked(row, code_column, name_column):
    """
    Lets this correction cell be safely rerun.

    If a row is already INTERNAL/EXTERNAL with a valid code,
    it remains unchanged.
    """

    bank_code = (row.get(code_column) or "").strip()
    bank_name = (row.get(name_column) or "").strip().upper()

    if (
        bank_name == INTERNAL_NAME
        and bank_code == INTERNAL_CODE
    ):
        return True

    if (
        bank_name == EXTERNAL_NAME
        and bank_code in EXTERNAL_CODES
    ):
        return True

    return False


def get_legacy_code(row, code_column, row_number):
    """
    Returns the old source-bank code before masking.

    Blank old codes are assigned a temporary unique label so they
    can still receive a valid EXTERNAL masked code.
    """

    bank_code = (row.get(code_column) or "").strip()

    if bank_code:
        return bank_code

    return f"BLANK_SOURCE_{row_number}"


def collect_codes_by_search(
    liabilities_rows,
    inquiry_history_rows
):
    """
    Collects old bank codes within each SEARCH_ID.

    One SEARCH_ID receives its own private masking map.
    Therefore, external code meaning is not stable across searches.
    """

    legacy_codes_by_search = defaultdict(set)

    for row_number, row in enumerate(
        liabilities_rows,
        start=1
    ):
        if is_already_masked(
            row,
            code_column="BANKID",
            name_column="BANKNAME"
        ):
            continue

        search_id = (row.get("SEARCH_ID") or "").strip()

        if not search_id:
            raise ValueError(
                "Blank SEARCH_ID found in liabilities."
            )

        legacy_code = get_legacy_code(
            row=row,
            code_column="BANKID",
            row_number=row_number
        )

        legacy_codes_by_search[search_id].add(
            legacy_code
        )

    for row_number, row in enumerate(
        inquiry_history_rows,
        start=1
    ):
        if is_already_masked(
            row,
            code_column="INQBANKID",
            name_column="INQBANKNAME"
        ):
            continue

        search_id = (row.get("SEARCH_ID") or "").strip()

        if not search_id:
            raise ValueError(
                "Blank SEARCH_ID found in inquiry history."
            )

        legacy_code = get_legacy_code(
            row=row,
            code_column="INQBANKID",
            row_number=row_number
        )

        legacy_codes_by_search[search_id].add(
            legacy_code
        )

    return legacy_codes_by_search


def build_masking_maps(legacy_codes_by_search):
    """
    Builds one mapping per SEARCH_ID.

    Example:
    In search A, old code 001 may become 1004.
    In search B, old code 001 may become 1017.

    This prevents a static code-to-bank relationship.
    """

    masking_maps = {}

    for search_id, legacy_codes in legacy_codes_by_search.items():
        rng = stable_random(search_id)

        legacy_codes = sorted(legacy_codes)

        shuffled_external_codes = EXTERNAL_CODES.copy()
        rng.shuffle(shuffled_external_codes)

        internal_legacy_code = None

        if (
            legacy_codes
            and rng.random() < INTERNAL_SEARCH_PROBABILITY
        ):
            internal_legacy_code = rng.choice(
                legacy_codes
            )

        search_mapping = {}
        external_code_position = 0

        for legacy_code in legacy_codes:
            if legacy_code == internal_legacy_code:
                search_mapping[legacy_code] = INTERNAL_CODE

            else:
                search_mapping[legacy_code] = (
                    shuffled_external_codes[
                        external_code_position
                    ]
                )

                external_code_position += 1

        masking_maps[search_id] = search_mapping

    return masking_maps


def apply_masking(
    rows,
    code_column,
    name_column,
    masking_maps
):
    """Replaces old codes/names with masked INTERNAL/EXTERNAL values."""

    for row_number, row in enumerate(rows, start=1):
        if is_already_masked(
            row,
            code_column=code_column,
            name_column=name_column
        ):
            continue

        search_id = (row.get("SEARCH_ID") or "").strip()

        legacy_code = get_legacy_code(
            row=row,
            code_column=code_column,
            row_number=row_number
        )

        masked_code = masking_maps[search_id][legacy_code]

        row[code_column] = masked_code

        if masked_code == INTERNAL_CODE:
            row[name_column] = INTERNAL_NAME

        else:
            row[name_column] = EXTERNAL_NAME


def validate_masked_rows(
    rows,
    code_column,
    name_column,
    table_name
):
    """
    Confirms:
    - INTERNAL always uses 1000.
    - EXTERNAL always uses 1001–1020.
    - No previous names remain.
    """

    for row in rows:
        bank_code = (row.get(code_column) or "").strip()
        bank_name = (row.get(name_column) or "").strip().upper()

        if bank_name == INTERNAL_NAME:
            if bank_code != INTERNAL_CODE:
                raise ValueError(
                    f"{table_name}: INTERNAL has invalid code "
                    f"{bank_code}."
                )

        elif bank_name == EXTERNAL_NAME:
            if bank_code not in EXTERNAL_CODES:
                raise ValueError(
                    f"{table_name}: EXTERNAL has invalid code "
                    f"{bank_code}."
                )

        else:
            raise ValueError(
                f"{table_name}: unmasked bank name found: "
                f"{bank_name}"
            )


# ============================================================
# MAIN
# ============================================================

def main_mask_bank_identifiers():
    liability_columns, liability_rows = read_csv(
        LIABILITIES_FILE
    )

    inquiry_columns, inquiry_rows = read_csv(
        INQUIRY_HISTORY_FILE
    )

    legacy_codes_by_search = collect_codes_by_search(
        liabilities_rows=liability_rows,
        inquiry_history_rows=inquiry_rows
    )

    masking_maps = build_masking_maps(
        legacy_codes_by_search
    )

    apply_masking(
        rows=liability_rows,
        code_column="BANKID",
        name_column="BANKNAME",
        masking_maps=masking_maps
    )

    apply_masking(
        rows=inquiry_rows,
        code_column="INQBANKID",
        name_column="INQBANKNAME",
        masking_maps=masking_maps
    )

    validate_masked_rows(
        rows=liability_rows,
        code_column="BANKID",
        name_column="BANKNAME",
        table_name="MKR liabilities"
    )

    validate_masked_rows(
        rows=inquiry_rows,
        code_column="INQBANKID",
        name_column="INQBANKNAME",
        table_name="MKR inquiry history"
    )

    # Rewrite the full raw files.
    write_csv(
        LIABILITIES_FILE,
        liability_columns,
        liability_rows
    )

    write_csv(
        INQUIRY_HISTORY_FILE,
        inquiry_columns,
        inquiry_rows
    )

    # Rewrite the small GitHub sample files.
    write_csv(
        LIABILITIES_SAMPLE_FILE,
        liability_columns,
        liability_rows[:100]
    )

    write_csv(
        INQUIRY_HISTORY_SAMPLE_FILE,
        inquiry_columns,
        inquiry_rows[:100]
    )

    liability_names = Counter(
        row["BANKNAME"]
        for row in liability_rows
    )

    inquiry_names = Counter(
        row["INQBANKNAME"]
        for row in inquiry_rows
    )

    summary = {
        "policy": {
            "internal_code": INTERNAL_CODE,
            "internal_name": INTERNAL_NAME,
            "external_code_range": "1001-1020",
            "external_name": EXTERNAL_NAME,
            "mapping_scope": "per SEARCH_ID",
            "static_external_bank_mapping": False
        },
        "liabilities": {
            "total_rows": len(liability_rows),
            "internal_rows": liability_names[INTERNAL_NAME],
            "external_rows": liability_names[EXTERNAL_NAME]
        },
        "inquiry_history": {
            "total_rows": len(inquiry_rows),
            "internal_rows": inquiry_names[INTERNAL_NAME],
            "external_rows": inquiry_names[EXTERNAL_NAME]
        }
    }

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Bank masking correction completed.")
    print()
    print("Liabilities")
    print(f"Total rows: {len(liability_rows):,}")
    print(
        f"INTERNAL rows (1000): "
        f"{liability_names[INTERNAL_NAME]:,}"
    )
    print(
        f"EXTERNAL rows (1001-1020): "
        f"{liability_names[EXTERNAL_NAME]:,}"
    )
    print()
    print("Inquiry history")
    print(f"Total rows: {len(inquiry_rows):,}")
    print(
        f"INTERNAL rows (1000): "
        f"{inquiry_names[INTERNAL_NAME]:,}"
    )
    print(
        f"EXTERNAL rows (1001-1020): "
        f"{inquiry_names[EXTERNAL_NAME]:,}"
    )
    print()
    print(
        "Validation passed: no real bank names and no "
        "static external bank-code mapping."
    )


main_mask_bank_identifiers()

Bank masking correction completed.

Liabilities
Total rows: 70,921
INTERNAL rows (1000): 6,287
EXTERNAL rows (1001-1020): 64,634

Inquiry history
Total rows: 98,219
INTERNAL rows (1000): 7,947
EXTERNAL rows (1001-1020): 90,272

Validation passed: no real bank names and no static external bank-code mapping.


In [8]:
import csv
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260629
MAX_DAYS_BETWEEN_MKR_AND_DISBURSEMENT = 7

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CONTRACT_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
)

CONTRACT_PROFILE_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_contract_profile_base.csv"
)

CUSTOMER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

APPLICATION_COLUMNS = [
    "APPLICATION_ID",
    "APPLICATION_NUM",
    "ACCOUNT_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_TYPE",
    "APPLICATION_DATE",
    "DECISION_DATE",
    "VALUE_DATE",
    "MKR_SEARCH_ID",
    "PRODUCT_CODE",
    "PRODUCT_CATEGORY",
    "BRANCH_CODE",
    "REQUESTED_AMOUNT",
    "APPROVED_AMOUNT",
    "REQUESTED_TERM_MONTHS",
    "APPROVED_TERM_MONTHS",
    "INTEREST_RATE_OFFERED",
    "APPLICATION_CHANNEL",
    "DECISION",
    "APPLICATION_STATUS",
    "APPLICATION_REASON",
    "SOURCE_SYSTEM"
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text to a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def read_csv_rows(file_path, required_columns, table_name):
    """Reads a CSV and validates its important columns."""

    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return list(reader)


def write_csv(file_path, fieldnames, rows):
    """Writes CSV with UTF-8 BOM so Excel can read it easily."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


def choose_application_channel(customer_type):
    """Creates a realistic application channel."""

    if customer_type == "I":
        return random.choices(
            ["BRANCH", "DIGITAL", "PARTNER"],
            weights=[48, 42, 10],
            k=1
        )[0]

    return random.choices(
        ["BRANCH", "RM"],
        weights=[35, 65],
        k=1
    )[0]


# ============================================================
# LOAD SOURCE DATA
# ============================================================

def load_customer_creation_dates():
    """Creates CUSTOMER_NO -> CREATION_DT mapping."""

    customer_rows = read_csv_rows(
        file_path=CUSTOMER_FILE,
        required_columns={
            "CUSTOMER_NO",
            "CREATION_DT"
        },
        table_name="Customers"
    )

    customer_creation_dates = {}

    for row in customer_rows:
        customer_id = (
            row.get("CUSTOMER_NO") or ""
        ).strip()

        creation_date = parse_date(
            row.get("CREATION_DT") or ""
        )

        if customer_id:
            customer_creation_dates[customer_id] = (
                creation_date or date(2010, 1, 1)
            )

    return customer_creation_dates


def load_contract_profiles():
    """Creates ACCOUNT_NUMBER -> contract profile mapping."""

    profile_rows = read_csv_rows(
        file_path=CONTRACT_PROFILE_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "CUSTOMER_TYPE",
            "AMOUNT_FINANCED",
            "TERM_MONTHS",
            "INTEREST_RATE"
        },
        table_name="Contract profiles"
    )

    profiles_by_account = {}

    for row in profile_rows:
        account_number = (
            row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        if account_number:
            profiles_by_account[account_number] = row

    return profiles_by_account


# ============================================================
# APPLICATION CREATION
# ============================================================

def create_application_row(
    contract_row,
    profile_row,
    customer_creation_dates,
    sequence_number
):
    """
    Creates one approved application for one internal contract.

    The MKR search occurs 0–7 days before VALUE_DATE.
    """

    account_number = (
        contract_row.get("ACCOUNT_NUMBER") or ""
    ).strip()

    application_num = (
        contract_row.get("APPLICATION_NUM") or ""
    ).strip()

    customer_id = (
        contract_row.get("CUSTOMER_ID") or ""
    ).strip()

    value_date = parse_date(
        contract_row.get("VALUE_DATE")
        or contract_row.get("BOOK_DATE")
        or ""
    )

    if not account_number:
        raise ValueError("Blank ACCOUNT_NUMBER found.")

    if not application_num:
        raise ValueError(
            f"Blank APPLICATION_NUM for {account_number}"
        )

    if not customer_id:
        raise ValueError(
            f"Blank CUSTOMER_ID for {account_number}"
        )

    if not value_date:
        raise ValueError(
            f"Invalid VALUE_DATE for {account_number}"
        )

    customer_creation_date = customer_creation_dates.get(
        customer_id,
        date(2010, 1, 1)
    )

    days_before_disbursement = random.randint(
        0,
        MAX_DAYS_BETWEEN_MKR_AND_DISBURSEMENT
    )

    application_date = value_date - timedelta(
        days=days_before_disbursement
    )

    # A customer cannot apply before their customer record exists.
    application_date = max(
        application_date,
        customer_creation_date
    )

    decision_date = min(
        value_date,
        application_date + timedelta(
            days=random.randint(0, 2)
        )
    )

    approved_amount = round(
        float(profile_row["AMOUNT_FINANCED"]),
        2
    )

    approved_term = int(
        float(profile_row["TERM_MONTHS"])
    )

    requested_amount = round(
        approved_amount * random.uniform(1.00, 1.20),
        2
    )

    requested_term = min(
        240,
        approved_term + random.choice([0, 0, 0, 6, 12])
    )

    customer_type = (
        profile_row.get("CUSTOMER_TYPE") or "I"
    ).strip().upper()

    return {
        "APPLICATION_ID": (
            f"APPID{sequence_number:010d}"
        ),
        "APPLICATION_NUM": application_num,
        "ACCOUNT_NUMBER": account_number,
        "CUSTOMER_ID": customer_id,
        "CUSTOMER_TYPE": customer_type,
        "APPLICATION_DATE": application_date.isoformat(),
        "DECISION_DATE": decision_date.isoformat(),
        "VALUE_DATE": value_date.isoformat(),
        "MKR_SEARCH_ID": (
            f"MKRAPP{sequence_number:010d}"
        ),
        "PRODUCT_CODE": (
            contract_row.get("PRODUCT_CODE") or ""
        ).strip(),
        "PRODUCT_CATEGORY": (
            profile_row.get("PRODUCT_CATEGORY") or ""
        ).strip(),
        "BRANCH_CODE": (
            contract_row.get("BRANCH_CODE") or ""
        ).strip(),
        "REQUESTED_AMOUNT": f"{requested_amount:.2f}",
        "APPROVED_AMOUNT": f"{approved_amount:.2f}",
        "REQUESTED_TERM_MONTHS": str(requested_term),
        "APPROVED_TERM_MONTHS": str(approved_term),
        "INTEREST_RATE_OFFERED": (
            profile_row.get("INTEREST_RATE") or ""
        ).strip(),
        "APPLICATION_CHANNEL": choose_application_channel(
            customer_type
        ),
        "DECISION": "APPROVED",
        "APPLICATION_STATUS": "DISBURSED",
        "APPLICATION_REASON": "NEW_LOAN",
        "SOURCE_SYSTEM": "INTERNAL"
    }


# ============================================================
# VALIDATION
# ============================================================

def validate_applications(
    application_rows,
    contract_rows
):
    """Checks one-to-one contract and application relationships."""

    application_ids = [
        row["APPLICATION_ID"]
        for row in application_rows
    ]

    application_nums = [
        row["APPLICATION_NUM"]
        for row in application_rows
    ]

    account_numbers = [
        row["ACCOUNT_NUMBER"]
        for row in application_rows
    ]

    mkr_search_ids = [
        row["MKR_SEARCH_ID"]
        for row in application_rows
    ]

    contract_account_numbers = {
        (row.get("ACCOUNT_NUMBER") or "").strip()
        for row in contract_rows
    }

    if len(application_rows) != len(contract_rows):
        raise ValueError(
            "Application count does not match contract count."
        )

    for values, field_name in [
        (application_ids, "APPLICATION_ID"),
        (application_nums, "APPLICATION_NUM"),
        (account_numbers, "ACCOUNT_NUMBER"),
        (mkr_search_ids, "MKR_SEARCH_ID")
    ]:
        if len(values) != len(set(values)):
            raise ValueError(
                f"Duplicate {field_name} values found."
            )

    if set(account_numbers) != contract_account_numbers:
        raise ValueError(
            "Some contracts do not have a matching application."
        )

    for row in application_rows:
        application_date = parse_date(
            row["APPLICATION_DATE"]
        )

        decision_date = parse_date(
            row["DECISION_DATE"]
        )

        value_date = parse_date(
            row["VALUE_DATE"]
        )

        if application_date > decision_date:
            raise ValueError(
                "APPLICATION_DATE is after DECISION_DATE."
            )

        if decision_date > value_date:
            raise ValueError(
                "DECISION_DATE is after VALUE_DATE."
            )

        if (
            value_date - application_date
        ).days > MAX_DAYS_BETWEEN_MKR_AND_DISBURSEMENT:
            raise ValueError(
                "MKR/application date is more than seven days "
                "before contract VALUE_DATE."
            )


# ============================================================
# MAIN
# ============================================================

def main_generate_applications():
    customer_creation_dates = (
        load_customer_creation_dates()
    )

    contract_rows = read_csv_rows(
        file_path=CONTRACT_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "APPLICATION_NUM",
            "CUSTOMER_ID",
            "VALUE_DATE",
            "PRODUCT_CODE",
            "BRANCH_CODE"
        },
        table_name="Contracts"
    )

    profiles_by_account = load_contract_profiles()

    application_rows = []

    for sequence_number, contract_row in enumerate(
        contract_rows,
        start=1
    ):
        account_number = (
            contract_row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        profile_row = profiles_by_account.get(
            account_number
        )

        if not profile_row:
            raise ValueError(
                "Missing contract profile for ACCOUNT_NUMBER: "
                f"{account_number}"
            )

        application_rows.append(
            create_application_row(
                contract_row=contract_row,
                profile_row=profile_row,
                customer_creation_dates=customer_creation_dates,
                sequence_number=sequence_number
            )
        )

    validate_applications(
        application_rows=application_rows,
        contract_rows=contract_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_applications_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_applications_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_applications_generation_summary.json"
    )

    write_csv(
        full_output_file,
        APPLICATION_COLUMNS,
        application_rows
    )

    write_csv(
        sample_output_file,
        APPLICATION_COLUMNS,
        application_rows[:100]
    )

    application_day_differences = [
        (
            parse_date(row["VALUE_DATE"])
            - parse_date(row["APPLICATION_DATE"])
        ).days
        for row in application_rows
    ]

    summary = {
        "table_name": "RS_APPLICATIONS",
        "generated_rows": len(application_rows),
        "decision_population": (
            "Approved and disbursed internal-loan applications"
        ),
        "contract_join_rule": (
            "RS_APPLICATIONS.ACCOUNT_NUMBER "
            "= RS_CONTRACTS.ACCOUNT_NUMBER"
        ),
        "future_mkr_join_rule": (
            "RS_APPLICATIONS.MKR_SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "minimum_days_before_value_date": min(
            application_day_differences
        ),
        "maximum_days_before_value_date": max(
            application_day_differences
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Application generation completed successfully.")
    print(f"Generated applications: {len(application_rows):,}")
    print(
        "Contracts without a matching application: 0"
    )
    print(
        "MKR/application timing range before VALUE_DATE: "
        f"{min(application_day_differences)}–"
        f"{max(application_day_differences)} days"
    )
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_applications()

Application generation completed successfully.
Generated applications: 20,000
Contracts without a matching application: 0
MKR/application timing range before VALUE_DATE: 0–7 days
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_applications_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_applications_base_sample.csv


In [9]:
import csv
import json
import random
from datetime import datetime
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260630
random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
)

APPLICATION_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_applications_base.csv"
)

CONTRACT_PROFILE_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_contract_profile_base.csv"
)

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"


# ============================================================
# REQUIRED MASTER COLUMNS
# ============================================================

MKR_MASTER_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "BALANCE",
    "REPORTID",
    "REPORTINGDATE",
    "INSERT_DATE",
    "DOCUMENT_NO"
]


# ============================================================
# FILE FUNCTIONS
# ============================================================

def read_csv_rows(file_path, required_columns, table_name):
    """Reads a CSV and checks required columns."""

    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        available_columns = set(reader.fieldnames or [])
        missing_columns = required_columns - available_columns

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return list(reader)


def write_csv(file_path, fieldnames, rows):
    """Writes a UTF-8 CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# LOOKUP TABLES
# ============================================================

def load_customers():
    """
    Creates:
    CUSTOMER_NO/CIF -> customer information
    """

    customer_rows = read_csv_rows(
        file_path=CUSTOMER_FILE,
        required_columns={
            "CUSTOMER_NO",
            "CUSTOMER_TYPE",
            "PIN"
        },
        table_name="Customers"
    )

    customers_by_cif = {}

    for row in customer_rows:
        customer_no = (
            row.get("CUSTOMER_NO") or ""
        ).strip()

        if customer_no:
            customers_by_cif[customer_no] = {
                "CUSTOMER_TYPE": (
                    row.get("CUSTOMER_TYPE") or "I"
                ).strip().upper(),
                "PIN": (row.get("PIN") or "").strip()
            }

    return customers_by_cif


def load_profiles():
    """
    Creates:
    ACCOUNT_NUMBER -> hidden generation risk pattern

    This is used only to make synthetic bureau data
    realistically related to later repayment behavior.
    """

    profile_rows = read_csv_rows(
        file_path=CONTRACT_PROFILE_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "RISK_PATTERN"
        },
        table_name="Contract profiles"
    )

    profiles_by_account = {}

    for row in profile_rows:
        account_number = (
            row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        if account_number:
            profiles_by_account[account_number] = row

    return profiles_by_account


# ============================================================
# BUREAU BALANCE GENERATION
# ============================================================

def generate_bureau_balance(customer_type, risk_pattern):
    """
    Creates total pre-application bureau debt.

    This balance represents existing liabilities only.
    It does not include the new loan being applied for.
    """

    if risk_pattern == "CLEAN":
        zero_probability = 0.28
        minimum = 200
        maximum = 15_000
        mode = 2_500

    elif risk_pattern == "SHORT_LATE":
        zero_probability = 0.15
        minimum = 500
        maximum = 25_000
        mode = 6_000

    elif risk_pattern == "SERIOUS_LATE":
        zero_probability = 0.07
        minimum = 1_000
        maximum = 45_000
        mode = 14_000

    else:  # BAD
        zero_probability = 0.03
        minimum = 3_000
        maximum = 75_000
        mode = 28_000

    if random.random() < zero_probability:
        return 0.00

    balance = random.triangular(
        low=minimum,
        high=maximum,
        mode=mode
    )

    if customer_type in {"C", "B"}:
        balance *= random.uniform(4, 20)

    return round(balance, 2)


# ============================================================
# APPLICATION MKR MASTER ROWS
# ============================================================

def create_application_mkr_row(
    application_row,
    customer_row,
    profile_row
):
    """
    Creates one MKR master row for a loan application.

    Individual:
        PIN_CODE filled
        DOCUMENT_NO blank

    Company / bank:
        PIN_CODE blank
        DOCUMENT_NO = CIF
    """

    search_id = (
        application_row.get("MKR_SEARCH_ID") or ""
    ).strip()

    customer_id = (
        application_row.get("CUSTOMER_ID") or ""
    ).strip()

    application_date = (
        application_row.get("APPLICATION_DATE") or ""
    ).strip()

    account_number = (
        application_row.get("ACCOUNT_NUMBER") or ""
    ).strip()

    customer_type = customer_row["CUSTOMER_TYPE"]
    customer_pin = customer_row["PIN"]

    risk_pattern = (
        profile_row.get("RISK_PATTERN") or "CLEAN"
    ).strip()

    if customer_type == "I":
        pin_code = customer_pin
        document_no = ""

        if not pin_code:
            raise ValueError(
                f"Individual customer has no PIN: {customer_id}"
            )

    else:
        pin_code = ""
        document_no = customer_id

    balance = generate_bureau_balance(
        customer_type=customer_type,
        risk_pattern=risk_pattern
    )

    sequence_number = int(search_id[-10:])

    return {
        "SEARCH_ID": search_id,
        "PIN_CODE": pin_code,
        "BALANCE": f"{balance:.2f}",
        "REPORTID": f"RPTA{sequence_number:011d}",
        "REPORTINGDATE": application_date,
        "INSERT_DATE": application_date,
        "DOCUMENT_NO": document_no
    }


# ============================================================
# VALIDATION
# ============================================================

def validate_application_mkr_data(
    application_rows,
    application_master_rows,
    unified_master_rows
):
    """Checks all applications can directly join to MKR master."""

    application_search_ids = {
        (row.get("MKR_SEARCH_ID") or "").strip()
        for row in application_rows
    }

    new_master_search_ids = {
        (row.get("SEARCH_ID") or "").strip()
        for row in application_master_rows
    }

    unified_master_search_ids = [
        (row.get("SEARCH_ID") or "").strip()
        for row in unified_master_rows
    ]

    if len(application_rows) != len(application_master_rows):
        raise ValueError(
            "Application count does not match application MKR rows."
        )

    if application_search_ids != new_master_search_ids:
        raise ValueError(
            "Some applications do not have matching MKR master rows."
        )

    if len(unified_master_search_ids) != len(
        set(unified_master_search_ids)
    ):
        raise ValueError(
            "Duplicate SEARCH_ID values exist in unified MKR master."
        )

    for application_row, master_row in zip(
        application_rows,
        application_master_rows
    ):
        if (
            application_row["MKR_SEARCH_ID"]
            != master_row["SEARCH_ID"]
        ):
            raise ValueError("MKR_SEARCH_ID mismatch found.")

        if (
            application_row["APPLICATION_DATE"]
            != master_row["REPORTINGDATE"]
        ):
            raise ValueError(
                "Application date and MKR reporting date differ."
            )


# ============================================================
# MAIN
# ============================================================

def main_generate_application_mkr_master():
    customers_by_cif = load_customers()
    profiles_by_account = load_profiles()

    application_rows = read_csv_rows(
        file_path=APPLICATION_FILE,
        required_columns={
            "MKR_SEARCH_ID",
            "CUSTOMER_ID",
            "ACCOUNT_NUMBER",
            "APPLICATION_DATE"
        },
        table_name="Applications"
    )

    existing_master_rows = read_csv_rows(
        file_path=MKR_MASTER_FILE,
        required_columns=set(MKR_MASTER_COLUMNS),
        table_name="Existing MKR master"
    )

    # Makes rerunning safe:
    # removes old application-search rows before recreating them.
    general_master_rows = [
        row
        for row in existing_master_rows
        if not (
            row.get("SEARCH_ID") or ""
        ).strip().startswith("MKRAPP")
    ]

    application_master_rows = []

    for application_row in application_rows:
        customer_id = (
            application_row.get("CUSTOMER_ID") or ""
        ).strip()

        account_number = (
            application_row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        customer_row = customers_by_cif.get(customer_id)
        profile_row = profiles_by_account.get(account_number)

        if not customer_row:
            raise ValueError(
                f"Customer not found for application: {customer_id}"
            )

        if not profile_row:
            raise ValueError(
                "Contract profile not found for account: "
                f"{account_number}"
            )

        application_master_rows.append(
            create_application_mkr_row(
                application_row=application_row,
                customer_row=customer_row,
                profile_row=profile_row
            )
        )

    unified_master_rows = (
        general_master_rows + application_master_rows
    )

    validate_application_mkr_data(
        application_rows=application_rows,
        application_master_rows=application_master_rows,
        unified_master_rows=unified_master_rows
    )

    application_master_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_application_master_base.csv"
    )

    application_master_sample_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_application_master_base_sample.csv"
    )

    unified_master_sample_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_master_base_sample.csv"
    )

    summary_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_application_master_generation_summary.json"
    )

    # Separate application-only source file.
    write_csv(
        application_master_file,
        MKR_MASTER_COLUMNS,
        application_master_rows
    )

    # Unified file used later for Oracle loading and joins.
    write_csv(
        MKR_MASTER_FILE,
        MKR_MASTER_COLUMNS,
        unified_master_rows
    )

    write_csv(
        application_master_sample_file,
        MKR_MASTER_COLUMNS,
        application_master_rows[:100]
    )

    # 50 general + 50 application records for a useful public sample.
    write_csv(
        unified_master_sample_file,
        MKR_MASTER_COLUMNS,
        general_master_rows[:50] + application_master_rows[:50]
    )

    summary = {
        "general_mkr_searches": len(general_master_rows),
        "new_application_mkr_searches": len(
            application_master_rows
        ),
        "total_unified_mkr_searches": len(
            unified_master_rows
        ),
        "application_join_rule": (
            "RS_APPLICATIONS.MKR_SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "application_timing_rule": (
            "RS_MKR_MASTER.REPORTINGDATE "
            "= RS_APPLICATIONS.APPLICATION_DATE"
        ),
        "new_loan_excluded_from_bureau_balance": True,
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Application MKR master generation completed successfully.")
    print(
        f"General MKR searches retained: "
        f"{len(general_master_rows):,}"
    )
    print(
        f"New application MKR searches: "
        f"{len(application_master_rows):,}"
    )
    print(
        f"Unified MKR master rows: "
        f"{len(unified_master_rows):,}"
    )
    print("Applications without a matching MKR search: 0")
    print(
        "New-loan balances included in pre-loan bureau data: 0"
    )
    print(f"Application MKR file: {application_master_file}")
    print(f"Unified MKR file updated: {MKR_MASTER_FILE}")


main_generate_application_mkr_master()

Application MKR master generation completed successfully.
General MKR searches retained: 40,000
New application MKR searches: 20,000
Unified MKR master rows: 60,000
Applications without a matching MKR search: 0
New-loan balances included in pre-loan bureau data: 0
Application MKR file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_application_master_base.csv
Unified MKR file updated: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_master_base.csv


In [10]:
import csv
import hashlib
import json
import random
from datetime import date, datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260701

random.seed(RANDOM_SEED)

INTERNAL_CODE = "1000"
INTERNAL_NAME = "INTERNAL"

EXTERNAL_NAME = "EXTERNAL"
EXTERNAL_CODES = [
    str(code)
    for code in range(1001, 1021)
]


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

APPLICATION_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_applications_base.csv"
)

APPLICATION_MKR_MASTER_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_master_base.csv"
)

CONTRACT_PROFILE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_contract_profile_base.csv"
)

UNIFIED_LIABILITIES_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_liabilities_base.csv"
)

APPLICATION_LIABILITIES_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_liabilities_base.csv"
)

SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_liabilities_base_sample.csv"
)

APPLICATION_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_application_liabilities_base_sample.csv"
)

SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_liabilities_generation_summary.json"
)


# ============================================================
# TABLE COLUMNS
# ============================================================

LIABILITY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "ACCOUNTNO",
    "BANKID",
    "BANKNAME",
    "COLLATERALANYINFO",
    "COLLATERALCODE",
    "COLLATERALMARKETVALUE",
    "COLLATERALREGISTRYAGENCY",
    "COLLATERALREGISTRYDATE",
    "COLLATERALREGISTRYNO",
    "COLLATERALTYPENAME",
    "CREDITPURPOSE",
    "CREDITPURPOSENAME",
    "CREDITSTATUS",
    "CREDITSTATUSCLOSEDATE",
    "CREDITSTATUSNAME",
    "CREDITTYPE",
    "CREDITTYPENAME",
    "CURRENCY",
    "CURRENCYNAME",
    "DAYSINTERESTOVERDUE",
    "DAYSMAINSUMOVERDUE",
    "GRANTEDON",
    "ID",
    "INITIALAMOUNT",
    "INTERESTRATE",
    "LASTPAYMENTDATE",
    "LASTUPDATEDDATE",
    "LINEAMOUNT",
    "MKRID",
    "MONTHLYPAYMENTAMOUNT",
    "ORGIDTYPE",
    "OUTSTANDINGDEBTINTEREST",
    "OUTSTANDINGDEBTMAIN",
    "PROLONGATIONS",
    "CONTRACTDUEON",
    "INS_DTTM"
]


# ============================================================
# CREDIT REFERENCE DATA
# ============================================================

INDIVIDUAL_CREDIT_TYPES = [
    {
        "code": "001",
        "name": "Consumer Loan",
        "purpose_code": "001",
        "purpose_name": "Consumer Expenses",
        "secured": False,
        "line": False,
        "term_min": 12,
        "term_max": 60
    },
    {
        "code": "002",
        "name": "Mortgage Loan",
        "purpose_code": "002",
        "purpose_name": "Property Purchase",
        "secured": True,
        "collateral_code": "001",
        "collateral_type": "Residential Property",
        "line": False,
        "term_min": 60,
        "term_max": 240
    },
    {
        "code": "003",
        "name": "Auto Loan",
        "purpose_code": "003",
        "purpose_name": "Vehicle Purchase",
        "secured": True,
        "collateral_code": "002",
        "collateral_type": "Vehicle",
        "line": False,
        "term_min": 24,
        "term_max": 84
    },
    {
        "code": "004",
        "name": "Credit Card",
        "purpose_code": "004",
        "purpose_name": "Card Spending",
        "secured": False,
        "line": True,
        "term_min": 12,
        "term_max": 48
    }
]

CORPORATE_CREDIT_TYPES = [
    {
        "code": "005",
        "name": "Business Loan",
        "purpose_code": "005",
        "purpose_name": "Business Development",
        "secured": True,
        "collateral_code": "003",
        "collateral_type": "Commercial Property",
        "line": False,
        "term_min": 12,
        "term_max": 84
    },
    {
        "code": "006",
        "name": "Business Credit Line",
        "purpose_code": "006",
        "purpose_name": "Working Capital",
        "secured": False,
        "line": True,
        "term_min": 12,
        "term_max": 60
    },
    {
        "code": "007",
        "name": "Equipment Financing",
        "purpose_code": "007",
        "purpose_name": "Equipment Purchase",
        "secured": True,
        "collateral_code": "007",
        "collateral_type": "Equipment",
        "line": False,
        "term_min": 24,
        "term_max": 72
    }
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def parse_date(value):
    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def to_decimal(value):
    try:
        return Decimal(str(value)).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )
    except Exception:
        return Decimal("0.00")


def format_decimal(value):
    return format(
        value.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        ),
        "f"
    )


def add_months(start_date, months):
    new_month = start_date.month - 1 + months
    new_year = start_date.year + new_month // 12
    new_month = new_month % 12 + 1

    days_in_month = [
        31,
        29 if new_year % 4 == 0 and (
            new_year % 100 != 0
            or new_year % 400 == 0
        ) else 28,
        31, 30, 31, 30,
        31, 31, 30, 31, 30, 31
    ]

    new_day = min(
        start_date.day,
        days_in_month[new_month - 1]
    )

    return date(new_year, new_month, new_day)


def read_csv_rows(file_path, required_columns, table_name):
    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return reader.fieldnames, list(reader)


def write_csv(file_path, fieldnames, rows):
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# LOOKUPS
# ============================================================

def load_application_lookup():
    _, application_rows = read_csv_rows(
        file_path=APPLICATION_FILE,
        required_columns={
            "MKR_SEARCH_ID",
            "ACCOUNT_NUMBER",
            "CUSTOMER_TYPE"
        },
        table_name="Applications"
    )

    return {
        (row.get("MKR_SEARCH_ID") or "").strip(): row
        for row in application_rows
    }


def load_profile_lookup():
    _, profile_rows = read_csv_rows(
        file_path=CONTRACT_PROFILE_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "RISK_PATTERN"
        },
        table_name="Contract profiles"
    )

    return {
        (row.get("ACCOUNT_NUMBER") or "").strip(): row
        for row in profile_rows
    }


# ============================================================
# SYNTHETIC BUREAU LOGIC
# ============================================================

def choose_liability_count(balance, risk_pattern):
    if balance <= Decimal("0.00"):
        return 0

    if risk_pattern == "CLEAN":
        choices = [1, 2, 3]
        weights = [60, 32, 8]

    elif risk_pattern == "SHORT_LATE":
        choices = [1, 2, 3, 4]
        weights = [35, 40, 20, 5]

    elif risk_pattern == "SERIOUS_LATE":
        choices = [1, 2, 3, 4]
        weights = [20, 40, 28, 12]

    else:
        choices = [1, 2, 3, 4, 5]
        weights = [10, 28, 32, 20, 10]

    return random.choices(
        choices,
        weights=weights,
        k=1
    )[0]


def split_balance(total_balance, liability_count):
    total_cents = int(total_balance * 100)

    if liability_count == 1:
        return [total_cents]

    minimum_cents = 100
    required_cents = minimum_cents * liability_count

    if total_cents <= required_cents:
        return [total_cents]

    remaining_cents = total_cents - required_cents

    weights = [
        random.uniform(0.4, 1.6)
        for _ in range(liability_count)
    ]

    total_weight = sum(weights)

    allocations = [
        minimum_cents + int(
            remaining_cents * weight / total_weight
        )
        for weight in weights
    ]

    allocations[-1] += total_cents - sum(allocations)

    return allocations


def choose_credit_type(customer_type):
    if customer_type == "I":
        return random.choices(
            INDIVIDUAL_CREDIT_TYPES,
            weights=[43, 13, 17, 27],
            k=1
        )[0]

    return random.choices(
        CORPORATE_CREDIT_TYPES,
        weights=[42, 35, 23],
        k=1
    )[0]


def choose_status_and_dpd(risk_pattern):
    if risk_pattern == "CLEAN":
        overdue_days = random.choices(
            [0, random.randint(1, 14)],
            weights=[94, 6],
            k=1
        )[0]

    elif risk_pattern == "SHORT_LATE":
        overdue_days = random.choices(
            [
                0,
                random.randint(1, 29),
                random.randint(30, 59)
            ],
            weights=[35, 45, 20],
            k=1
        )[0]

    elif risk_pattern == "SERIOUS_LATE":
        overdue_days = random.choices(
            [
                0,
                random.randint(30, 89),
                random.randint(90, 120)
            ],
            weights=[18, 60, 22],
            k=1
        )[0]

    else:
        overdue_days = random.choices(
            [
                random.randint(30, 89),
                random.randint(90, 180),
                random.randint(181, 360)
            ],
            weights=[25, 52, 23],
            k=1
        )[0]

    if overdue_days == 0:
        return "001", "Active", 0

    if risk_pattern in {"SERIOUS_LATE", "BAD"}:
        status = random.choices(
            [
                ("005", "Past Due"),
                ("007", "Restructured")
            ],
            weights=[78, 22],
            k=1
        )[0]

        return status[0], status[1], overdue_days

    return "005", "Past Due", overdue_days


def create_bank_assignments(search_id, liability_count):
    """
    1000 = INTERNAL.

    External code mappings are local to this SEARCH_ID.
    The same external number has no fixed bank identity
    in another SEARCH_ID.
    """

    hash_value = hashlib.sha256(
        f"{search_id}|bank-mask-v2".encode("utf-8")
    ).hexdigest()

    local_rng = random.Random(
        int(hash_value[:16], 16)
    )

    shuffled_codes = EXTERNAL_CODES.copy()
    local_rng.shuffle(shuffled_codes)

    use_internal = (
        liability_count > 0
        and local_rng.random() < 0.28
    )

    assignments = []

    for position in range(liability_count):
        if use_internal and position == 0:
            assignments.append(
                (INTERNAL_CODE, INTERNAL_NAME)
            )
        else:
            external_position = (
                position - 1
                if use_internal
                else position
            )

            assignments.append(
                (
                    shuffled_codes[external_position],
                    EXTERNAL_NAME
                )
            )

    local_rng.shuffle(assignments)

    return assignments


# ============================================================
# LIABILITY ROW CREATION
# ============================================================

def create_application_liabilities(
    application_master_row,
    application_row,
    profile_row,
    starting_id
):
    search_id = (
        application_master_row.get("SEARCH_ID") or ""
    ).strip()

    pin_code = (
        application_master_row.get("PIN_CODE") or ""
    ).strip()

    reporting_date = parse_date(
        application_master_row.get("REPORTINGDATE") or ""
    )

    balance = to_decimal(
        application_master_row.get("BALANCE") or "0"
    )

    customer_type = (
        application_row.get("CUSTOMER_TYPE") or "I"
    ).strip().upper()

    risk_pattern = (
        profile_row.get("RISK_PATTERN") or "CLEAN"
    ).strip()

    if not search_id:
        raise ValueError("Blank application SEARCH_ID found.")

    if not reporting_date:
        raise ValueError(
            f"Invalid REPORTINGDATE for {search_id}"
        )

    liability_count = choose_liability_count(
        balance=balance,
        risk_pattern=risk_pattern
    )

    if liability_count == 0:
        return []

    balance_parts = split_balance(
        total_balance=balance,
        liability_count=liability_count
    )

    bank_assignments = create_bank_assignments(
        search_id=search_id,
        liability_count=liability_count
    )

    created_rows = []

    for slot, balance_cents in enumerate(
        balance_parts,
        start=1
    ):
        total_debt = Decimal(balance_cents) / Decimal("100")

        bank_id, bank_name = bank_assignments[slot - 1]

        credit_type = choose_credit_type(customer_type)

        (
            status_code,
            status_name,
            overdue_days
        ) = choose_status_and_dpd(risk_pattern)

        term_months = random.randint(
            credit_type["term_min"],
            credit_type["term_max"]
        )

        granted_on = reporting_date - timedelta(
            days=random.randint(
                max(60, term_months * 8),
                max(180, term_months * 35)
            )
        )

        contract_due_on = add_months(
            granted_on,
            term_months
        )

        if overdue_days > 0:
            last_payment_date = reporting_date - timedelta(
                days=overdue_days
            )
        else:
            last_payment_date = reporting_date - timedelta(
                days=random.randint(0, 28)
            )

        interest_ratio = Decimal(
            str(
                random.uniform(
                    0.04 if overdue_days else 0.01,
                    0.20 if overdue_days else 0.08
                )
            )
        )

        interest_debt = (
            total_debt * interest_ratio
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        main_debt = total_debt - interest_debt

        initial_amount = (
            total_debt
            * Decimal(
                str(random.uniform(1.15, 3.40))
            )
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        interest_rate = round(
            random.uniform(8.0, 30.0),
            2
        )

        if credit_type["line"]:
            line_amount = (
                initial_amount
                * Decimal(
                    str(random.uniform(1.05, 1.50))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            monthly_payment = (
                total_debt
                * Decimal(
                    str(random.uniform(0.05, 0.16))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

        else:
            line_amount = Decimal("0.00")

            monthly_payment = max(
                Decimal("20.00"),
                (
                    total_debt
                    * Decimal(
                        str(random.uniform(0.03, 0.12))
                    )
                ).quantize(
                    Decimal("0.01"),
                    rounding=ROUND_HALF_UP
                )
            )

        is_secured = credit_type["secured"]

        if is_secured:
            collateral_value = (
                initial_amount
                * Decimal(
                    str(random.uniform(1.05, 1.50))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            collateral_code = credit_type.get(
                "collateral_code",
                "007"
            )

            collateral_type = credit_type.get(
                "collateral_type",
                "Other Collateral"
            )

            collateral_registry_date = (
                granted_on + timedelta(
                    days=random.randint(0, 25)
                )
            )

            collateral_registry_no = (
                f"COL{starting_id + slot:010d}"
            )

            collateral_registry_agency = (
                "COLLATERAL_REGISTRY"
            )

            collateral_info = (
                f"{collateral_type.upper()} SECURES CREDIT"
            )

        else:
            collateral_value = Decimal("0.00")
            collateral_code = "000"
            collateral_type = ""
            collateral_registry_date = ""
            collateral_registry_no = ""
            collateral_registry_agency = ""
            collateral_info = ""

        liability_id = str(starting_id + slot)

        created_timestamp = datetime.combine(
            reporting_date,
            datetime.min.time()
        ) + timedelta(
            hours=random.randint(8, 19),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        created_rows.append({
            "SEARCH_ID": search_id,
            "PIN_CODE": pin_code,
            "ACCOUNTNO": (
                f"BUREAU{search_id[-10:]}{slot:02d}"
            ),
            "BANKID": bank_id,
            "BANKNAME": bank_name,
            "COLLATERALANYINFO": collateral_info,
            "COLLATERALCODE": collateral_code,
            "COLLATERALMARKETVALUE": format_decimal(
                collateral_value
            ),
            "COLLATERALREGISTRYAGENCY": (
                collateral_registry_agency
            ),
            "COLLATERALREGISTRYDATE": (
                collateral_registry_date.isoformat()
                if collateral_registry_date
                else ""
            ),
            "COLLATERALREGISTRYNO": collateral_registry_no,
            "COLLATERALTYPENAME": collateral_type,
            "CREDITPURPOSE": credit_type["purpose_code"],
            "CREDITPURPOSENAME": credit_type["purpose_name"],
            "CREDITSTATUS": status_code,
            "CREDITSTATUSCLOSEDATE": "",
            "CREDITSTATUSNAME": status_name,
            "CREDITTYPE": credit_type["code"],
            "CREDITTYPENAME": credit_type["name"],
            "CURRENCY": "AZN",
            "CURRENCYNAME": "Azerbaijani Manat",
            "DAYSINTERESTOVERDUE": str(overdue_days),
            "DAYSMAINSUMOVERDUE": str(overdue_days),
            "GRANTEDON": granted_on.isoformat(),
            "ID": liability_id,
            "INITIALAMOUNT": format_decimal(initial_amount),
            "INTERESTRATE": str(interest_rate),
            "LASTPAYMENTDATE": last_payment_date.isoformat(),
            "LASTUPDATEDDATE": reporting_date.isoformat(),
            "LINEAMOUNT": format_decimal(line_amount),
            "MKRID": f"MKR-{liability_id}",
            "MONTHLYPAYMENTAMOUNT": format_decimal(
                monthly_payment
            ),
            "ORGIDTYPE": "001",
            "OUTSTANDINGDEBTINTEREST": format_decimal(
                interest_debt
            ),
            "OUTSTANDINGDEBTMAIN": format_decimal(
                main_debt
            ),
            "PROLONGATIONS": str(
                random.randint(1, 3)
                if status_code == "007"
                else 0
            ),
            "CONTRACTDUEON": contract_due_on.isoformat(),
            "INS_DTTM": created_timestamp.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        })

    return created_rows


# ============================================================
# VALIDATION
# ============================================================

def validate_application_liabilities(
    application_master_rows,
    application_liability_rows
):
    master_by_search = {
        row["SEARCH_ID"].strip(): to_decimal(
            row["BALANCE"]
        )
        for row in application_master_rows
    }

    child_balance_by_search = {
        search_id: Decimal("0.00")
        for search_id in master_by_search
    }

    liability_ids = set()

    for row in application_liability_rows:
        search_id = row["SEARCH_ID"].strip()

        if search_id not in master_by_search:
            raise ValueError(
                f"Unexpected application liability SEARCH_ID: "
                f"{search_id}"
            )

        if row["ID"] in liability_ids:
            raise ValueError(
                f"Duplicate liability ID: {row['ID']}"
            )

        liability_ids.add(row["ID"])

        bank_id = row["BANKID"].strip()
        bank_name = row["BANKNAME"].strip()

        if bank_name == INTERNAL_NAME:
            if bank_id != INTERNAL_CODE:
                raise ValueError(
                    "INTERNAL row does not use code 1000."
                )

        elif bank_name == EXTERNAL_NAME:
            if bank_id not in EXTERNAL_CODES:
                raise ValueError(
                    "EXTERNAL row has invalid bank code."
                )

        else:
            raise ValueError(
                f"Unmasked bank name found: {bank_name}"
            )

        child_balance_by_search[search_id] += (
            to_decimal(row["OUTSTANDINGDEBTMAIN"])
            + to_decimal(row["OUTSTANDINGDEBTINTEREST"])
        )

    for search_id, master_balance in master_by_search.items():
        child_balance = child_balance_by_search[search_id]

        if child_balance != master_balance:
            raise ValueError(
                f"Balance mismatch for {search_id}. "
                f"Master={master_balance}; "
                f"Children={child_balance}"
            )


# ============================================================
# MAIN
# ============================================================

def main_generate_application_liabilities():
    application_lookup = load_application_lookup()
    profile_lookup = load_profile_lookup()

    _, application_master_rows = read_csv_rows(
        file_path=APPLICATION_MKR_MASTER_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "BALANCE",
            "REPORTINGDATE"
        },
        table_name="Application MKR master"
    )

    liability_columns, all_existing_liability_rows = (
        read_csv_rows(
            file_path=UNIFIED_LIABILITIES_FILE,
            required_columns=set(LIABILITY_COLUMNS),
            table_name="Unified MKR liabilities"
        )
    )

    # Remove any old application rows if this cell is rerun.
    general_liability_rows = [
        row
        for row in all_existing_liability_rows
        if not (
            row.get("SEARCH_ID") or ""
        ).strip().startswith("MKRAPP")
    ]

    existing_ids = []

    for row in general_liability_rows:
        try:
            existing_ids.append(int(row["ID"]))
        except (TypeError, ValueError):
            pass

    next_id = max(existing_ids, default=0) + 1

    application_liability_rows = []

    for application_master_row in application_master_rows:
        search_id = (
            application_master_row.get("SEARCH_ID") or ""
        ).strip()

        application_row = application_lookup.get(search_id)

        if not application_row:
            raise ValueError(
                f"Application not found for SEARCH_ID: {search_id}"
            )

        account_number = (
            application_row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        profile_row = profile_lookup.get(account_number)

        if not profile_row:
            raise ValueError(
                f"Profile not found for account: {account_number}"
            )

        new_rows = create_application_liabilities(
            application_master_row=application_master_row,
            application_row=application_row,
            profile_row=profile_row,
            starting_id=next_id
        )

        application_liability_rows.extend(new_rows)
        next_id += len(new_rows)

    validate_application_liabilities(
        application_master_rows=application_master_rows,
        application_liability_rows=application_liability_rows
    )

    unified_liability_rows = (
        general_liability_rows
        + application_liability_rows
    )

    write_csv(
        APPLICATION_LIABILITIES_FILE,
        LIABILITY_COLUMNS,
        application_liability_rows
    )

    write_csv(
        UNIFIED_LIABILITIES_FILE,
        liability_columns,
        unified_liability_rows
    )

    write_csv(
        APPLICATION_SAMPLE_FILE,
        LIABILITY_COLUMNS,
        application_liability_rows[:100]
    )

    write_csv(
        SAMPLE_FILE,
        LIABILITY_COLUMNS,
        general_liability_rows[:50]
        + application_liability_rows[:50]
    )

    internal_rows = sum(
        1 for row in application_liability_rows
        if row["BANKNAME"] == INTERNAL_NAME
    )

    external_rows = sum(
        1 for row in application_liability_rows
        if row["BANKNAME"] == EXTERNAL_NAME
    )

    application_searches_with_liabilities = len({
        row["SEARCH_ID"]
        for row in application_liability_rows
    })

    summary = {
        "application_mkr_searches": len(
            application_master_rows
        ),
        "application_liability_rows": len(
            application_liability_rows
        ),
        "application_searches_with_liabilities": (
            application_searches_with_liabilities
        ),
        "application_searches_without_liabilities": (
            len(application_master_rows)
            - application_searches_with_liabilities
        ),
        "internal_liability_rows": internal_rows,
        "external_liability_rows": external_rows,
        "unified_liability_rows": len(
            unified_liability_rows
        ),
        "balance_validation": (
            "sum(OUTSTANDINGDEBTMAIN + "
            "OUTSTANDINGDEBTINTEREST) "
            "= application MKR master BALANCE"
        ),
        "bank_masking_policy": {
            "internal": "1000 / INTERNAL",
            "external": "1001-1020 / EXTERNAL",
            "external_mapping_scope": "per SEARCH_ID"
        },
        "random_seed": RANDOM_SEED
    }

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print(
        "Application MKR liabilities generation completed "
        "successfully."
    )
    print(
        f"New application liability rows: "
        f"{len(application_liability_rows):,}"
    )
    print(
        "Application searches with liabilities: "
        f"{application_searches_with_liabilities:,}"
    )
    print(
        "Application searches without liabilities "
        "(zero bureau balance): "
        f"{len(application_master_rows) - application_searches_with_liabilities:,}"
    )
    print(
        f"INTERNAL rows (1000): {internal_rows:,}"
    )
    print(
        f"EXTERNAL rows (1001-1020): {external_rows:,}"
    )
    print(
        f"Unified MKR liabilities rows: "
        f"{len(unified_liability_rows):,}"
    )
    print("Parent-to-child balance validation: passed")
    print(
        f"Application-only file: "
        f"{APPLICATION_LIABILITIES_FILE}"
    )
    print(
        f"Unified file updated: "
        f"{UNIFIED_LIABILITIES_FILE}"
    )


main_generate_application_liabilities()

Application MKR liabilities generation completed successfully.
New application liability rows: 27,553
Application searches with liabilities: 15,514
Application searches without liabilities (zero bureau balance): 4,486
INTERNAL rows (1000): 4,271
EXTERNAL rows (1001-1020): 23,282
Unified MKR liabilities rows: 98,474
Parent-to-child balance validation: passed
Application-only file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_application_liabilities_base.csv
Unified file updated: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_liabilities_base.csv


In [11]:
import csv
import json
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

APPLICATION_MKR_MASTER_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_master_base.csv"
)

UNIFIED_LIABILITIES_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_liabilities_base.csv"
)

UNIFIED_HISTORY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_liability_history_base.csv"
)

APPLICATION_HISTORY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_liability_history_base.csv"
)

SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_liability_history_base_sample.csv"
)

APPLICATION_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_application_liability_history_base_sample.csv"
)

SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_liability_history_generation_summary.json"
)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

HISTORY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "LIABILITY_ID",
    "OVERDUEDAYS",
    "REPORTINGPERIOD",
    "CREDIT_STATUS",
    "INS_DTTM"
]


# ============================================================
# CSV FUNCTIONS
# ============================================================

def read_csv_rows(file_path, required_columns, table_name):
    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return reader.fieldnames, list(reader)


def write_csv(file_path, fieldnames, rows):
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# HISTORY CREATION
# ============================================================

def create_application_history_row(master_row, liability_row):
    return {
        "SEARCH_ID": liability_row["SEARCH_ID"].strip(),
        "PIN_CODE": liability_row.get("PIN_CODE", "").strip(),
        "LIABILITY_ID": liability_row["ID"].strip(),
        "OVERDUEDAYS": liability_row[
            "DAYSMAINSUMOVERDUE"
        ].strip(),
        "REPORTINGPERIOD": master_row[
            "REPORTINGDATE"
        ].strip(),
        "CREDIT_STATUS": liability_row[
            "CREDITSTATUS"
        ].strip(),
        "INS_DTTM": liability_row[
            "INS_DTTM"
        ].strip()
    }


# ============================================================
# VALIDATION
# ============================================================

def validate_application_history(
    application_master_rows,
    application_liability_rows,
    application_history_rows
):
    master_by_search_id = {
        row["SEARCH_ID"].strip(): row
        for row in application_master_rows
    }

    liability_keys = {
        (
            row["SEARCH_ID"].strip(),
            row["ID"].strip()
        )
        for row in application_liability_rows
    }

    history_keys = set()

    for row in application_history_rows:
        search_id = row["SEARCH_ID"].strip()
        liability_id = row["LIABILITY_ID"].strip()

        history_key = (search_id, liability_id)

        if search_id not in master_by_search_id:
            raise ValueError(
                f"History SEARCH_ID has no application master: {search_id}"
            )

        if history_key not in liability_keys:
            raise ValueError(
                "History row has no matching application liability: "
                f"{search_id} / {liability_id}"
            )

        if history_key in history_keys:
            raise ValueError(
                "Duplicate SEARCH_ID + LIABILITY_ID in history: "
                f"{search_id} / {liability_id}"
            )

        history_keys.add(history_key)

        parent_reporting_date = master_by_search_id[
            search_id
        ]["REPORTINGDATE"].strip()

        if row["REPORTINGPERIOD"].strip() != parent_reporting_date:
            raise ValueError(
                f"REPORTINGPERIOD mismatch for {search_id}"
            )

    if history_keys != liability_keys:
        raise ValueError(
            "Some application liabilities do not have "
            "exactly one matching history row."
        )


# ============================================================
# MAIN
# ============================================================

def main_generate_application_liability_history():
    _, application_master_rows = read_csv_rows(
        file_path=APPLICATION_MKR_MASTER_FILE,
        required_columns={
            "SEARCH_ID",
            "REPORTINGDATE"
        },
        table_name="Application MKR master"
    )

    _, unified_liability_rows = read_csv_rows(
        file_path=UNIFIED_LIABILITIES_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "ID",
            "DAYSMAINSUMOVERDUE",
            "CREDITSTATUS",
            "INS_DTTM"
        },
        table_name="Unified MKR liabilities"
    )

    history_columns, existing_history_rows = read_csv_rows(
        file_path=UNIFIED_HISTORY_FILE,
        required_columns=set(HISTORY_COLUMNS),
        table_name="Unified MKR liability history"
    )

    application_master_by_search_id = {
        row["SEARCH_ID"].strip(): row
        for row in application_master_rows
    }

    application_liability_rows = [
        row
        for row in unified_liability_rows
        if (row.get("SEARCH_ID") or "").strip().startswith("MKRAPP")
    ]

    # Safe rerun:
    # Remove older application-history rows before rebuilding them.
    general_history_rows = [
        row
        for row in existing_history_rows
        if not (
            row.get("SEARCH_ID") or ""
        ).strip().startswith("MKRAPP")
    ]

    application_history_rows = []

    for liability_row in application_liability_rows:
        search_id = liability_row["SEARCH_ID"].strip()

        master_row = application_master_by_search_id.get(search_id)

        if not master_row:
            raise ValueError(
                f"Application liability has no master row: {search_id}"
            )

        application_history_rows.append(
            create_application_history_row(
                master_row=master_row,
                liability_row=liability_row
            )
        )

    validate_application_history(
        application_master_rows=application_master_rows,
        application_liability_rows=application_liability_rows,
        application_history_rows=application_history_rows
    )

    unified_history_rows = (
        general_history_rows
        + application_history_rows
    )

    write_csv(
        APPLICATION_HISTORY_FILE,
        HISTORY_COLUMNS,
        application_history_rows
    )

    write_csv(
        UNIFIED_HISTORY_FILE,
        history_columns,
        unified_history_rows
    )

    write_csv(
        APPLICATION_SAMPLE_FILE,
        HISTORY_COLUMNS,
        application_history_rows[:100]
    )

    write_csv(
        SAMPLE_FILE,
        HISTORY_COLUMNS,
        general_history_rows[:50]
        + application_history_rows[:50]
    )

    summary = {
        "application_liability_rows": len(application_liability_rows),
        "application_history_rows": len(application_history_rows),
        "unified_history_rows": len(unified_history_rows),
        "join_rule": (
            "RS_MKR_LIABILITY_HISTORY.SEARCH_ID "
            "= RS_MKR_LIABILITIES.SEARCH_ID "
            "AND RS_MKR_LIABILITY_HISTORY.LIABILITY_ID "
            "= RS_MKR_LIABILITIES.ID"
        ),
        "validation_rule": (
            "Every application liability has exactly one history row."
        )
    }

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print(
        "Application MKR liability-history generation "
        "completed successfully."
    )
    print(
        f"Application liability rows: "
        f"{len(application_liability_rows):,}"
    )
    print(
        f"Application history rows: "
        f"{len(application_history_rows):,}"
    )
    print(
        f"Unified liability-history rows: "
        f"{len(unified_history_rows):,}"
    )
    print(
        "Every application liability has one matching "
        "history row: passed"
    )


main_generate_application_liability_history()

Application MKR liability-history generation completed successfully.
Application liability rows: 27,553
Application history rows: 27,553
Unified liability-history rows: 98,474
Every application liability has one matching history row: passed


In [12]:
import csv
import hashlib
import json
import random
from datetime import datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

LOOKBACK_DAYS = 730

INTERNAL_CODE = "1000"
INTERNAL_NAME = "INTERNAL"

EXTERNAL_NAME = "EXTERNAL"
EXTERNAL_CODES = [
    str(code)
    for code in range(1001, 1021)
]

MASKING_SEED = "risk_scoring_application_inquiry_mask_v1"


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

APPLICATION_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_applications_base.csv"
)

APPLICATION_MKR_MASTER_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_master_base.csv"
)

CONTRACT_PROFILE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_contract_profile_base.csv"
)

UNIFIED_INQUIRY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_inquiry_history_base.csv"
)

APPLICATION_INQUIRY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_inquiry_history_base.csv"
)

SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_inquiry_history_base_sample.csv"
)

APPLICATION_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "rs_mkr_application_inquiry_history_base_sample.csv"
)

SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_mkr_application_inquiry_history_generation_summary.json"
)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

INQUIRY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "INQBANKID",
    "INQBANKNAME",
    "INQDATE",
    "INQORGIDTYPE",
    "INQPURPOSEID",
    "INQPURPOSENAME",
    "INS_DTTM",
    "DOCUMENTNO",
    "REPORTINGDATE"
]


# ============================================================
# REFERENCE DATA
# ============================================================

INQUIRY_PURPOSES = [
    ("001", "New Loan Application"),
    ("002", "Credit Line Review"),
    ("003", "Credit Card Application"),
    ("004", "Mortgage Application"),
    ("005", "Business Financing"),
    ("006", "Limit Increase Review")
]


# ============================================================
# HELPERS
# ============================================================

def parse_date(value):
    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def read_csv_rows(file_path, required_columns, table_name):
    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return reader.fieldnames, list(reader)


def write_csv(file_path, fieldnames, rows):
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


def get_search_rng(search_id):
    """
    Makes generated history repeatable for each SEARCH_ID.
    The code mapping remains private to that search only.
    """

    text = f"{MASKING_SEED}|{search_id}"

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return random.Random(int(digest[:16], 16))


def choose_inquiry_count(risk_pattern, rng):
    """
    Number of earlier bureau checks in the prior two years.
    Higher-risk synthetic profiles tend to have more inquiries.
    """

    if risk_pattern == "CLEAN":
        choices = [0, 1, 2, 3]
        weights = [45, 34, 16, 5]

    elif risk_pattern == "SHORT_LATE":
        choices = [0, 1, 2, 3, 4]
        weights = [25, 34, 25, 11, 5]

    elif risk_pattern == "SERIOUS_LATE":
        choices = [0, 1, 2, 3, 4, 5]
        weights = [10, 23, 29, 21, 12, 5]

    else:  # BAD
        choices = [0, 1, 2, 3, 4, 5, 6]
        weights = [5, 13, 22, 25, 19, 11, 5]

    return rng.choices(
        choices,
        weights=weights,
        k=1
    )[0]


def create_bank_assignments(search_id, inquiry_count, rng):
    """
    INTERNAL always uses 1000.

    External codes are shuffled separately for every SEARCH_ID.
    Therefore, for example, 1001 has no stable bank identity.
    """

    shuffled_external_codes = EXTERNAL_CODES.copy()
    rng.shuffle(shuffled_external_codes)

    include_internal = (
        inquiry_count > 0
        and rng.random() < 0.22
    )

    assignments = []

    for position in range(inquiry_count):
        if include_internal and position == 0:
            assignments.append(
                (INTERNAL_CODE, INTERNAL_NAME)
            )
        else:
            external_position = (
                position - 1
                if include_internal
                else position
            )

            assignments.append(
                (
                    shuffled_external_codes[external_position],
                    EXTERNAL_NAME
                )
            )

    rng.shuffle(assignments)

    return assignments


# ============================================================
# LOOKUPS
# ============================================================

def load_application_lookup():
    _, application_rows = read_csv_rows(
        file_path=APPLICATION_FILE,
        required_columns={
            "MKR_SEARCH_ID",
            "ACCOUNT_NUMBER",
            "CUSTOMER_ID",
            "CUSTOMER_TYPE"
        },
        table_name="Applications"
    )

    return {
        (row.get("MKR_SEARCH_ID") or "").strip(): row
        for row in application_rows
    }


def load_profile_lookup():
    _, profile_rows = read_csv_rows(
        file_path=CONTRACT_PROFILE_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "RISK_PATTERN"
        },
        table_name="Contract profiles"
    )

    return {
        (row.get("ACCOUNT_NUMBER") or "").strip(): row
        for row in profile_rows
    }


# ============================================================
# APPLICATION INQUIRY HISTORY CREATION
# ============================================================

def create_application_inquiries(
    master_row,
    application_row,
    profile_row
):
    search_id = (
        master_row.get("SEARCH_ID") or ""
    ).strip()

    reporting_date = parse_date(
        master_row.get("REPORTINGDATE") or ""
    )

    pin_code = (
        master_row.get("PIN_CODE") or ""
    ).strip()

    document_no = (
        master_row.get("DOCUMENT_NO") or ""
    ).strip()

    risk_pattern = (
        profile_row.get("RISK_PATTERN") or "CLEAN"
    ).strip()

    customer_type = (
        application_row.get("CUSTOMER_TYPE") or "I"
    ).strip().upper()

    if not search_id:
        raise ValueError("Blank SEARCH_ID found.")

    if not reporting_date:
        raise ValueError(
            f"Invalid reporting date for {search_id}"
        )

    if customer_type == "I" and not pin_code:
        raise ValueError(
            f"Individual application has no PIN: {search_id}"
        )

    if customer_type != "I" and not document_no:
        raise ValueError(
            f"Non-individual application has no CIF: {search_id}"
        )

    rng = get_search_rng(search_id)

    inquiry_count = choose_inquiry_count(
        risk_pattern=risk_pattern,
        rng=rng
    )

    if inquiry_count == 0:
        return []

    bank_assignments = create_bank_assignments(
        search_id=search_id,
        inquiry_count=inquiry_count,
        rng=rng
    )

    # Unique numbers between 1 and 730:
    # every inquiry is strictly before the current MKR search.
    days_before_reporting = sorted(
        rng.sample(
            range(1, LOOKBACK_DAYS + 1),
            inquiry_count
        ),
        reverse=True
    )

    rows = []

    for position, days_before in enumerate(
        days_before_reporting
    ):
        inquiry_date = reporting_date - timedelta(
            days=days_before
        )

        bank_id, bank_name = bank_assignments[position]

        purpose_id, purpose_name = rng.choice(
            INQUIRY_PURPOSES
        )

        insert_timestamp = datetime.combine(
            inquiry_date,
            datetime.min.time()
        ) + timedelta(
            hours=rng.randint(8, 19),
            minutes=rng.randint(0, 59),
            seconds=rng.randint(0, 59)
        )

        rows.append({
            "SEARCH_ID": search_id,
            "PIN_CODE": pin_code if customer_type == "I" else "",
            "INQBANKID": bank_id,
            "INQBANKNAME": bank_name,
            "INQDATE": inquiry_date.isoformat(),
            "INQORGIDTYPE": "001",
            "INQPURPOSEID": purpose_id,
            "INQPURPOSENAME": purpose_name,
            "INS_DTTM": insert_timestamp.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "DOCUMENTNO": (
                document_no
                if customer_type != "I"
                else ""
            ),
            "REPORTINGDATE": reporting_date.isoformat()
        })

    return rows


# ============================================================
# VALIDATION
# ============================================================

def validate_application_inquiries(
    application_master_rows,
    application_inquiry_rows
):
    master_by_search_id = {
        row["SEARCH_ID"].strip(): row
        for row in application_master_rows
    }

    unique_rows = set()

    for row in application_inquiry_rows:
        search_id = row["SEARCH_ID"].strip()

        if search_id not in master_by_search_id:
            raise ValueError(
                f"Unexpected SEARCH_ID in inquiry history: "
                f"{search_id}"
            )

        reporting_date = parse_date(
            master_by_search_id[search_id]["REPORTINGDATE"]
        )

        inquiry_date = parse_date(row["INQDATE"])
        child_reporting_date = parse_date(
            row["REPORTINGDATE"]
        )

        if inquiry_date >= reporting_date:
            raise ValueError(
                f"Inquiry is not earlier than reporting date: "
                f"{search_id}"
            )

        if (
            reporting_date - inquiry_date
        ).days > LOOKBACK_DAYS:
            raise ValueError(
                f"Inquiry is outside two-year lookback: "
                f"{search_id}"
            )

        if child_reporting_date != reporting_date:
            raise ValueError(
                f"Reporting date mismatch: {search_id}"
            )

        bank_id = row["INQBANKID"].strip()
        bank_name = row["INQBANKNAME"].strip().upper()

        if bank_name == INTERNAL_NAME:
            if bank_id != INTERNAL_CODE:
                raise ValueError(
                    "INTERNAL inquiry does not use code 1000."
                )

        elif bank_name == EXTERNAL_NAME:
            if bank_id not in EXTERNAL_CODES:
                raise ValueError(
                    "EXTERNAL inquiry has invalid code."
                )

        else:
            raise ValueError(
                f"Unmasked inquiry bank name found: {bank_name}"
            )

        unique_key = (
            search_id,
            row["INQDATE"],
            row["INQBANKID"],
            row["INQPURPOSEID"]
        )

        if unique_key in unique_rows:
            raise ValueError(
                f"Duplicate inquiry-history row: {unique_key}"
            )

        unique_rows.add(unique_key)


# ============================================================
# MAIN
# ============================================================

def main_generate_application_inquiry_history():
    application_lookup = load_application_lookup()
    profile_lookup = load_profile_lookup()

    _, application_master_rows = read_csv_rows(
        file_path=APPLICATION_MKR_MASTER_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "DOCUMENT_NO",
            "REPORTINGDATE"
        },
        table_name="Application MKR master"
    )

    inquiry_columns, existing_inquiry_rows = read_csv_rows(
        file_path=UNIFIED_INQUIRY_FILE,
        required_columns=set(INQUIRY_COLUMNS),
        table_name="Unified MKR inquiry history"
    )

    # Safe rerun:
    # preserve general history, recreate only MKRAPP rows.
    general_inquiry_rows = [
        row
        for row in existing_inquiry_rows
        if not (
            row.get("SEARCH_ID") or ""
        ).strip().startswith("MKRAPP")
    ]

    application_inquiry_rows = []

    for master_row in application_master_rows:
        search_id = (
            master_row.get("SEARCH_ID") or ""
        ).strip()

        application_row = application_lookup.get(search_id)

        if not application_row:
            raise ValueError(
                f"Application not found: {search_id}"
            )

        account_number = (
            application_row.get("ACCOUNT_NUMBER") or ""
        ).strip()

        profile_row = profile_lookup.get(account_number)

        if not profile_row:
            raise ValueError(
                f"Profile not found: {account_number}"
            )

        application_inquiry_rows.extend(
            create_application_inquiries(
                master_row=master_row,
                application_row=application_row,
                profile_row=profile_row
            )
        )

    validate_application_inquiries(
        application_master_rows=application_master_rows,
        application_inquiry_rows=application_inquiry_rows
    )

    unified_inquiry_rows = (
        general_inquiry_rows
        + application_inquiry_rows
    )

    write_csv(
        APPLICATION_INQUIRY_FILE,
        INQUIRY_COLUMNS,
        application_inquiry_rows
    )

    write_csv(
        UNIFIED_INQUIRY_FILE,
        inquiry_columns,
        unified_inquiry_rows
    )

    write_csv(
        APPLICATION_SAMPLE_FILE,
        INQUIRY_COLUMNS,
        application_inquiry_rows[:100]
    )

    write_csv(
        SAMPLE_FILE,
        INQUIRY_COLUMNS,
        general_inquiry_rows[:50]
        + application_inquiry_rows[:50]
    )

    searches_with_history = len({
        row["SEARCH_ID"]
        for row in application_inquiry_rows
    })

    internal_rows = sum(
        1 for row in application_inquiry_rows
        if row["INQBANKNAME"] == INTERNAL_NAME
    )

    external_rows = sum(
        1 for row in application_inquiry_rows
        if row["INQBANKNAME"] == EXTERNAL_NAME
    )

    summary = {
        "application_mkr_searches": len(
            application_master_rows
        ),
        "application_inquiry_history_rows": len(
            application_inquiry_rows
        ),
        "application_searches_with_prior_inquiries": (
            searches_with_history
        ),
        "application_searches_without_prior_inquiries": (
            len(application_master_rows)
            - searches_with_history
        ),
        "unified_inquiry_history_rows": len(
            unified_inquiry_rows
        ),
        "lookback_rule": (
            "INQDATE is 1–730 days before REPORTINGDATE"
        ),
        "bank_masking_policy": {
            "internal": "1000 / INTERNAL",
            "external": "1001-1020 / EXTERNAL",
            "external_mapping_scope": "per SEARCH_ID"
        }
    }

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print(
        "Application MKR inquiry-history generation "
        "completed successfully."
    )
    print(
        f"New application inquiry-history rows: "
        f"{len(application_inquiry_rows):,}"
    )
    print(
        "Application searches with prior inquiries: "
        f"{searches_with_history:,}"
    )
    print(
        "Application searches without prior inquiries: "
        f"{len(application_master_rows) - searches_with_history:,}"
    )
    print(
        f"INTERNAL rows (1000): {internal_rows:,}"
    )
    print(
        f"EXTERNAL rows (1001-1020): {external_rows:,}"
    )
    print(
        f"Unified inquiry-history rows: "
        f"{len(unified_inquiry_rows):,}"
    )
    print(
        "Two-year lookback and masking validation: passed"
    )


main_generate_application_inquiry_history()

Application MKR inquiry-history generation completed successfully.
New application inquiry-history rows: 23,362
Application searches with prior inquiries: 12,668
Application searches without prior inquiries: 7,332
INTERNAL rows (1000): 2,801
EXTERNAL rows (1001-1020): 20,561
Unified inquiry-history rows: 121,581
Two-year lookback and masking validation: passed


In [13]:
import shutil
from pathlib import Path


project_folder = Path.cwd()
total_bytes, used_bytes, free_bytes = shutil.disk_usage(project_folder)

gb = 1024 ** 3

print(f"Project folder: {project_folder}")
print(f"Total C drive space: {total_bytes / gb:.1f} GB")
print(f"Used space: {used_bytes / gb:.1f} GB")
print(f"Free space: {free_bytes / gb:.1f} GB")

if free_bytes / gb >= 35:
    print("\nDisk-space check: PASSED")
    print("There is enough space to generate the daily contract_current CSV.")
else:
    print("\nDisk-space check: STOP")
    print("Less than 35 GB is free. We should generate the daily table in smaller partitions instead.")

Project folder: c:\Users\ASUS\Desktop\project\risk_scoring_project
Total C drive space: 953.0 GB
Used space: 628.1 GB
Free space: 324.8 GB

Disk-space check: PASSED
There is enough space to generate the daily contract_current CSV.


In [14]:
import csv
import hashlib
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

REPORTING_CUTOFF_DATE = date(2026, 6, 1)
RANDOM_SEED = "contract_snapshot_and_outcome_v1"


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

PROFILE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "rs_contract_profile_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

SNAPSHOT_FILE = (
    RAW_DATA_FOLDER
    / "rs_contract_current_base.csv"
)

OUTCOME_FILE = (
    RAW_DATA_FOLDER
    / "rs_contract_performance_outcome_base.csv"
)

SNAPSHOT_SAMPLE_FILE = (
    SAMPLE_DATA_FOLDER
    / "rs_contract_current_base_sample.csv"
)

OUTCOME_SAMPLE_FILE = (
    SAMPLE_DATA_FOLDER
    / "rs_contract_performance_outcome_base_sample.csv"
)

SUMMARY_FILE = (
    RAW_DATA_FOLDER
    / "rs_contract_snapshot_generation_summary.json"
)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

SNAPSHOT_COLUMNS = [
    "TRN_DT",
    "ACCOUNT_NUMBER",
    "BRANCH_CODE",
    "START_DATE_P",
    "START_DATE_I",
    "NEXT_PAYMENT_DATE",
    "CURR_PAYMENT_RECD",
    "S_PRINCIPAL_OUT_BAL_ACY",
    "EOP_CURR_PRIN_BAL",
    "EOP_CURR_MAIN_BAL",
    "EOP_PREV_PRIN_BAL",
    "EOP_BAL",
    "S_PRINCIPAL_OVERDUE",
    "S_MAIN_OVERDUE",
    "S_DELINQUENT_DAYS",
    "NUMB_INSTAL_PRIN",
    "S_OVERDUE_DAYS_PRIN",
    "S_OVERDUE_DAYS_INT",
    "S_DELQ_LIFE_TIMES",
    "S_DELQ_YEAR_TIMES",
    "DELQ_HISTORY",
    "REMAIN_NO_OF_PMTS",
    "ORIG_NEXT_PAYMENT_DATE",
    "EFF_MATURITY_DATE",
    "S_MAX_LIFETIME_DPD",
    "S_LAST_DELINQUENT_DATE",
    "S_INTEREST_OUT_BAL_ACY",
    "M_ACCOUNT_NUMBER",
    "M_BRANCH_CODE",
    "M_CUSTOMER_ID",
    "M_PRODUCT_CODE",
    "M_VALUE_DATE",
    "M_MATURITY_DATE",
    "M_AMOUNT_FINANCED",
    "M_PRODUCT_CATEGORY",
    "M_CURRENCY",
    "M_ORIGINAL_ST_DATE",
    "M_TENOR",
    "M_CURR_DATE",
    "M_ACCOUNT_STATUS",
    "M_CUSTOMER_TYPE",
    "M_RATE",
    "AM_PRN_LAST_PMNT_AMT",
    "AM_INT_LAST_PMNT_DATE",
    "AM_INT_LAST_PMNT_AMT",
    "S_LAST_PMNT_VALUE_DATE",
    "A_TERM",
    "A_ORIG_AMT",
    "A_EMI",
    "A_PERIODS_PASSED",
    "A_OVERDUE_DAYS",
    "A_OVERDUE_AMNT",
    "A_OVERDUE_MON",
    "CS_REPAYMENT_TYPE",
    "CLOSE_DATE",
    "INS_DTTM"
]

OUTCOME_COLUMNS = [
    "ACCOUNT_NUMBER",
    "VALUE_DATE",
    "SNAPSHOT_DATE",
    "OBSERVATION_END_DATE",
    "FULL_12M_OBSERVED_FLAG",
    "MAX_DPD_OBSERVED",
    "FIRST_30_DPD_DATE",
    "FIRST_90_DPD_DATE",
    "BAD_12M_FLAG",
    "FINAL_ACCOUNT_STATUS",
    "CLOSE_DATE"
]


# ============================================================
# HELPERS
# ============================================================

def parse_date(value):
    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def add_months(start_date, months):
    new_month = start_date.month - 1 + months
    new_year = start_date.year + new_month // 12
    new_month = new_month % 12 + 1

    month_days = [
        31,
        29 if new_year % 4 == 0 and (
            new_year % 100 != 0
            or new_year % 400 == 0
        ) else 28,
        31, 30, 31, 30,
        31, 31, 30, 31, 30, 31
    ]

    new_day = min(
        start_date.day,
        month_days[new_month - 1]
    )

    return date(new_year, new_month, new_day)


def months_between(start_date, end_date):
    months = (
        (end_date.year - start_date.year) * 12
        + end_date.month
        - start_date.month
    )

    if end_date.day < start_date.day:
        months -= 1

    return max(0, months)


def money(value):
    return f"{max(0, value):.2f}"


def stable_rng(account_number):
    seed_text = (
        f"{RANDOM_SEED}|{account_number}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).hexdigest()

    return random.Random(
        int(digest[:16], 16)
    )


def read_csv_rows(file_path, required_columns, table_name):
    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return list(reader)


def write_csv(file_path, fieldnames, rows):
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# SYNTHETIC PERFORMANCE LOGIC
# ============================================================

def choose_max_dpd(risk_pattern, observed_days, rng):
    """
    Creates the maximum delinquency observed up to the
    observation end date.

    This is used only to create a realistic synthetic outcome.
    It is not included in the application feature dataset.
    """

    if observed_days <= 0:
        return 0

    if risk_pattern == "CLEAN":
        choices = [0, rng.randint(1, 14)]
        weights = [94, 6]

    elif risk_pattern == "SHORT_LATE":
        choices = [
            0,
            rng.randint(1, 29),
            rng.randint(30, 59),
            rng.randint(90, 120)
        ]
        weights = [28, 48, 21, 3]

    elif risk_pattern == "SERIOUS_LATE":
        choices = [
            rng.randint(1, 29),
            rng.randint(30, 89),
            rng.randint(90, 180)
        ]
        weights = [12, 58, 30]

    else:  # BAD
        choices = [
            rng.randint(30, 89),
            rng.randint(90, 180),
            rng.randint(181, 365)
        ]
        weights = [14, 56, 30]

    selected_dpd = rng.choices(
        choices,
        weights=weights,
        k=1
    )[0]

    return min(selected_dpd, observed_days)


def build_delq_history(max_dpd, current_dpd, rng):
    """
    Six most recent synthetic payment-cycle delinquency values.
    """

    history = [0, 0, 0, 0, 0, 0]

    if max_dpd == 0:
        return ",".join(str(value) for value in history)

    affected_cycles = min(
        6,
        max(1, int(max_dpd / 30) + 1)
    )

    positions = rng.sample(
        range(6),
        affected_cycles
    )

    for position in positions:
        history[position] = min(
            max_dpd,
            rng.choice([
                0,
                min(max_dpd, 15),
                min(max_dpd, 30),
                min(max_dpd, 60),
                max_dpd
            ])
        )

    history[-1] = current_dpd

    return ",".join(str(value) for value in history)


def create_contract_rows(profile_row):
    account_number = (
        profile_row.get("ACCOUNT_NUMBER") or ""
    ).strip()

    rng = stable_rng(account_number)

    value_date = parse_date(
        profile_row.get("VALUE_DATE") or ""
    )

    maturity_date = parse_date(
        profile_row.get("MATURITY_DATE") or ""
    )

    close_date = parse_date(
        profile_row.get("CLOSE_DATE") or ""
    )

    if not account_number or not value_date or not maturity_date:
        raise ValueError(
            f"Invalid profile row for account: {account_number}"
        )

    if (
        close_date
        and close_date <= REPORTING_CUTOFF_DATE
    ):
        snapshot_date = close_date
        account_status = "L"

    else:
        snapshot_date = REPORTING_CUTOFF_DATE
        account_status = "A"
        close_date = None

    observation_end_date = min(
        snapshot_date,
        value_date + timedelta(days=365)
    )

    observed_days = max(
        0,
        (observation_end_date - value_date).days
    )

    full_12m_observed = (
        account_status == "L"
        or snapshot_date >= value_date + timedelta(days=365)
    )

    risk_pattern = (
        profile_row.get("RISK_PATTERN") or "CLEAN"
    ).strip()

    max_dpd_observed = choose_max_dpd(
        risk_pattern=risk_pattern,
        observed_days=observed_days,
        rng=rng
    )

    first_90_dpd_date = None
    first_30_dpd_date = None

    if max_dpd_observed >= 90:
        first_90_day = rng.randint(
            90,
            min(365, observed_days)
        )

        first_90_dpd_date = (
            value_date
            + timedelta(days=first_90_day)
        )

        first_30_dpd_date = (
            first_90_dpd_date
            - timedelta(days=60)
        )

    elif max_dpd_observed >= 30:
        first_30_day = rng.randint(
            30,
            observed_days
        )

        first_30_dpd_date = (
            value_date
            + timedelta(days=first_30_day)
        )

    if first_90_dpd_date:
        bad_12m_flag = "1"

    elif full_12m_observed:
        bad_12m_flag = "0"

    else:
        bad_12m_flag = ""

    if account_status == "L" or max_dpd_observed == 0:
        current_dpd = 0

    else:
        active_delinquency_probability = {
            "CLEAN": 0.15,
            "SHORT_LATE": 0.45,
            "SERIOUS_LATE": 0.60,
            "BAD": 0.82
        }.get(risk_pattern, 0.20)

        if rng.random() <= active_delinquency_probability:
            lower_limit = 1

            if (
                max_dpd_observed >= 90
                and risk_pattern == "BAD"
                and rng.random() < 0.70
            ):
                lower_limit = 90

            current_dpd = rng.randint(
                lower_limit,
                max_dpd_observed
            )

        else:
            current_dpd = 0

    amount_financed = float(
        profile_row.get("AMOUNT_FINANCED") or 0
    )

    term_months = int(
        float(
            profile_row.get("TERM_MONTHS") or 1
        )
    )

    interest_rate = float(
        profile_row.get("INTEREST_RATE") or 0
    )

    emi_amount = float(
        profile_row.get("EMI_AMOUNT") or 0
    )

    periods_passed = min(
        term_months,
        months_between(value_date, snapshot_date)
    )

    if account_status == "L":
        principal_outstanding = 0.0
        interest_outstanding = 0.0
        remaining_payments = 0
        current_payment_received = emi_amount

    else:
        if periods_passed >= term_months:
            outstanding_ratio = rng.uniform(
                0.12,
                0.65
            )

        else:
            scheduled_ratio = periods_passed / term_months

            outstanding_ratio = max(
                0.03,
                1 - (
                    scheduled_ratio
                    * rng.uniform(0.85, 1.05)
                )
            )

        principal_outstanding = (
            amount_financed
            * min(1, outstanding_ratio)
        )

        interest_outstanding = (
            principal_outstanding
            * (interest_rate / 100)
            / 12
            * rng.uniform(0.20, 0.90)
        )

        remaining_payments = max(
            0,
            term_months - periods_passed
        )

        if current_dpd == 0:
            current_payment_received = emi_amount

        else:
            current_payment_received = (
                emi_amount
                * rng.uniform(0.00, 0.45)
            )

    if current_dpd > 0:
        overdue_ratio = min(
            0.80,
            0.03 + current_dpd / 1000
        )

        principal_overdue = (
            principal_outstanding
            * overdue_ratio
        )

        interest_overdue = (
            interest_outstanding
            * min(
                1,
                0.25 + current_dpd / 500
            )
        )

    else:
        principal_overdue = 0.0
        interest_overdue = 0.0

    end_of_period_balance = (
        principal_outstanding
        + interest_outstanding
        + principal_overdue
        + interest_overdue
    )

    previous_principal_balance = (
        principal_outstanding
        + max(
            0,
            amount_financed / max(term_months, 1)
        )
    )

    if account_status == "A":
        next_payment_date = add_months(
            snapshot_date,
            1
        )

    else:
        next_payment_date = None

    first_payment_date = parse_date(
        profile_row.get("FIRST_PAYMENT_DATE") or ""
    )

    if not first_payment_date:
        first_payment_date = add_months(
            value_date,
            1
        )

    if current_dpd > 0:
        last_delinquent_date = snapshot_date

    elif first_90_dpd_date:
        last_delinquent_date = min(
            snapshot_date,
            first_90_dpd_date
            + timedelta(
                days=min(max_dpd_observed, 30)
            )
        )

    elif first_30_dpd_date:
        last_delinquent_date = min(
            snapshot_date,
            first_30_dpd_date
        )

    else:
        last_delinquent_date = None

    if max_dpd_observed == 0:
        delinquency_life_times = 0

    elif max_dpd_observed < 30:
        delinquency_life_times = 1

    elif max_dpd_observed < 90:
        delinquency_life_times = rng.randint(1, 3)

    else:
        delinquency_life_times = rng.randint(2, 6)

    delinquency_year_times = min(
        delinquency_life_times,
        rng.randint(
            0,
            delinquency_life_times
        )
    )

    delq_history = build_delq_history(
        max_dpd=max_dpd_observed,
        current_dpd=current_dpd,
        rng=rng
    )

    repayment_type = rng.choice([
        "ANNUITY",
        "ANNUITY",
        "ANNUITY",
        "LEVEL_PRINCIPAL"
    ])

    snapshot_row = {
        "TRN_DT": snapshot_date.isoformat(),
        "ACCOUNT_NUMBER": account_number,
        "BRANCH_CODE": (
            profile_row.get("BRANCH_CODE") or ""
        ).strip(),
        "START_DATE_P": first_payment_date.isoformat(),
        "START_DATE_I": first_payment_date.isoformat(),
        "NEXT_PAYMENT_DATE": (
            next_payment_date.isoformat()
            if next_payment_date
            else ""
        ),
        "CURR_PAYMENT_RECD": money(
            current_payment_received
        ),
        "S_PRINCIPAL_OUT_BAL_ACY": money(
            principal_outstanding
        ),
        "EOP_CURR_PRIN_BAL": money(
            principal_outstanding
        ),
        "EOP_CURR_MAIN_BAL": money(
            interest_outstanding
        ),
        "EOP_PREV_PRIN_BAL": money(
            previous_principal_balance
        ),
        "EOP_BAL": money(
            end_of_period_balance
        ),
        "S_PRINCIPAL_OVERDUE": money(
            principal_overdue
        ),
        "S_MAIN_OVERDUE": money(
            interest_overdue
        ),
        "S_DELINQUENT_DAYS": str(current_dpd),
        "NUMB_INSTAL_PRIN": str(term_months),
        "S_OVERDUE_DAYS_PRIN": str(current_dpd),
        "S_OVERDUE_DAYS_INT": str(current_dpd),
        "S_DELQ_LIFE_TIMES": str(
            delinquency_life_times
        ),
        "S_DELQ_YEAR_TIMES": str(
            delinquency_year_times
        ),
        "DELQ_HISTORY": delq_history,
        "REMAIN_NO_OF_PMTS": str(
            remaining_payments
        ),
        "ORIG_NEXT_PAYMENT_DATE": (
            first_payment_date.isoformat()
        ),
        "EFF_MATURITY_DATE": maturity_date.isoformat(),
        "S_MAX_LIFETIME_DPD": str(
            max_dpd_observed
        ),
        "S_LAST_DELINQUENT_DATE": (
            last_delinquent_date.isoformat()
            if last_delinquent_date
            else ""
        ),
        "S_INTEREST_OUT_BAL_ACY": money(
            interest_outstanding
        ),
        "M_ACCOUNT_NUMBER": account_number,
        "M_BRANCH_CODE": (
            profile_row.get("BRANCH_CODE") or ""
        ).strip(),
        "M_CUSTOMER_ID": (
            profile_row.get("CUSTOMER_ID") or ""
        ).strip(),
        "M_PRODUCT_CODE": (
            profile_row.get("PRODUCT_CODE") or ""
        ).strip(),
        "M_VALUE_DATE": value_date.isoformat(),
        "M_MATURITY_DATE": maturity_date.isoformat(),
        "M_AMOUNT_FINANCED": money(
            amount_financed
        ),
        "M_PRODUCT_CATEGORY": (
            profile_row.get("PRODUCT_CATEGORY") or ""
        ).strip(),
        "M_CURRENCY": "AZN",
        "M_ORIGINAL_ST_DATE": value_date.isoformat(),
        "M_TENOR": str(term_months),
        "M_CURR_DATE": snapshot_date.isoformat(),
        "M_ACCOUNT_STATUS": account_status,
        "M_CUSTOMER_TYPE": (
            profile_row.get("CUSTOMER_TYPE") or ""
        ).strip(),
        "M_RATE": f"{interest_rate:.2f}",
        "AM_PRN_LAST_PMNT_AMT": money(
            current_payment_received
        ),
        "AM_INT_LAST_PMNT_DATE": (
            snapshot_date.isoformat()
            if current_payment_received > 0
            else ""
        ),
        "AM_INT_LAST_PMNT_AMT": money(
            min(
                current_payment_received,
                interest_outstanding
            )
        ),
        "S_LAST_PMNT_VALUE_DATE": (
            snapshot_date.isoformat()
            if current_payment_received > 0
            else ""
        ),
        "A_TERM": str(term_months),
        "A_ORIG_AMT": money(amount_financed),
        "A_EMI": money(emi_amount),
        "A_PERIODS_PASSED": str(periods_passed),
        "A_OVERDUE_DAYS": str(current_dpd),
        "A_OVERDUE_AMNT": money(
            principal_overdue
            + interest_overdue
        ),
        "A_OVERDUE_MON": str(
            int(current_dpd / 30)
        ),
        "CS_REPAYMENT_TYPE": repayment_type,
        "CLOSE_DATE": (
            close_date.isoformat()
            if close_date
            else ""
        ),
        "INS_DTTM": snapshot_date.isoformat()
    }

    outcome_row = {
        "ACCOUNT_NUMBER": account_number,
        "VALUE_DATE": value_date.isoformat(),
        "SNAPSHOT_DATE": snapshot_date.isoformat(),
        "OBSERVATION_END_DATE": (
            observation_end_date.isoformat()
        ),
        "FULL_12M_OBSERVED_FLAG": (
            "1" if full_12m_observed else "0"
        ),
        "MAX_DPD_OBSERVED": str(
            max_dpd_observed
        ),
        "FIRST_30_DPD_DATE": (
            first_30_dpd_date.isoformat()
            if first_30_dpd_date
            else ""
        ),
        "FIRST_90_DPD_DATE": (
            first_90_dpd_date.isoformat()
            if first_90_dpd_date
            else ""
        ),
        "BAD_12M_FLAG": bad_12m_flag,
        "FINAL_ACCOUNT_STATUS": account_status,
        "CLOSE_DATE": (
            close_date.isoformat()
            if close_date
            else ""
        )
    }

    return snapshot_row, outcome_row


# ============================================================
# VALIDATION
# ============================================================

def validate_rows(snapshot_rows, outcome_rows):
    snapshot_accounts = [
        row["ACCOUNT_NUMBER"]
        for row in snapshot_rows
    ]

    outcome_accounts = [
        row["ACCOUNT_NUMBER"]
        for row in outcome_rows
    ]

    if len(snapshot_accounts) != len(set(snapshot_accounts)):
        raise ValueError(
            "Duplicate ACCOUNT_NUMBER in contract snapshot."
        )

    if len(outcome_accounts) != len(set(outcome_accounts)):
        raise ValueError(
            "Duplicate ACCOUNT_NUMBER in outcome table."
        )

    if set(snapshot_accounts) != set(outcome_accounts):
        raise ValueError(
            "Snapshot and outcome account lists differ."
        )

    for snapshot_row, outcome_row in zip(
        snapshot_rows,
        outcome_rows
    ):
        snapshot_date = parse_date(
            snapshot_row["TRN_DT"]
        )

        close_date = parse_date(
            snapshot_row["CLOSE_DATE"]
        )

        status = snapshot_row["M_ACCOUNT_STATUS"]

        if status == "A":
            if snapshot_date != REPORTING_CUTOFF_DATE:
                raise ValueError(
                    "Active contract does not use cutoff date."
                )

            if close_date:
                raise ValueError(
                    "Active contract has a close date."
                )

        elif status == "L":
            if not close_date:
                raise ValueError(
                    "Closed contract has no close date."
                )

            if snapshot_date != close_date:
                raise ValueError(
                    "Closed contract snapshot is not on close date."
                )

        else:
            raise ValueError(
                f"Invalid account status: {status}"
            )

        bad_flag = outcome_row["BAD_12M_FLAG"]
        full_observed = outcome_row[
            "FULL_12M_OBSERVED_FLAG"
        ]

        first_90_dpd_date = parse_date(
            outcome_row["FIRST_90_DPD_DATE"]
        )

        value_date = parse_date(
            outcome_row["VALUE_DATE"]
        )

        if bad_flag == "1" and not first_90_dpd_date:
            raise ValueError(
                "BAD_12M_FLAG = 1 without FIRST_90_DPD_DATE."
            )

        if bad_flag == "0" and full_observed != "1":
            raise ValueError(
                "BAD_12M_FLAG = 0 without full observation."
            )

        if first_90_dpd_date:
            if first_90_dpd_date > (
                value_date + timedelta(days=365)
            ):
                raise ValueError(
                    "First 90+ DPD date is outside 12 months."
                )


# ============================================================
# MAIN
# ============================================================

def main_generate_contract_snapshot_and_outcome():
    profile_rows = read_csv_rows(
        file_path=PROFILE_FILE,
        required_columns={
            "ACCOUNT_NUMBER",
            "BRANCH_CODE",
            "CUSTOMER_ID",
            "CUSTOMER_TYPE",
            "PRODUCT_CODE",
            "PRODUCT_CATEGORY",
            "VALUE_DATE",
            "MATURITY_DATE",
            "CLOSE_DATE",
            "AMOUNT_FINANCED",
            "TERM_MONTHS",
            "INTEREST_RATE",
            "EMI_AMOUNT",
            "RISK_PATTERN"
        },
        table_name="Contract profiles"
    )

    snapshot_rows = []
    outcome_rows = []

    for profile_row in profile_rows:
        snapshot_row, outcome_row = (
            create_contract_rows(profile_row)
        )

        snapshot_rows.append(snapshot_row)
        outcome_rows.append(outcome_row)

    validate_rows(
        snapshot_rows=snapshot_rows,
        outcome_rows=outcome_rows
    )

    write_csv(
        SNAPSHOT_FILE,
        SNAPSHOT_COLUMNS,
        snapshot_rows
    )

    write_csv(
        OUTCOME_FILE,
        OUTCOME_COLUMNS,
        outcome_rows
    )

    write_csv(
        SNAPSHOT_SAMPLE_FILE,
        SNAPSHOT_COLUMNS,
        snapshot_rows[:100]
    )

    write_csv(
        OUTCOME_SAMPLE_FILE,
        OUTCOME_COLUMNS,
        outcome_rows[:100]
    )

    active_contracts = sum(
        1 for row in snapshot_rows
        if row["M_ACCOUNT_STATUS"] == "A"
    )

    closed_contracts = len(snapshot_rows) - active_contracts

    full_12m_observed = sum(
        1 for row in outcome_rows
        if row["FULL_12M_OBSERVED_FLAG"] == "1"
    )

    bad_12m_contracts = sum(
        1 for row in outcome_rows
        if row["BAD_12M_FLAG"] == "1"
    )

    unknown_target_contracts = sum(
        1 for row in outcome_rows
        if row["BAD_12M_FLAG"] == ""
    )

    summary = {
        "contract_snapshot_rows": len(snapshot_rows),
        "daily_history_rows_generated": 0,
        "grain": "One row per ACCOUNT_NUMBER",
        "active_contract_snapshot_date": (
            REPORTING_CUTOFF_DATE.isoformat()
        ),
        "closed_contract_snapshot_rule": (
            "TRN_DT = CLOSE_DATE"
        ),
        "active_contracts": active_contracts,
        "closed_contracts": closed_contracts,
        "performance_outcome_rows": len(outcome_rows),
        "full_12m_observed_contracts": full_12m_observed,
        "bad_12m_contracts": bad_12m_contracts,
        "unknown_target_contracts": unknown_target_contracts,
        "target_definition": (
            "BAD_12M_FLAG = 1 when FIRST_90_DPD_DATE "
            "occurs within 365 days after VALUE_DATE"
        )
    }

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print(
        "Contract snapshot and performance-outcome "
        "generation completed successfully."
    )
    print(
        f"Contract snapshot rows: {len(snapshot_rows):,}"
    )
    print(
        "Daily contract rows generated: 0"
    )
    print(
        f"Closed-contract snapshots: {closed_contracts:,}"
    )
    print(
        f"Active-contract snapshots at 2026-06-01: "
        f"{active_contracts:,}"
    )
    print(
        f"Full 12-month observed contracts: "
        f"{full_12m_observed:,}"
    )
    print(
        f"BAD_12M_FLAG = 1 contracts: "
        f"{bad_12m_contracts:,}"
    )
    print(
        f"Target not yet known contracts: "
        f"{unknown_target_contracts:,}"
    )
    print(f"Snapshot file: {SNAPSHOT_FILE}")
    print(f"Outcome file: {OUTCOME_FILE}")


main_generate_contract_snapshot_and_outcome()

Contract snapshot and performance-outcome generation completed successfully.
Contract snapshot rows: 20,000
Daily contract rows generated: 0
Closed-contract snapshots: 9,401
Active-contract snapshots at 2026-06-01: 10,599
Full 12-month observed contracts: 17,641
BAD_12M_FLAG = 1 contracts: 2,001
Target not yet known contracts: 2,163
Snapshot file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contract_current_base.csv
Outcome file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contract_performance_outcome_base.csv


In [15]:
import csv
from pathlib import Path

project_root = Path.cwd()
raw_folder = project_root / "data" / "raw"

files = [
    "rs_customers_base.csv",
    "rs_contracts_base.csv",
    "rs_applications_base.csv",
    "rs_mkr_master_base.csv",
    "rs_mkr_liabilities_base.csv",
    "rs_mkr_liability_history_base.csv",
    "rs_mkr_inquiry_history_base.csv",
    "rs_contract_current_base.csv",
    "rs_contract_performance_outcome_base.csv",
]

for file_name in files:
    file_path = raw_folder / file_name

    if not file_path.exists():
        print(f"\n{file_name}")
        print("STATUS: FILE NOT FOUND")
        continue

    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        headers = next(reader)

    print("\n" + "=" * 80)
    print(file_name)
    print(f"Column count: {len(headers)}")
    print("Columns:")
    print(", ".join(headers))


rs_customers_base.csv
Column count: 100
Columns:
GEN_DIM_CUST_ID, CUSTOMER_NO, PIN, DOCUMENT_NO, TAX_ID, CUSTOMER_TYPE, CUSTOMER_SUB_TYPE, FULL_NAME, FIRST_NAME, LAST_NAME, MIDDLE_NAME, GENDER, BIRTH_DT, PLACE_OF_BIRTH, MARITAL_STATUS, NATIONALITY, OPEN_REASON, CREATION_DT, OPENING_BRN, FLG_IS_COURT_ISSUE, OCCUPATION, WORKPLACE_CUSTOMER_NO, WORKPLACE_NAME, FLG_IS_BLACK_LIST, FLG_IS_VIP, FLG_IS_EMPLOYEER, NUM_OF_EMPLOYEER, RM_NAME, CUSTOMER_GENERATION, FLG_IS_AFFLUENT, FLG_IS_SALARY, FLG_IS_PENSION, FLG_IS_EDV, RISK_LEVEL, RESIDENT_STATUS, RM_ID_CORP, GROUP_ID_CORP, CHECKER_DT_STAMP, MAKER_DT_STAMP, EMPL_PROPERTY_TYPE, SHORT_NAME, COUNTRY, DOCUMENT_NAME, LEGAL_GUARDIAN, PASSPORT_COUNTRY, PASSPORT_NATIONAL_ID, PASSPORT_ISS_DT, PASSPORT_EXP_DT, P_ADDRESS1, P_ADDRESS2, P_ADDRESS3, ENGLISH_FULL_NAME, FLG_IS_BENEFICIARY, CUSTOMER_SINCE, DIRECTOR_CUST_NO, FLG_IS_GREEN_CARD, US_BIRTH, FLG_IS_US_CITIZEN, US_PAYMENT, FLG_IS_US_RESIDENT_POA, FLG_IS_US_ADDRESS, FLG_IS_US_LIVING, US_PASSPORT, FLG_

In [16]:
from oracle_connection import get_oracle_connection


expected_tables = [
    "RS_CUSTOMERS",
    "RS_CONTRACTS",
    "RS_MKR_MASTER",
    "RS_APPLICATIONS",
    "RS_MKR_LIABILITIES",
    "RS_MKR_LIABILITY_HISTORY",
    "RS_MKR_INQUIRY_HISTORY",
    "RS_CONTRACT_CURRENT",
    "RS_CONTRACT_PERFORMANCE_OUTCOME"
]

connection = get_oracle_connection()

try:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT table_name
            FROM user_tables
            WHERE table_name LIKE 'RS_%'
            ORDER BY table_name
        """)

        existing_tables = {
            row[0]
            for row in cursor.fetchall()
        }

    print("Oracle table check completed.\n")

    for table_name in expected_tables:
        status = (
            "EXISTS"
            if table_name in existing_tables
            else "NOT CREATED"
        )
        print(f"{table_name}: {status}")

finally:
    connection.close()

Oracle table check completed.

RS_CUSTOMERS: NOT CREATED
RS_CONTRACTS: NOT CREATED
RS_MKR_MASTER: NOT CREATED
RS_APPLICATIONS: NOT CREATED
RS_MKR_LIABILITIES: NOT CREATED
RS_MKR_LIABILITY_HISTORY: NOT CREATED
RS_MKR_INQUIRY_HISTORY: NOT CREATED
RS_CONTRACT_CURRENT: NOT CREATED
RS_CONTRACT_PERFORMANCE_OUTCOME: NOT CREATED


In [ ]:
from oracle_connection import get_oracle_connection


connection = get_oracle_connection()

try:
    with connection.cursor() as cursor:
        cursor.execute("""
            ALTER TABLE RS_CUSTOMERS
            MODIFY (
                DT_QHT_GROUP VARCHAR2(35 CHAR)
            )
        """)

    connection.commit()

    print("Schema fix completed.")
    print("RS_CUSTOMERS.DT_QHT_GROUP is now VARCHAR2(35).")

finally:
    connection.close()
    print("Oracle connection closed.")